# Model transfer, feature selection and maps

**Original file:** `1_Spectors/1_JypiterLabProcessing/01_SpectralAnalysis_Part02_01.ipynb` in the author's project archive.

Transfers the models between years by actively selecting samples of the target dates, repeats them with the bands selected through VIP, draws Figures 3 to 9 and writes the files of the maps. The equivalent code in this repository is in `src/model_transfer/transfer.py`, `feature_selection.py` and `figures.py`, which corrects the two issues explained at the end of the main README. This notebook keeps the original behaviour, including those issues.

**Status:** kept for reference; it does not run as it is. It needs Python 3.6, library versions of that time, files that are not shared and, in places, manual changes between runs. `<PROJECT_ROOT>`, `<RAW_DATA>` and `<CONDA_ENVS>` replace local folders. To reproduce the results, use the scripts in the main folder of this repository.

In [ ]:
# Switches for tasks that were turned off in the saved notebook. Leave them off to read saved files.
SAVE_OUTPUTS = False
EXPORT_SHARED_TABLES = False

## 1. Setup

### Packages

Libraries and working folder. *(Original cell 1.)*

In [ ]:
import os, sys
import collections
import copy
import cv2
from decimal import Decimal
import fiona
os.environ['GDAL_DATA'] = r'<CONDA_ENVS>\Python365_PCSE\Library\share\gdal'
import gdal
import gdalconst
import geopandas as gpd
import glob
import itertools
import math
import matplotlib
import matplotlib.pyplot as plt
%matplotlib qt
import numpy as np
np.set_printoptions(suppress=True)
import ogr
import osr
import pandas as pd
import pickle
import progressbar as pbar
import pylab
os.environ['PROJ_LIB'] = r'<CONDA_ENVS>\Python365\Library\share\proj'
import pyproj
import random
import rasterio
import rasterio.features
import rasterio.mask
import rasterio.plot
from shapely.geometry import LinearRing as LinearR
from shapely.geometry import Point
from shapely.geometry import LineString as LineS
from shapely.geometry.polygon import Polygon as PolyS

import seaborn as sns

import SimpleITK as sitk

import affine
import scipy
import scipy.stats

import sklearn
from   sklearn.cross_decomposition import PLSRegression
import sklearn.cluster
import sklearn.decomposition
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold, RepeatedKFold
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler

import spectral

HomeDir = r'<PROJECT_ROOT>\DataProcessing\1_Script_source'
os.chdir( HomeDir )
from RemoveIrregularPad_v3 import PadIrregRem

print("This notebook was built with:")
print("python version: %s " % sys.version)

## 2. Data preparation

### Extracted spectra

Reads the spectra produced by notebook 01. *(Original cell 2.)*

In [ ]:
# 2014 and 2017 - after buffering
os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV' ) # get folder with images - acq 1
compRefT = 'New_UpD.'  # latest file version

PatternUpload = '*SU_VegProp_B_FLD_' + compRefT + '*pkl' # nearest flight line to the plot center
DataExtAll = glob.glob(PatternUpload)
SpecAll = []
for ii in range(len(DataExtAll)):  # acq
    SpecAll.append( pd.read_pickle(DataExtAll[ii]) )

# column names - spectral information
col_ref  = [ 'b'+ [str(x) if x>=100 else ( '00' + str(x) if x<10 else '0'+str(x) )][0] for x in range(0, 101, 1) ]

# get only spectral information ; 0-1 or 0-100 interval
SpecAll_sel_0_1   = []
SpecAll_sel_0_100 = []
for ii in range(len(SpecAll)): # acq
    SpecAll_sel_0_1.append(   np.expand_dims(SpecAll[ii].loc[:,col_ref] / 10000, axis=0) )
    SpecAll_sel_0_100.append( np.expand_dims(SpecAll[ii].loc[:,col_ref] / 100, axis=0) )

# 2014 and 2017 - Full dataset - before buffering
PatternUpload  = '*SU_VegProp_FLD_' + compRefT +'*pkl' # nearest flight line to the plot center
DataExtAllFull = glob.glob(PatternUpload)

SpecAllFull = []
for ii in range(len(DataExtAllFull)):  # acq
    SpecAllFull.append( pd.read_pickle(DataExtAllFull[ii]) )

# get only spectral information ; 0-1 or 0-100 interval
SpecAllFull_sel_0_1   = []
SpecAllFull_sel_0_100 = []
for ii in range(len(SpecAllFull)): # acq
    SpecAllFull_sel_0_1.append(   np.expand_dims(SpecAllFull[ii].loc[:,col_ref] / 10000, axis=0) )
    SpecAllFull_sel_0_100.append( np.expand_dims(SpecAllFull[ii].loc[:,col_ref] / 100, axis=0) )

### Spatial coordinates

Adds the coordinates of each pixel. *(Original cell 3.)*

In [ ]:
os.chdir(r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV') # get folder with images - acq 1
PatternUpload  = '*SU_geo_VegProp_B_FLD_' + compRefT + '*pkl' # nearest flight line to the plot center
DataExt_geoAll = glob.glob(PatternUpload)

# coord. syst.
crsT = 'EPSG:32632'
projRefT_wkt = pyproj.crs.CRS(crsT).to_wkt()

SpecAll_geo = []
for ii in range(len(DataExt_geoAll)):  # acq
    SpecAll_geo.append( gpd.GeoDataFrame( data=pd.DataFrame( pd.read_pickle(DataExt_geoAll[ii]).drop(['geometry'], axis=1) ), \
                                          crs=projRefT_wkt, \
                                          geometry=pd.read_pickle(DataExt_geoAll[ii]).geometry.tolist() ) )

# get points with spatial coordinates  -  Full spectra
PatternUpload      = '*SU_geo_VegProp_FLD_' + compRefT + '*pkl' # nearest flight line to the plot center
DataExt_geoAllFull = glob.glob(PatternUpload)

SpecAll_geoFull = []
for ii in range(len(DataExt_geoAllFull)):  # acq
    SpecAll_geoFull.append( gpd.GeoDataFrame( data=pd.DataFrame( pd.read_pickle(DataExt_geoAllFull[ii]).drop(['geometry'], axis=1) ), \
                                              crs=projRefT_wkt, \
                                              geometry=pd.read_pickle(DataExt_geoAllFull[ii]).geometry.tolist() ) )

### Spectral smoothing

Resamples the spectra to a bandwidth of 30 nm and removes the last two bands. *(Original cell 4.)*

In [ ]:
# if wanted >> resample spectra - https://github.com/spectralpython/spectral/blob/master/spectral/algorithms/resampling.py

# number of bands to consider in the 'smoothing window'
newFWHM = 30

# resampler - without last 2 bands (attention: original dataset should be resized >> nbr of bands should match the resampler -
FWHM_origin = [5] * len(range(450, 941, 5))
FWHM_transf = [newFWHM] * len(range(450, 941, 5))
center_origin = range(450, 941, 5)

# resampler for raw >> smoothed conversion
resamplerO = spectral.BandResampler(centers1=center_origin, fwhm1=FWHM_origin, centers2=center_origin,
                                    fwhm2=FWHM_transf)

SpecAll_sel_0_1_res = copy.deepcopy(SpecAll_sel_0_1)
for ii in range(len(SpecAll_sel_0_1_res)):  # acq
    SpecAll_sel_0_1_res[ii] =  SpecAll_sel_0_1_res[ii][:,:,:len(center_origin)]
    for jj in range(SpecAll_sel_0_1[ii].shape[1]):  # pxls
        SpecAll_sel_0_1_res[ii][0,jj,:] = resamplerO( SpecAll_sel_0_1[ii][0,jj,:len(center_origin)] )  # resample

# get 0-100 interval dataset - smoothed spectra
SpecAll_sel_0_100_res = copy.deepcopy(SpecAll_sel_0_100)
for ii in range(len(SpecAll_sel_0_1_res)):   # date
    SpecAll_sel_0_100_res[ii] = SpecAll_sel_0_1_res[ii] * 100

# resample full dataset used for predictions
SpecAllFull_sel_0_1_res = copy.deepcopy(SpecAllFull_sel_0_1)
for ii in range(len(SpecAllFull_sel_0_1_res)):  # acq
    SpecAllFull_sel_0_1_res[ii] =  SpecAllFull_sel_0_1_res[ii][:,:,:len(center_origin)]
    for jj in range(SpecAllFull_sel_0_1[ii].shape[1]):  # pxls..
        SpecAllFull_sel_0_1_res[ii][0,jj,:] = resamplerO( SpecAllFull_sel_0_1[ii][0,jj,:len(center_origin)] )  # resample

# get 0-100 interval dataset for smoothed spectra
SpecAllFull_sel_0_100_res = copy.deepcopy(SpecAllFull_sel_0_100)
for ii in range( len(SpecAllFull_sel_0_1_res) ):   # date
    SpecAllFull_sel_0_100_res[ii] = SpecAllFull_sel_0_1_res[ii] * 100

### Ground truth

Reads the laboratory measurements of the traits. *(Original cell 5.)*

In [ ]:
os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\2_Ground_aux' )

# fields - long format
ToSRefT = [ 'Location', 'Year', 'Date', 'Treatment', 'Name', 'Replicate', \
            'Variable', 'Value', 'DateMonth', 'DateDay' ]

# load - Kleve | 2012 to 2019
HarvestKleveRaw = pd.read_csv( '568_2012-2017_2019_Kleve_All_Merge_New_Adjusted' + '.csv', sep=';')

# Rearrange in wide format

metaRefT = [ 'Location', 'Date', 'Year', 'DateMonth', 'DateDay', \
             'Treatment', 'Name', 'Replicate', 'Cut' ]

# reference to variables
varRefT = [ 'FM_kg_plot', 'DM_dt_ha', 'CanHeight_cm', 'CrudeAsh_perc_DM', \
            'CrudeFiber_perc_DM', 'CrudeProt_perc_DM', 'N_perc_DM' ]

# reference to dates

Gdata_df_all = []
for ii in range( len(varRefT) ):  # trait
    # dates with observations for a given trait
    dataToPSelT = HarvestKleveRaw[ (HarvestKleveRaw.Variable == varRefT[ii]) ]
    dataToPSelT = dataToPSelT.reset_index(drop=True)
    Gdata_df_allT = dataToPSelT.loc[:, metaRefT + ['Value'] ].copy(deep=True)
    Gdata_df_allT.columns =  metaRefT + [ varRefT[ii] ]
    if len(Gdata_df_all) == 0:
        Gdata_df_all =  Gdata_df_allT.copy(deep=True)
    else:
        Gdata_df_all[ varRefT[ii] ] = Gdata_df_allT[ varRefT[ii] ]

### Average per plot

Combines spectra and traits at plot level, used for dry matter. Set `EXPORT_SHARED_TABLES` to write the table shared in the dataset. *(Original cell 6.)*

In [ ]:
# subset data for each date
HarvRef  = [ '20140515', '20141015', '20170510', '20170830', '20171026' ]

for ii in range( len(SpecAll_sel_0_1_res) ):
    TreatRefAllT = [ (SpecAll[ii].TreatCode.tolist()[val]).split('.')[1] for val in range(SpecAll[ii].shape[0]) ]
    TreatRefT    = np.unique(TreatRefAllT).tolist()
    for kk in pbar.progressbar( range(len( TreatRefT )) ):  # treat
        RepRefT = sorted( SpecAll[ii].loc[ (np.array(TreatRefAllT) == TreatRefT[kk]), :].Replicate.unique().tolist() )
        for jj in range( len(RepRefT) ):  # replicate
            MetaST = SpecAll[ii].loc[ (np.array(TreatRefAllT) == TreatRefT[kk]) & (SpecAll[ii].Replicate == RepRefT[jj]), :]
            SpecAll_sel_0_1_res_dfT = pd.DataFrame( SpecAll_sel_0_1_res[ii][0,:,:] )
            SpecAll_sel_0_1_res_dfT_sel  = pd.DataFrame( SpecAll_sel_0_1_res_dfT.iloc[ MetaST.index.tolist(), :].mean().values.reshape((1, SpecAll_sel_0_1_res_dfT.shape[1])) )
            SpecAll_sel_0_1_res_dfT_meta = pd.DataFrame( np.array( (HarvRef[ii], TreatRefT[kk], RepRefT[jj]) ).reshape((1, len((HarvRef[ii], TreatRefT[kk], RepRefT[jj])))) )
            GdataT = Gdata_df_all.loc[ (Gdata_df_all.Date == (HarvRef[ii][0:4] + '-' + HarvRef[ii][4:6] + '-' + HarvRef[ii][6:]) ) & (Gdata_df_all.Replicate == RepRefT[jj]) & \
                                       (Gdata_df_all.Treatment == int( TreatRefT[kk] )), [ 'DM_dt_ha', 'N_perc_DM', 'CrudeProt_perc_DM']]

            if( all([(ii == 0), (jj == 0), (kk == 0)]) ):
                SpecAll_sel_0_1_res_Avg = pd.concat( [SpecAll_sel_0_1_res_dfT_meta.reset_index(drop=True), GdataT.reset_index(drop=True), \
                                                      SpecAll_sel_0_1_res_dfT_sel.reset_index(drop=True)], \
                                                      axis=1, ignore_index=True, sort=False )
                col_ref = ['Date', 'Treatment', 'Replicate'] + ['DM_dt_ha', 'N_perc_DM', 'CrudeProt_perc_DM'] + \
                          list( range(SpecAll_sel_0_1_res_dfT_sel.shape[1]) )
                SpecAll_sel_0_1_res_Avg.columns = col_ref

            else:
                SpecAll_sel_0_1_res_AvgT = pd.concat( [SpecAll_sel_0_1_res_dfT_meta.reset_index(drop=True), \
                                                       GdataT.reset_index(drop=True), SpecAll_sel_0_1_res_dfT_sel.reset_index(drop=True)], \
                                                       axis=1, ignore_index=True, sort=False )
                col_ref = ['Date', 'Treatment', 'Replicate'] + ['DM_dt_ha', 'N_perc_DM', 'CrudeProt_perc_DM'] + \
                          list( range(SpecAll_sel_0_1_res_dfT_sel.shape[1]) )
                SpecAll_sel_0_1_res_AvgT.columns = col_ref
                SpecAll_sel_0_1_res_Avg = SpecAll_sel_0_1_res_Avg.append(SpecAll_sel_0_1_res_AvgT, ignore_index=True, sort=False)

# add year
SpecAll_sel_0_1_res_Avg.loc[:,'Year']    = [ SpecAll_sel_0_1_res_Avg.Date[val][:4] for val in range(len(SpecAll_sel_0_1_res_Avg.Date)) ]

# add N content per ha (canopy level) - normally area based (derived using LAI, not possible in our case)
SpecAll_sel_0_1_res_Avg.loc[:,'N_kg_ha'] = (SpecAll_sel_0_1_res_Avg.DM_dt_ha * (SpecAll_sel_0_1_res_Avg.N_perc_DM/100)).tolist()
SpecAll_sel_0_1_res_Avg.loc[:,'N_kg_ha'] = SpecAll_sel_0_1_res_Avg.N_kg_ha * 100  # transform dt in kg
SpecAll_sel_0_1_res_Avg.loc[:,'CrudeProt_kg_ha'] = (SpecAll_sel_0_1_res_Avg.DM_dt_ha * (SpecAll_sel_0_1_res_Avg.CrudeProt_perc_DM/100)).tolist()
SpecAll_sel_0_1_res_Avg.loc[:,'CrudeProt_kg_ha'] = SpecAll_sel_0_1_res_Avg.CrudeProt_kg_ha * 100  # transform dt in kg

# get dataset in the right order
MetaT = ['Date', 'Year', 'Treatment', 'Replicate', 'DM_dt_ha', 'N_perc_DM', 'N_kg_ha', 'CrudeProt_perc_DM', 'CrudeProt_kg_ha']
NbrBands = SpecAll_sel_0_1_res_Avg.shape[1] - len(MetaT)
SpecAll_sel_0_1_res_Avg = SpecAll_sel_0_1_res_Avg.loc[:, MetaT+list(range(NbrBands)) ]

if( EXPORT_SHARED_TABLES ):
    os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\2_Ground_aux' )
    MetaT_ToSave = ['Date', 'Year', 'Treatment', 'Replicate', 'DM_dt_ha', 'N_perc_DM', 'N_kg_ha']
    SpecAll_sel_0_1_res_Avg_ToSave = SpecAll_sel_0_1_res_Avg.loc[:, MetaT_ToSave+list(range(NbrBands)) ]
    SpecAll_sel_0_1_res_Avg_ToSave.columns = MetaT_ToSave+[ 'Band_' + str(list(range(NbrBands))[val]+1) if (val > 9) else \
                                                            'Band_0' + str(list(range(NbrBands))[val]+1) for val in range(NbrBands) ]
    SpecAll_sel_0_1_res_Avg_ToSave.to_csv( '20220215_Traits_Spectra_PlotLevel.csv', sep=';', na_rep='NaN', header=True, index=False )

### Average per treatment

Combines spectra and traits at treatment level, used for the N traits. Set `EXPORT_SHARED_TABLES` to write the table shared in the dataset. *(Original cell 7.)*

In [ ]:
DataRef    = SpecAll_sel_0_1_res_Avg.Date.unique().tolist()
colRef     = ['Date', 'Year', 'Treatment', 'Replicate', 'DM_dt_ha', 'N_perc_DM', 'N_kg_ha', 'CrudeProt_perc_DM', 'CrudeProt_kg_ha']
colRefRed  = ['Date', 'Treatment', 'Year', 'DM_dt_ha', 'N_perc_DM', 'N_kg_ha', 'CrudeProt_perc_DM', 'CrudeProt_kg_ha']

for ii in range( len(DataRef) ):  # date
    TreatRef = sorted( SpecAll_sel_0_1_res_Avg.loc[SpecAll_sel_0_1_res_Avg.Date == DataRef[ii], :].Treatment.unique() )
    for jj in pbar.progressbar( range(len(TreatRef)) ):  # treat
        DataSelT = SpecAll_sel_0_1_res_Avg.loc[ (SpecAll_sel_0_1_res_Avg.Date      == DataRef[ii]) & \
                                                (SpecAll_sel_0_1_res_Avg.Treatment == TreatRef[jj]), :]
        DataSelT1 = DataSelT.loc[:, ['DM_dt_ha', 'N_perc_DM', 'N_kg_ha', 'CrudeProt_perc_DM', 'CrudeProt_kg_ha'] ].mean().values  # traits
        DataSelT2 = DataSelT.loc[:, range(SpecAll_sel_0_1_res_Avg.shape[1]-len(colRef)) ].mean().values  # spectra
        dict1 = dict( enumerate( np.transpose(DataSelT2) ) )
        dict2 = { (list(range(SpecAll_sel_0_1_res_Avg.shape[1]-len(colRef))))[x]: dict1[x] for x in dict1.keys() }
        dict2['Date']              = DataRef[ii]
        dict2['Treatment']         = TreatRef[jj]
        dict2['Year']              = int(str(DataRef[ii])[:4])
        dict2['DM_dt_ha']          = DataSelT1[0]
        dict2['N_perc_DM']         = DataSelT1[1]
        dict2['N_kg_ha']           = DataSelT1[2]
        dict2['CrudeProt_perc_DM'] = DataSelT1[3]
        dict2['CrudeProt_kg_ha']   = DataSelT1[4]
        DataSel_temp = pd.DataFrame( dict2, index=range(1), columns=colRefRed+list(range(SpecAll_sel_0_1_res_Avg.shape[1]-len(colRef))) )
        if all([(ii==0), (jj==0)]):
            SpecAll_sel_0_1_res_Avg_treat = DataSel_temp.copy(deep=True)
        else:
            SpecAll_sel_0_1_res_Avg_treat = SpecAll_sel_0_1_res_Avg_treat.append( DataSel_temp, ignore_index=True )

if( EXPORT_SHARED_TABLES ):
    os.chdir( r'<PROJECT_ROOT>\DataProcessing\2_Data\2_Ground_aux' )
    MetaT_ToSave = ['Date', 'Year', 'Treatment', 'DM_dt_ha', 'N_perc_DM', 'N_kg_ha']
    SpecAll_sel_0_1_res_Avg_treat_ToSave = SpecAll_sel_0_1_res_Avg_treat.loc[:, MetaT_ToSave+list(range(NbrBands)) ]
    SpecAll_sel_0_1_res_Avg_treat_ToSave.columns = MetaT_ToSave+[ 'Band_' + str(list(range(NbrBands))[val]+1) if (val > 9) else \
                                                                  'Band_0' + str(list(range(NbrBands))[val]+1) for val in range(NbrBands) ]
    SpecAll_sel_0_1_res_Avg_treat_ToSave.to_csv( '20220215_Traits_Spectra_TreatLevel.csv', sep=';', na_rep='NaN', header=True, index=False )

### PCA of each date

Calculates the principal components used to cluster the samples of each target date. *(Original cell 8.)*

In [ ]:
col_ref = list( range(len(center_origin)) )

# set center and scaling modes - using robust scaler for the moment
centerOptPCA = True
scaleOptPCA  = False

# random seed
DateRefT   = SpecAll_sel_0_1_res_Avg.Date.unique().tolist()
np.random.seed(seed=965327)
seedPCACombAll = np.random.uniform( low=0, high=1000000000, size=len(DateRefT) ).astype(int)

PCAmOM_scoresAll    = []  # average at the plot level
PCAsumm             = []
PCAmOM_scoresAllRed = []  # average at treatment level
PCAsummRed          = []
for ii in pbar.progressbar( range(len(DateRefT)) ):  # acquisition date

    # average at the plot level
    dataT   = SpecAll_sel_0_1_res_Avg.loc[ SpecAll_sel_0_1_res_Avg.Date == DateRefT[ii], col_ref ].copy(deep=True)
    scalerT = sklearn.preprocessing.RobustScaler(with_centering=centerOptPCA, with_scaling=scaleOptPCA, quantile_range=(25.0, 75.0), copy=True)  # only centering
    scalerT.fit( dataT )
    dataT   = scalerT.transform( dataT )
    ncompF = np.min( [SpecAll_sel_0_1_res_Avg.loc[:,col_ref].shape[1], dataT.shape[0]] )  # select max number of features or observations as number of PCs
    PCAm   = sklearn.decomposition.PCA(n_components=ncompF, copy=True, whiten=False, svd_solver='auto',
                                       tol=0.0, iterated_power='auto', random_state=seedPCACombAll[ii])
    PCAmO    = PCAm.fit( dataT )
    eigenVec = PCAmO.components_
    eigenVal = np.reshape( PCAmO.explained_variance_, (ncompF,1))
    ExpVar   = np.reshape(PCAmO.explained_variance_ratio_, (ncompF,1))
    PCAsummTemp = np.hstack( (copy.deepcopy(ExpVar), copy.deepcopy(eigenVal), copy.deepcopy(eigenVec)) )
    PCAmOM_scoresAll.append( copy.deepcopy(PCAmO.transform( dataT )) )
    PCAsumm.append( copy.deepcopy(PCAsummTemp) )

    # average at the treatment level
    dataT  = SpecAll_sel_0_1_res_Avg_treat.loc[ SpecAll_sel_0_1_res_Avg_treat.Date == DateRefT[ii], col_ref ]
    scalerT.fit( dataT )
    dataT   = scalerT.transform( dataT )
    ncompF = np.min( [SpecAll_sel_0_1_res_Avg_treat.loc[:,col_ref].shape[1], dataT.shape[0]] )  # select max number of features or observations as number of PCs
    PCAm   = sklearn.decomposition.PCA(n_components=ncompF, copy=True, whiten=False, svd_solver='auto',
                                       tol=0.0, iterated_power='auto', random_state=seedPCACombAll[ii])
    PCAmO    = PCAm.fit( dataT )
    eigenVec = PCAmO.components_
    eigenVal = np.reshape( PCAmO.explained_variance_, (ncompF,1))
    ExpVar   = np.reshape(PCAmO.explained_variance_ratio_, (ncompF,1))
    PCAsummTemp = np.hstack( (copy.deepcopy(ExpVar), copy.deepcopy(eigenVal), copy.deepcopy(eigenVec)) )
    PCAmOM_scoresAllRed.append( copy.deepcopy(PCAmO.transform( dataT )) )
    PCAsummRed.append( copy.deepcopy(PCAsummTemp) )

## 3. Model transfer

### Baseline outputs

Reads the outputs of notebook 02, which give the starting uncertainty. *(Original cell 10.)*

In [ ]:
# import outputs of PLSR part I - currently only '01_SpectralAnalysis_Part00.ipynb' and '01_SpectralAnalysis_Part02_01.ipynb' are relevant
os.chdir( r'<PROJECT_ROOT>\1_JypiterLabProcessing\5_Outputs' )

versionRef = '01'

# open
with open('CoefBooF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    CoefBooF_allO_PLS = pickle.load(f)

with open('InterBooF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    InterBooF_allO_PLS = pickle.load(f)

with open('PLSRFactF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    PLSRFactF_allO_PLS = pickle.load(f)

with open('R2EvalF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    R2EvalF_allO_PLS = pickle.load(f)

with open('RMSEEvalF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    RMSEEvalF_allO_PLS = pickle.load(f)

with open('R2ValF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    R2ValF_allO_PLS = pickle.load(f)

with open('RMSEValF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    RMSEValF_allO_PLS = pickle.load(f)

with open('TestUnF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    TestUnF_allO_PLS = pickle.load(f)

with open('ValiUnF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    ValiUnF_allO_PLS = pickle.load(f)

with open('OutDUnF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    OutDUnF_allO_PLS = pickle.load(f)

with open('VIPOUnF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    VIPOUnF_allO_PLS = pickle.load(f)

with open('PredMapF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    PredMapF_allO_PLS = pickle.load(f)

with open('sampleValF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    sampleValF_allO_PLS = pickle.load(f)

with open('sampleBooF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    sampleBooF_allO_PLS = pickle.load(f)

### Parameters

Settings of the run used in the article (v106). `EmodeRef` chooses between selecting with uncertainty (`stdS`) or by diversity only (`centroid`); the first must run first. *(Original cell 12.)*

In [ ]:
# Variable Importance in Projection - formulation for VIP calculation   # https://github.com/scikit-learn/scikit-learn/issues/7050
def vip(model):
    t = model.x_scores_
    w = model.x_weights_
    q = model.y_loadings_
    p, h = w.shape
    vips = np.zeros((p,))
    s = np.diag(t.T @ t @ q.T @ q).reshape(h, -1)
    total_s = np.sum(s)
    for i in range(p):
        weight  = np.array([ (w[i,j] / np.linalg.norm(w[:,j]))**2 for j in range(h) ])
        vips[i] = np.sqrt(p*(s.T @ weight)/total_s)
    return vips

# max and min nbr of factors to use
nbrFactMax_thrshld = 10
nbrFactMin         = 3

TraitRef  = ['DM_dt_ha', 'N_perc_DM', 'N_kg_ha' ]  # reduced version of the traits - since crude protein seems to be estimated from N content
TraitRefL = ['Dry matter [dt/ha]', 'N [%]', 'N [kg/ha]']

DateRef  = SpecAll_sel_0_1_res_Avg.Date.unique().tolist()
DateRefY = np.unique( [ int(str(DateRef[val])[:4]) for val in range(len(DateRef)) ] ).tolist()

col_ref = list( range(len(center_origin)) )

# seed for calib and valid
seedAll = [325685, 73256841, 568]

# seed for CV repetition
BootSNbr   = 100 # nbr of resampling repetitions
seedRepTmF = []
np.random.seed(seed=23577)
for elmnt in range( len(TraitRef) ):  # traits
    seedRepTmF.append( np.random.uniform(low=0, high=1000000000, size=BootSNbr) )
    seedRepTmF[-1] = [ int(round(seedRepTmF[-1][val])) for val in range(len(seedRepTmF[-1])) ]

# random seed for k-means - part I
seedKmeanCombAll = []
np.random.seed(seed=27581425)
for elmnt in range( len(DateRefY) ):  # dataset source |  year
    if( elmnt == 0) :
        DateOutRefT =  HarvRef[2:5]
    else:
        DateOutRefT =  HarvRef[0:2]
    seedKmeanCombAllT = []
    for elmnt1 in range( len(TraitRef) ):  # trait  # nbr of samples available for a given trait | large or small set 0:2
        seedKmeanCombAllT.append( np.random.randint(low=0, high=1000000000, size=len(DateOutRefT)) )
    seedKmeanCombAll.append( seedKmeanCombAllT )

ScaleOpt = True
# indicate how 'uncertainty' is estimated while selecting the best number of samples to add from target domain
modeRef  = 'mean'
# attention - this parameter also affects feature selection - use 'stdS' in this case
EmodeRef = 'stdS'  # 'centroid' = select external samples according to cluster applied to the spectral data - closest sample to centroid ;
                   # 'stdS' = select samples according to 'uncertainty' criterion - largest uncertainty within the cluster
                   # obs.: run first 'stdS', because best sample selection is only performed based on this approach
if( EmodeRef == 'centroid' ):  # load best number of samples selected based on stdS (to compare outputs)
    os.chdir( r'<PROJECT_ROOT>\1_JypiterLabProcessing\5_Outputs' )
    versionRef = '106'  # latest version in which 'stdS' was saved (run 'stdS' first)
    # load
    with open('bestSampleSelF_allO_BootPLS_spike_opt_v' + versionRef + '_' + 'stdS' + '.pkl', 'rb') as f:
        bestSampleSelF_allO = pickle.load(f)

# maximum nbr of samples to add to the training data ( proportion from external data )
maxPropSamples = 1/2

# number of samples to add at the time - indicate how many clusters are adopted in k-means - starting from 2:
clstCntrlRef = 2

# threshold for selecting dataset providing satisfactory 'uncertainty' reduction
selThrshld   = 0.200

# threshold - 2nd criteria RMSE
selThrshldRMSE   = 0.300

# threshold for selecting PCA factors
selThrshldPCA   = 0.995

# select the criteria used to select optimal nbr of samples  --  'UncO' == uncertainty only ;
# 'UncRMSETO' == uncertainty + RMSE for out-of-bag sample + uncertainty (min uncertainty) ;
# 'UncRMSEO'  == uncertainty + RMSE for out-of-bag samples (min RMSE) ;
# 'UncRMSEOO' == uncertainty + RMSE for out-of-bag samples (min nbr fo samples)
# 'UncAvgO'   == closest to average uncertainty + min nbr of samples/uncertainty
criteriaOptT = 'UncAvgO'
if( criteriaOptT == 'UncAvgO' ): # select nbr of samples given minimum nbr of observations or uncertainty
    minSelCritOpt = 'MidSamp'

### Active selection of samples

For each target date, repeatedly clusters the samples, adds the most uncertain sample of each cluster to the training data and refits the 100 models, then chooses the number of samples to retain. *(Original cell 13.)*

In [ ]:
InterBooF_all        = []
PLSRFactF_all        = []
R2EvalF_all          = []
RMSEEvalF_all        = []
R2EEvalF_all         = []
RMSEEEvalF_all       = []
R2ValF_all           = []
RMSEValF_all         = []
R2OutF_all           = []
RMSEOutF_all         = []
R2IndF_all           = []
RMSEIndF_all         = []
R2IndVF_all          = []
RMSEIndVF_all        = []
bestSampleSelF_all   = []
TestUnF_all          = []
ETestUnF_all         = []
EvalUnF_all          = []
ValiUnF_all          = []
OutDUnF_all          = []
VIPOUnF_all          = []
CoefUnF_all          = []
sampleValF_all       = []
selSampleRefAllF_all = []
sampleBooF_all       = []
for ii in pbar.progressbar( range( len(DateRefY) ) ):  # dataset source |  year
    if( ii == 0 ) :
        DateOutRefT =  HarvRef[2:5]
    else:
        DateOutRefT =  HarvRef[0:2]
    InterBooF        = []
    PLSRFactF        = []
    R2EvalF          = []
    RMSEEvalF        = []
    R2EEvalF         = []
    RMSEEEvalF       = []
    R2ValF           = []
    RMSEValF         = []
    R2OutF           = []
    RMSEOutF         = []
    R2IndF           = []
    RMSEIndF         = []
    R2IndVF          = []
    RMSEIndVF        = []
    bestSampleSelF   = []
    TestUnF          = []
    ETestUnF         = []
    EvalUnF          = []
    ValiUnF          = []
    OutDUnF          = []
    VIPOUnF          = []
    CoefUnF          = []
    sampleValF       = []
    selSampleRefAllF = []
    sampleBooF       = []
    for jj in range( len(TraitRef) ):  # trait
        if( jj == 0 ): # Dry Biomass data / 49 observations per date
            DataP    = SpecAll_sel_0_1_res_Avg.loc[ (SpecAll_sel_0_1_res_Avg.Year).astype(int) == int(DateRefY[ii]), :]  # source data
            GroupF   = [ str(DataP.Replicate.tolist()[val]) + '_' + str(DataP.Treatment.tolist()[val]) for val in range(DataP.shape[0]) ]  # reference to each plot...
            DataPOut = SpecAll_sel_0_1_res_Avg.loc[ ~((SpecAll_sel_0_1_res_Avg.Year).astype(int) == int(DateRefY[ii])), :]  # data outside the source domain (different season)
            DataP_PC = copy.deepcopy( PCAmOM_scoresAll )
        else:  # N data / 15 observations per date
            DataP    = SpecAll_sel_0_1_res_Avg_treat.loc[ (SpecAll_sel_0_1_res_Avg_treat.Year).astype(int) == int(DateRefY[ii]), :]
            GroupF   = DataP.Treatment.tolist()
            DataPOut = SpecAll_sel_0_1_res_Avg_treat.loc[~((SpecAll_sel_0_1_res_Avg_treat.Year).astype(int) == int(DateRefY[ii])), :]
            DataP_PC = copy.deepcopy( PCAmOM_scoresAllRed )
        DataM_df    = DataP.loc[ :, (DataP.columns == TraitRef[jj]) | DataP.columns.isin(col_ref) ]  # select trait of interest + spectra
        DataPOut_df = DataPOut.loc[ :, (DataPOut.columns == TraitRef[jj]) | DataPOut.columns.isin(col_ref) ]  # data outside the source domain (different season/year)
        BRef = [ 'b' + str(range(len(col_ref))[val]) if range(len(col_ref))[val] > 9 else 'b0' + str( range(len(col_ref) )[val]) for val in range(len(col_ref)) ]
        DataM_df.columns = ['Trait'] + BRef
        DataM_df = DataM_df.reset_index(drop=True)
        DataPOut_df.columns = ['Trait'] + BRef
        DataPOut_df = DataPOut_df.reset_index(drop=True)

        # select calibration and validation datasets - source domain | valid = 0.3*n
        gsTV = sklearn.model_selection.GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seedAll[ii])
        trainIF_Oall = []
        testIF_Oall  = []
        for trainI, testI in gsTV.split(DataM_df.loc[:,BRef], DataM_df.Trait, GroupF):
            trainIF_Oall.append( trainI )
            testIF_Oall.append( testI )
        trainIF_Oall = trainIF_Oall[0].tolist()
        testIF_Oall  = testIF_Oall[0].tolist()

        # save reference to validation set - attention: based on row nbr not index
        sampleValF.append( testIF_Oall )

        # get calibration and validation sets
        Cal_df   = DataM_df.iloc[ trainIF_Oall,: ]
        GroupFTr = [ GroupF[val] for val in range(len(GroupF)) if val in trainIF_Oall ]
        Val_df   = DataM_df.iloc[ testIF_Oall,: ]
        GroupFTt = [ GroupF[val] for val in range(len(GroupF)) if val in testIF_Oall ]

        # Create arrays to store predictions - focus on avrg at the plot level | maps done later on (after optimal parameters are selected)
        TestUn  = []
        ETestUn = []
        EvalUn  = []
        ValiUn  = []
        OutDUn  = []
        VIPOUn  = []
        CoefUn  = []
        for val in range( len(DateOutRefT) ):   # date in the outside dataset (target)
            # Nbr of observations - selected data for a given target date
            DataPOutSelDimT = DataPOut_df.loc[ DataPOut.Date.isin( [ np.array(DateOutRefT[val]).astype('str').tolist() ] ).values ].shape[0]
            # max nbr of samples - use even or odd nbr to allow selection of 2: observation at the time (combine represetativity + uncertainty as sample selection criteria)
            MaxNbrSmplsOutRefTemp = int( np.ceil( DataPOutSelDimT*maxPropSamples ) )
            if( clstCntrlRef % 2 == 0 ):   # if nbr of samples to add at the time is even
                MaxNbrSmplsOutRefTemp = MaxNbrSmplsOutRefTemp if (MaxNbrSmplsOutRefTemp % 2 == 0) else (MaxNbrSmplsOutRefTemp-1)
            else:  # if nbr of samples to add at the time is odd
                MaxNbrSmplsOutRefTemp = (MaxNbrSmplsOutRefTemp-1) if (MaxNbrSmplsOutRefTemp % 2 == 0) else MaxNbrSmplsOutRefTemp
            TestUnT  = []
            ETestUnT = []
            EvalUnT  = []
            ValiUnT  = []
            OutDUnT  = []
            VIPOUnT  = []
            CoefUnT  = []
            for val1 in range( int(MaxNbrSmplsOutRefTemp/clstCntrlRef) ):   # nbr of samples to add from the external dataset (target domain)  -  progressive increase
                                                                            # depending on: acq | dataset size
                TestUnT.append( np.full( (DataM_df.shape[0], len(seedRepTmF[jj])), np.nan ) )  # samples in the calib x repetitions (boot)
                EvalUnT.append( np.full( (DataM_df.shape[0], len(seedRepTmF[jj])), np.nan ) )
                ValiUnT.append( np.full( (DataM_df.shape[0], len(seedRepTmF[jj])), np.nan ) )
                ETestUnT.append( np.full( (DataPOut_df.shape[0], len(seedRepTmF[jj])), np.nan ) )   # extra samples in the calib x repetitions (boot) - using full array dimensions (standard dimensions)
                OutDUnT.append( np.full(  (DataPOut_df.shape[0], len(seedRepTmF[jj])), np.nan ) )
                VIPOUnT.append( np.full( (len(BRef), len(seedRepTmF[jj])), np.nan ) )
                CoefUnT.append( np.full( (len(BRef), len(seedRepTmF[jj])), np.nan ) )
            TestUn.append( TestUnT  )
            ETestUn.append( ETestUnT )
            EvalUn.append( EvalUnT )
            ValiUn.append( ValiUnT )
            OutDUn.append( OutDUnT )
            VIPOUn.append( VIPOUnT )
            CoefUn.append( CoefUnT )

        InterBooTT      = []
        PLSRFactTT      = []
        R2EvalTT        = []
        RMSEEvalTT      = []
        R2EEvalTT       = []
        RMSEEEvalTT     = []
        R2ValTT         = []
        RMSEValTT       = []
        R2OutTT         = []
        RMSEOutTT       = []
        R2IndTT         = []
        RMSEIndTT       = []
        R2IndVTT        = []
        RMSEIndVTT      = []
        sampleBooTT     = []
        bestSampleSelTT = []
        Cal_dfUpInT       = []
        selSampleRefAllTT = []
        for kk in range( len(DateOutRefT) ):   # date in the outside dataset (target)
            DataPOutSelT = DataPOut_df.loc[ DataPOut.Date.isin( [ np.array(DateOutRefT[kk]).astype('str').tolist() ] ).values ]  # selected dataset for a given target date
            # max nbr of samples - use nbr to allow selection of 2: observation at the time (combine diversity + uncertainty as sample selection criteria)
            MaxNbrSmplsOutRefTemp = int( np.ceil( DataPOutSelT.shape[0]*maxPropSamples ) )
            if( clstCntrlRef % 2 == 0 ):   # if nbr of samples to add at the time is even
                MaxNbrSmplsOutRefTemp = MaxNbrSmplsOutRefTemp if (MaxNbrSmplsOutRefTemp % 2 == 0) else (MaxNbrSmplsOutRefTemp-1)
            else:  # if nbr of samples to add at the time is odd
                MaxNbrSmplsOutRefTemp = (MaxNbrSmplsOutRefTemp-1) if (MaxNbrSmplsOutRefTemp % 2 == 0) else MaxNbrSmplsOutRefTemp
            if( jj == 0 ):  # larger dataset - DM
                CompKeep     = np.where( np.cumsum(PCAsumm[ HarvRef.index(DateOutRefT[kk]) ][:,0]) >= selThrshldPCA )[0].min()
            else:   # smaller dataset - N
                CompKeep     = np.where( np.cumsum(PCAsummRed[ HarvRef.index(DateOutRefT[kk]) ][:,0]) >= selThrshldPCA )[0].min()
            CompSel      = CompKeep+1 if CompKeep+1 > 3 else 3  # select at least 3 components
            DataP_PCSelT = DataP_PC[ HarvRef.index(DateOutRefT[kk]) ][ :, 0:CompSel ]

            # random seed for k-means - part II
            np.random.seed(seed=seedKmeanCombAll[ii][jj][kk])
            seedKmeanCombAllF = np.random.randint(low=0, high=1000000000, size=int(MaxNbrSmplsOutRefTemp/clstCntrlRef)).astype(int)

            InterBooT  = []
            PLSRFactT  = []
            R2EvalT    = []
            RMSEEvalT  = []
            R2EEvalT   = []
            RMSEEEvalT = []
            R2ValT     = []
            RMSEValT   = []
            R2OutT     = []
            RMSEOutT   = []
            R2IndT     = []
            RMSEIndT   = []
            R2IndVT    = []
            RMSEIndVT  = []
            sampleBooT = []
            selSampleRefAllT = []
            for ll in range( int(MaxNbrSmplsOutRefTemp/clstCntrlRef) ):   # nbr of samples to add from the external dataset (target domain) >> progressive seletion
                                                                                              # depending on: acq / dataset size (proportional)

                # uncertainty for samples from the outside dataset >> already modelled previously (PLSR part) or based on previous selection loop (ll)
                if( ll == 0 ):  # all samples
                    DataPOutSelTT = DataPOutSelT.copy(deep=True)
                    DataP_PCSelTT = copy.deepcopy( DataP_PCSelT )
                    # uncertainty for samples available
                    stdOutAllT    = scipy.stats.iqr( OutDUnF_allO_PLS[ii][jj], nan_policy='omit', axis=1 ).tolist() if modeRef == 'med' else np.nanstd( OutDUnF_allO_PLS[ii][jj], axis=1 ).tolist()
                    stdOutSelTT   = np.array(stdOutAllT)[ DataPOut.Date.isin( [ np.array(DateOutRefT[kk]).astype('str').tolist() ] ).values ]
                else:
                    # remove samples selected previously - incremental dataset
                    DataPOutSelTT = DataPOutSelT[ ~DataPOutSelT.index.isin( list(itertools.chain(*selSampleRefAllT)) ) ]  # samples still available
                    DataP_PCSelTT = DataP_PCSelT[ ~DataPOutSelT.index.isin( list(itertools.chain(*selSampleRefAllT)) ) ]  # PCs for samples still available ;
                                                                                                                          # using representation derived for the complete dataset
# for elmnt in range(DataPOutSelT.shape[0]) ]
                    # uncertainty for samples still available
                    stdOutSelT  = OutDUn[kk][int(ll-1)][ DataPOut.Date.isin( [ np.array(DateOutRefT[kk]).astype('str').tolist() ] ).values ]
                    stdOutSelT  = scipy.stats.iqr( stdOutSelT, nan_policy='omit', axis=1 ).tolist() if modeRef == 'med' else np.nanstd( stdOutSelT, axis=1 ).tolist()
                    stdOutSelTT = np.array(stdOutSelT)[ ~DataPOutSelT.index.isin( list(itertools.chain(*selSampleRefAllT)) ) ]

                # fit k-means and assign each sample to a cluster (predict)
                KmeansM = sklearn.cluster.KMeans(n_clusters=clstCntrlRef, init='k-means++', n_init=10, max_iter=300, tol=0.0001,
                                                 verbose=0, copy_x=True, algorithm='auto', random_state=seedKmeanCombAllF[ll] )
                KmeansMO    = KmeansM.fit( DataP_PCSelTT )
                KmeansMPred = KmeansMO.predict( DataP_PCSelTT )

                selSampleRefAllTemp = []
                for elmnt in range( clstCntrlRef ):  # number of samples to select at the time - n:
                    DataPOutSelTTT = DataPOutSelTT.loc[ (KmeansMPred == elmnt) ]  # select elmnts in a given cluster
                    DataP_PCSelTTT = DataP_PCSelTT[     (KmeansMPred == elmnt) ]
                    stdOutSelTTT   = stdOutSelTT[       (KmeansMPred == elmnt) ]  # uncertainty for samples in the cluster
                    clstCntrT      = DataP_PCSelTTT.mean(axis=0).reshape(1, DataP_PCSelTTT.shape[1])  # cluster centroid
                    if( EmodeRef == 'centroid' ):   # select sample closest to the cluster centroid - opt1
                        distClstrOT     = scipy.spatial.distance.cdist( DataP_PCSelTTT, clstCntrT )
                        selSampleClstrT = np.where(distClstrOT == np.nanmin(distClstrOT))[0][0]
                        selSampleRefAllTemp.append( DataPOutSelTTT.index.tolist()[ selSampleClstrT ] )  # final sample selected in the outside dataset | target
                    else:   # select sample with largest uncertainty - opt2
                        selSampleStdT   = np.where( stdOutSelTTT == np.nanmax(stdOutSelTTT) )[0][0]
                        selSampleRefAllTemp.append( DataPOutSelTTT.index.tolist()[ selSampleStdT ] )
                selSampleRefAllT.append( selSampleRefAllTemp )

                # add samples from the external dataset - incremental
                if( all([ (ll==0) ]) ):
                    Cal_dfUpInT     = Cal_dfUpInT + [ DataPOutSelT.loc[ selSampleRefAllT[ll], :] ]
                else:
                    Cal_dfUpInTemp  = DataPOutSelT.loc[ selSampleRefAllT[ll], :]
                    Cal_dfUpInT[kk] = Cal_dfUpInT[kk].append( Cal_dfUpInTemp )  # not ignoring indices >> keep original row numbers  -  used for reference

                # Bootstrap internal repetition
                InterBooTemp  = []
                PLSRFactTemp  = []
                R2EvalTemp    = []
                RMSEEvalTemp  = []
                R2EEvalTemp   = []
                RMSEEEvalTemp = []
                R2ValTemp     = []
                RMSEValTemp   = []
                R2OutTemp     = []
                RMSEOutTemp   = []
                R2IndTemp     = []
                RMSEIndTemp   = []
                R2IndVTemp    = []
                RMSEIndVTemp  = []
                sampleBooTemp = []
                for mm in range( len(seedRepTmF[jj]) ):  # repetition for bootstrap (according to trait)

                    # get model fitting and test sets >> in and out of bag samples - only source data
                    Cal_dfT   = Cal_df.copy(deep=True)
                    GroupFTrT = copy.deepcopy(GroupFTr)
                    sizeRef   =  int( (Cal_dfT.shape[0]*0.3333) / len(DataP.Date.unique().tolist()) )
                    # divide by nbr of acquistions since its grouped by this factor and the selected samples will be multiplied by the same factor
                    gsBoo = sklearn.model_selection.GroupShuffleSplit(n_splits=1, test_size=sizeRef, random_state=seedRepTmF[jj][mm])
                    for trainI, testI in gsBoo.split(Cal_dfT.loc[:,BRef], Cal_dfT.Trait, GroupFTrT):
                        testIF    = testI.tolist()
                        testIFidx = Cal_dfT.iloc[testIF,:].index.tolist()
                    # save reference to test set within each boot repetition - attention: based on row nbr not index
                    sampleBooTemp.append( testIF )
                    Cal_dfT      = Cal_df.drop( labels = testIFidx )
                    GroupFTrTidx = [ val for val in range(Cal_df.shape[0]) if Cal_df.index.tolist()[val] not in testIFidx ]
                    GroupFTrT    = [ GroupFTr[val] for val in GroupFTrTidx ]
                    Eval_dfT     = Cal_df.loc[ testIFidx, :]

                    # add samples to the training data
                    Cal_dfUpT      = Cal_dfT.copy(deep=True)  # updated dataset
                    DataPOut_dfUpT = []
                    for elmnt in range( len(Cal_dfUpInT) ):  # date in the outside dataset (target)
                        DataPOutSelT = DataPOut_df.loc[ DataPOut.Date.isin( [ np.array(DateOutRefT[elmnt]).astype('str').tolist() ] ).values ]  # selected data for a given target date - outside samples
                        if( elmnt != kk ):  # previous dates
                            selSampleRefAllTT_sel = itertools.chain(*selSampleRefAllTT[elmnt][:(bestSampleSelTT[elmnt]+1) ])
                            selSampleRefAllTT_sel = [ True if Cal_dfUpInT[elmnt].index.tolist()[val] in selSampleRefAllTT_sel else False for val in range( Cal_dfUpInT[elmnt].shape[0] ) ]
                            Cal_dfUpT = Cal_dfUpT.append( Cal_dfUpInT[elmnt].loc[ selSampleRefAllTT_sel,: ], ignore_index=True )  # only optimal nbr of samples (for previous dates) - added at the end of the DataFrame
                            OutSampleIdxRefT = Cal_dfUpInT[elmnt].loc[ selSampleRefAllTT_sel,: ].index.tolist()  # samples included in the calibration model
                        else:  # current date
                            Cal_dfUpT = Cal_dfUpT.append( Cal_dfUpInT[elmnt], ignore_index=True )   # all samples selected until now (for current date)
                            OutSampleIdxRefT = Cal_dfUpInT[elmnt].index.tolist()  # samples included in the calibration model
                        # remove selected samples from available pool - incremental (date by date)
                        if( len(DataPOut_dfUpT) == 0 ):
                            DataPOut_dfUpT = DataPOutSelT.loc[ [ True if DataPOutSelT.index.tolist()[val] not in OutSampleIdxRefT else False for val in range(DataPOutSelT.shape[0]) ],: ]
                        else:
                            DataPOut_dfUpT = DataPOut_dfUpT.append( DataPOutSelT.loc[ [ True if DataPOutSelT.index.tolist()[val] not in OutSampleIdxRefT else False for val in range(DataPOutSelT.shape[0]) ],: ], \
                                                                    ignore_index=False )

                    # outputs according to number of PLS factors - not saved for posterior use , only outputs for best model are retained
                    RMSETO     = []
                    R2TO       = []
                    RMSETO_Out = []
                    R2TO_Out   = []
                    nbrFactMax = np.min( [ min(Cal_dfUpT.shape), nbrFactMax_thrshld ] )  # maximun nbr of PLS factors should be <= nbr of samples
                    for elmnt in range( (nbrFactMax-nbrFactMin)+1 ):  # nbr of PLS factors to try
                        plsM = PLSRegression( n_components=range(nbrFactMin, nbrFactMax+1, 1)[elmnt], scale=ScaleOpt )
                        plsM.fit( Cal_dfUpT.loc[:,BRef].values,  Cal_dfUpT.loc[:,'Trait'].values.reshape( (Cal_dfUpT.loc[:,'Trait'].shape[0], 1) ) )
                        # evaluation over the source domain
                        PredEvalO = plsM.predict(Eval_dfT.loc[:,BRef].values).reshape(Eval_dfT.shape[0],).tolist()
                        ResT =  Eval_dfT.Trait - np.array(PredEvalO)
                        uVal = (ResT**2).sum()
                        vVal = ((Eval_dfT.Trait.values.reshape(-1, 1) - Eval_dfT.Trait.values.reshape(-1, 1).mean())**2).sum()
                        R2TO.append( (1-(uVal/vVal)) )
                        RMSETO.append( np.sqrt( uVal/Eval_dfT.shape[0] ) )
                        # evaluation over the outside data in the source domain (calibration set - source)
                        DataPOut_df_EvalT = Cal_dfUpT.iloc[ (Cal_dfT.shape[0]):(Cal_dfUpT.shape[0]), :]  # only samples outside dataset - incremental
                        PredEvalO = plsM.predict(DataPOut_df_EvalT.loc[:,BRef].values).reshape(DataPOut_df_EvalT.shape[0],).tolist()
                        ResT =  DataPOut_df_EvalT.Trait - np.array(PredEvalO)
                        uVal = (ResT**2).sum()
                        vVal = ((DataPOut_df_EvalT.Trait.values.reshape(-1, 1) - DataPOut_df_EvalT.Trait.values.reshape(-1, 1).mean())**2).sum()
                        R2TO_Out.append( (1-(uVal/vVal)) )
                        RMSETO_Out.append( np.sqrt( uVal/DataPOut_df_EvalT.shape[0] ) )

                    # select factors | compromise between source and target accuracy
                    RMSETO_Out_sortedT = list( enumerate( RMSETO_Out ) )
                    RMSETO_Out_sortedT = sorted( RMSETO_Out_sortedT, key=lambda x:x[1], reverse=True )
                    RMSETO_Out_sortedTRef = [ RMSETO_Out_sortedT[val][0] for val in range(len(RMSETO_Out_sortedT)) ]
                    RMSETO_Out_sortedT    = [ RMSETO_Out_sortedT[val][1] for val in range(len(RMSETO_Out_sortedT)) ]
                    RMSETO_sortedT = [ RMSETO[ RMSETO_Out_sortedTRef[val] ] for val in range(len(RMSETO_Out_sortedTRef)) ]
                    RMSETO_joinT   = [ RMSETO_Out_sortedT[val]+RMSETO_sortedT[val] for val in range(len(RMSETO_Out_sortedT)) ]      # sum of RMSE for source and target domains
                    nbrFactSel = range(nbrFactMin, nbrFactMax+1, 1)[ np.where( (RMSETO_joinT == np.nanmin(RMSETO_joinT)) )[0][0] ]
                    PLSRFactTemp.append( nbrFactSel )
                    plsM = PLSRegression( n_components=nbrFactSel, scale=ScaleOpt )
                    plsM.fit( Cal_dfUpT.loc[:,BRef].values,  Cal_dfUpT.loc[:,'Trait'].values.reshape( (Cal_dfUpT.loc[:,'Trait'].shape[0], 1) ) )

                    # get predictions + model parameters
                    TestUn[kk][ll][ Cal_dfT.index.tolist(), mm] = plsM.predict(Cal_dfT.loc[:,BRef].values).reshape(len(Cal_dfT.index.tolist()),)  # same as >> plsM.predict(Cal_dfT.loc[:,BRef].values)
                                                                                                                                                  # source data - complete calibration set
                    # outside samples added to the training set
                    for elmnt in range( len(Cal_dfUpInT) ):  # date in the outside dataset (target)
                        # results from previous dates and current date are stored together - best set of samples for previous dates (need to be selected) + incremental dataset for current date
                        if( elmnt != kk ):  # previous dates
                            selSampleRefAllTT_sel = itertools.chain(*selSampleRefAllTT[elmnt][:(bestSampleSelTT[elmnt]+1) ])
                            selSampleRefAllTT_sel = [ True if Cal_dfUpInT[elmnt].index.tolist()[val] in selSampleRefAllTT_sel else False for val in range( Cal_dfUpInT[elmnt].shape[0] ) ]  # reference to best set of samples for previous dates
                            ETestUn[kk][ll][  Cal_dfUpInT[elmnt].loc[ selSampleRefAllTT_sel, :].index.tolist(), mm] = \
                                plsM.predict( Cal_dfUpInT[elmnt].loc[ selSampleRefAllTT_sel, BRef ].values ).reshape( Cal_dfUpInT[elmnt].loc[ selSampleRefAllTT_sel, :].shape[0], )
                        else:   # current date
                            ETestUn[kk][ll][ Cal_dfUpInT[elmnt].index.tolist(), mm] = plsM.predict( Cal_dfUpInT[elmnt].loc[:,BRef].values ).reshape( Cal_dfUpInT[elmnt].shape[0], )

                    EvalUn[kk][ll][testIFidx, mm] = plsM.predict( Eval_dfT.loc[:,BRef].values).reshape(len(testIFidx),)   # test samples - source dataset - boot repetition
                    ValiUn[kk][ll][ Val_df.index.tolist(), mm] = plsM.predict(Val_df.loc[:,BRef].values).reshape(len(Val_df.index.tolist()),)
                    OutDUn[kk][ll][ DataPOut_dfUpT.index.tolist(), mm] = plsM.predict( DataPOut_dfUpT.loc[:,BRef].values ).reshape( DataPOut_dfUpT.shape[0], )
                    VIPOUn[kk][ll][:, mm] = vip(plsM)
                    CoefUn[kk][ll][:, mm] = [ plsM.coef_.tolist()[val][0] for val in range(len(plsM.coef_.tolist())) ]
                    InterBooTemp.append( (plsM._y_mean - np.dot(plsM._x_mean, plsM.coef_)).tolist()[0] )
                    # https://stackoverflow.com/questions/41188410/how-obtain-the-intercept-of-the-pls-regression-sklearn

                    # R2 and RMSE for calib set - Eval (source) - out-of-bag samples
                    PredT = copy.deepcopy( EvalUn[kk][ll][testIFidx, mm] )
                    ResT = Eval_dfT.Trait.values - np.array( PredT )
                    uVal = (ResT**2).sum()
                    vVal = ((Eval_dfT.Trait.values.reshape(-1, 1) - Eval_dfT.Trait.values.reshape(-1, 1).mean())**2).sum()
                    R2EvalTemp.append( (1-(uVal/vVal)) )
                    RMSEEvalTemp.append( np.sqrt( uVal/Eval_dfT.shape[0] ) )

                    # R2 and RMSE for external dataset - added to the training data - EEval
                    PredT = copy.deepcopy( ETestUn[kk][ll][ :, mm] )
                    ECal_dfUpT    = np.empty( ETestUn[kk][ll].shape[0], )  # same as ETestUn[kk][ll][:,mm] but for traits not predictions
                    ECal_dfUpT[:] = np.nan
                    for elmnt in range( len(Cal_dfUpInT) ):  # date in the outside dataset (target)
                        # samples from previous dates and current date are stored together - best set of samples for previous dates (need to be selected) + incremental dataset (for current date)
                        if( elmnt != kk ):  # previous dates
                            selSampleRefAllTT_sel = itertools.chain(*selSampleRefAllTT[elmnt][:(bestSampleSelTT[elmnt]+1) ])
                            selSampleRefAllTT_sel = [ True if Cal_dfUpInT[elmnt].index.tolist()[val] in selSampleRefAllTT_sel else False for val in range( Cal_dfUpInT[elmnt].shape[0] ) ]  # reference to best set of samples for previous dates
                            ECal_dfUpT[ Cal_dfUpInT[elmnt].loc[ selSampleRefAllTT_sel, :].index.tolist(), ] = \
                                Cal_dfUpInT[elmnt].loc[ selSampleRefAllTT_sel,'Trait'].values.reshape( Cal_dfUpInT[elmnt].loc[ selSampleRefAllTT_sel, :].shape[0], )
                        else:   # current date
                            ECal_dfUpT[ Cal_dfUpInT[elmnt].index.tolist(), ] = Cal_dfUpInT[elmnt].loc[:,'Trait'].values.reshape( Cal_dfUpInT[elmnt].shape[0], )
                    ResT = ECal_dfUpT - np.array( PredT )
                    uVal = np.nansum( (ResT**2) )
                    vVal = np.nansum( ((ECal_dfUpT.reshape(-1, 1) - np.nanmean(ECal_dfUpT.reshape(-1, 1)))**2) )
                    R2EEvalTemp.append( (1-(uVal/vVal)) )
                    RMSEEEvalTemp.append( np.sqrt( uVal/(ECal_dfUpT.shape[0]-np.isnan(ECal_dfUpT).sum()) ) )

                    # R2 and RMSE - validation set - source
                    PredT = copy.deepcopy( ValiUn[kk][ll][ Val_df.index.tolist(), mm] )
                    ResT = Val_df.Trait.values - np.array( PredT )
                    uVal = (ResT**2).sum()
                    vVal = ((Val_df.Trait.values.reshape(-1, 1) - Val_df.Trait.values.reshape(-1, 1).mean())**2).sum()
                    R2ValTemp.append( (1-(uVal/vVal)) )
                    RMSEValTemp.append( np.sqrt( uVal/Val_df.shape[0] ) )

                    # R2 and RMSE - outside dataset - not included in the training data
                    PredT = copy.deepcopy( OutDUn[kk][ll][ DataPOut_dfUpT.index.tolist(), mm] )
                    ResT = DataPOut_dfUpT.Trait.values - np.array( PredT )
                    uVal = (ResT**2).sum()
                    vVal = ((DataPOut_dfUpT.Trait.values.reshape(-1, 1) - DataPOut_dfUpT.Trait.values.reshape(-1, 1).mean())**2).sum()
                    R2OutTemp.append( (1-(uVal/vVal)) )
                    RMSEOutTemp.append( np.sqrt( uVal/ DataPOut_dfUpT.shape[0] ) )

                    # R2 and RMSE - all independent data (source + target) - careful source samples are out-of-bag not validation
                    ResT = np.append( (DataPOut_dfUpT.Trait.values - OutDUn[kk][ll][ DataPOut_dfUpT.index.tolist(), mm]), \
                                      (Eval_dfT.Trait.values - EvalUn[kk][ll][ testIFidx, mm]) )
                    uVal = (ResT**2).sum()
                    vVal = (( np.append(DataPOut_dfUpT.Trait.values.reshape(-1, 1), Eval_dfT.Trait.values.reshape(-1, 1)) - \
                              np.append(DataPOut_dfUpT.Trait.values.reshape(-1, 1), Eval_dfT.Trait.values.reshape(-1, 1)).mean() )**2).sum()
                    R2IndTemp.append( (1-(uVal/vVal)) )
                    RMSEIndTemp.append( np.sqrt( uVal/ (DataPOut_dfUpT.shape[0]+Eval_dfT.shape[0]) ) )

                    # R2 and RMSE - all independent data (source + target) >> validation (source ; stable) + samples not included in the training (target ; dynamic)
                    ResT = np.append( (DataPOut_dfUpT.Trait.values - OutDUn[kk][ll][ DataPOut_dfUpT.index.tolist(), mm]), \
                                      (Val_df.Trait.values - ValiUn[kk][ll][ Val_df.index.tolist(), mm]) )
                    uVal = (ResT**2).sum()
                    vVal = (( np.append(DataPOut_dfUpT.Trait.values.reshape(-1, 1), Val_df.Trait.values.reshape(-1, 1)) - \
                              np.append(DataPOut_dfUpT.Trait.values.reshape(-1, 1), Val_df.Trait.values.reshape(-1, 1)).mean() )**2).sum()
                    R2IndVTemp.append( (1-(uVal/vVal)) )
                    RMSEIndVTemp.append( np.sqrt( uVal/ (DataPOut_dfUpT.shape[0]+Val_df.shape[0]) ) )

                InterBooT.append( InterBooTemp )
                PLSRFactT.append( PLSRFactTemp )
                R2EvalT.append( R2EvalTemp )
                RMSEEvalT.append( RMSEEvalTemp )
                R2EEvalT.append( R2EEvalTemp )
                RMSEEEvalT.append( RMSEEEvalTemp )
                R2ValT.append( R2ValTemp )
                RMSEValT.append( RMSEValTemp )
                R2OutT.append( R2OutTemp )
                RMSEOutT.append( RMSEOutTemp )
                R2IndT.append( R2IndTemp )
                RMSEIndT.append( RMSEIndTemp )
                R2IndVT.append( R2IndVTemp )
                RMSEIndVT.append( RMSEIndVTemp )
                sampleBooT.append( sampleBooTemp )

                # find best number of samples to retain for a given outside dataset (target) - only after final samples have been considered
                if( ll == int((MaxNbrSmplsOutRefTemp/clstCntrlRef)-1) ):

                    if( EmodeRef != 'centroid' ):
                        # estimate variance (not absolute error)
                        UncSummT       = []
                        UncSummCalT    = []
                        RMSEEvalT_Summ = []
                        RMSEIndVT_Summ = []  # all validation data
                        for elmnt in range( len(OutDUn[kk]) ):  # nbr of samples to add from the external dataset (target domain) >> progressive seletion
                            # samples from the target dateset
                            PredSamplMaskT = np.isnan( OutDUn[kk][elmnt][:,0] )
                            OutDUn_selT    = OutDUn[kk][elmnt][ ~PredSamplMaskT,:]
                            UncEstAllT     = scipy.stats.iqr(OutDUn_selT, axis=1, nan_policy='omit') if modeRef == 'med' else np.nanstd(OutDUn_selT, axis=1)
                            UncSummT.append( np.nanmedian(UncEstAllT) if modeRef == 'med' else np.nanmean(UncEstAllT) )
                            # RMSEIndVT >> RMSE for external data - cannot be known if samples havee not been analysed before hand

                           # samples from the source dateset - out-of-bag samples
                            PredSamplMaskT_calib = np.isnan( EvalUn[kk][elmnt][:,0] )
                            EvalUn_selT          = EvalUn[kk][elmnt][ ~PredSamplMaskT_calib,:]
                            UncEstAllT           = scipy.stats.iqr(EvalUn_selT, axis=1, nan_policy='omit') if modeRef == 'med' else np.nanstd(EvalUn_selT, axis=1)
                            UncSummCalT.append( np.nanmedian(UncEstAllT) if modeRef == 'med' else np.nanmean(UncEstAllT) )
                            RMSEEvalT_Summ.append( scipy.stats.iqr(RMSEEvalT[elmnt], axis=0, nan_policy='omit') if modeRef == 'med' else np.nanstd(RMSEEvalT[elmnt], axis=0) )  # RMSE for out of thebag samples
                            RMSEIndVT_Summ.append( scipy.stats.iqr(RMSEIndVT[elmnt], axis=0, nan_policy='omit') if modeRef == 'med' else np.nanstd(RMSEIndVT[elmnt], axis=0) )
                        UncSummT_all = [ UncSummT[val] + UncSummCalT[val] for val in range(len(UncSummT)) ]
                        if( criteriaOptT != 'UncAvgO' ):  # UncO, etc...
                            selThrshldCalcO = np.nanmin(UncSummT_all)+(selThrshld*(np.nanmax(UncSummT_all)-np.nanmin(UncSummT_all)))
                        else:
                            selThrshldCalcO = (selThrshld*(np.nanmax(UncSummT_all)-np.nanmin(UncSummT_all)))

                        if( criteriaOptT == 'UncO' ):  # UncO
                            bestSampleSelTT.append(  np.nanmin( np.where((np.array(UncSummT_all) <= selThrshldCalcO)) ) )
                        elif( criteriaOptT == 'UncRMSETO' ):  # UncRMSETO
                            # v2.1 - two criteria | uncertainty outside dataset + RMSE for out-of-bag samples - uncertainty considered twice
                            bestSampleSelToPT      = np.where((np.array(UncSummT_all) <= selThrshldCalcO))
                            if( bestSampleSelToPT[0].shape[0] > 1 ):  # select dataset with lowest RMSE for out-of-bag samples
                                RMSERMSEEvalT_SummSelT = [ RMSEEvalT_Summ[val] for val in range(len(RMSEEvalT_Summ)) if val in bestSampleSelToPT[0].tolist() ]
                                RMSEThrsldSelT = np.where( (RMSERMSEEvalT_SummSelT <=
                                                            (np.nanmin(RMSERMSEEvalT_SummSelT)+((np.nanmax(RMSEEvalT_Summ)-np.nanmin(RMSEEvalT_Summ))*selThrshldRMSE))) )[0].tolist()
                                RMSEThrsldSelT = [ RMSERMSEEvalT_SummSelT[val] for val in RMSEThrsldSelT ]
                                RMSEThrsldSelT = [ np.where( RMSEEvalT_Summ == RMSEThrsldSelT[val] )[0][0] for val in range(len(RMSEThrsldSelT)) ]
                                UncThrsldSelT  = [ UncSummT_all[val] for val in RMSEThrsldSelT ]
                                bestSampleSelTT.append( np.where( UncSummT_all == np.nanmin(UncThrsldSelT) )[0][0] )  # select nbr of samples resulting in the lowest uncertainty after RMSE ccriterion was considered
                            else:
                                bestSampleSelTT.append(  np.nanmin( np.where((np.array(UncSummT_all) <= selThrshldCalcO)) ) )  # same as v1 in this case - lowest uncertainty
                        elif( criteriaOptT == 'UncRMSEO' ):  # UncRMSEO
                            # v2.2 - two criteria | uncertainty outside dataset + RMSE for out-of-bag samples - uncertainty considered only once (min RMSE)
                            bestSampleSelToPT      = np.where((np.array(UncSummT_all) <= selThrshldCalcO))
                            if( bestSampleSelToPT[0].shape[0] > 1 ):  # select dataset with lowest RMSE for out-of-bag samples
                                RMSERMSEEvalT_SummSelT = [ RMSEEvalT_Summ[val] for val in range(len(RMSEEvalT_Summ)) if val in bestSampleSelToPT[0].tolist() ]
                                RMSERMSEEvalT_SummSelT = [ RMSERMSEEvalT_SummSelT[val] for val in np.where( (RMSERMSEEvalT_SummSelT <= (np.nanmin(RMSERMSEEvalT_SummSelT)+
                                                                                        ((np.nanmax(RMSERMSEEvalT_SummSelT)-np.nanmin(RMSERMSEEvalT_SummSelT))*selThrshldRMSE))) )[0].tolist() ]
                                # select nbr of samples considering minimum RMSE only
                                bestSampleSelTT.append( np.where( (RMSEEvalT_Summ == np.nanmin(RMSERMSEEvalT_SummSelT)) )[0][0] )
                            else:
                                bestSampleSelTT.append(  np.nanmin( np.where((np.array(UncSummT_all) <= selThrshldCalcO)) ) )  # same as v1 in this case - lowest uncertainty
                        elif( criteriaOptT == 'UncRMSEOO' ): # UncRMSEOO
                            # v2.3 - two criteria | uncertainty outside dataset + RMSE for out-of-bag samples - uncertainty considered only once (min nbr of samples)
                            bestSampleSelToPT      = np.where((np.array(UncSummT_all) <= selThrshldCalcO))
                            if( bestSampleSelToPT[0].shape[0] > 1 ):  # select dataset with lowest RMSE for out-of-bag samples
                                RMSERMSEEvalT_SummSelT = [ RMSEEvalT_Summ[val] for val in range(len(RMSEEvalT_Summ)) if val in bestSampleSelToPT[0].tolist() ]
                                # select minimum nbr of samples considering RMSE only
                                bestSampleSelTT.append( np.where(RMSEEvalT_Summ ==
                                                                 RMSERMSEEvalT_SummSelT[ np.nanmin( np.where( (RMSERMSEEvalT_SummSelT <=
                                                                                                              (np.nanmin(RMSERMSEEvalT_SummSelT)+
                                                                                                              ((np.nanmax(RMSERMSEEvalT_SummSelT)-np.nanmin(RMSERMSEEvalT_SummSelT))*selThrshldRMSE))) )[0] ) ])[0][0] )
                            else:
                                bestSampleSelTT.append(  np.nanmin( np.where((np.array(UncSummT_all) <= selThrshldCalcO)) ) )  # same as v1 in this case - lowest uncertainty
                        else: # UncAvgO
                            lowThrshldT, highThrshldT = (np.nanmean(UncSummT_all) - selThrshldCalcO), (np.nanmean(UncSummT_all) + selThrshldCalcO)
                            bestSampleSelToPT         = np.where( ((np.array(UncSummT_all) >= lowThrshldT) & (np.array(UncSummT_all) <= highThrshldT)) )[0].tolist()
                            if( (len(bestSampleSelToPT) == 0) ):
                                UncSummT_allUpD = copy.deepcopy(UncSummT_all)
                                while( (len(bestSampleSelToPT) == 0) ):
                                    UncSummT_allUpD   = [ UncSummT_allUpD[val] for val in range(len(UncSummT_allUpD)) if UncSummT_allUpD[val] != np.nanmax(UncSummT_allUpD) ]
                                    lowThrshldT, highThrshldT = (np.nanmean(UncSummT_allUpD) - selThrshldCalcO), (np.nanmean(UncSummT_allUpD) + selThrshldCalcO)
                                    bestSampleSelToPT = np.where( ((np.array(UncSummT_allUpD) >= lowThrshldT) & (np.array(UncSummT_allUpD) <= highThrshldT)) )[0].tolist()
                                UncSummT_allUpD = [ UncSummT_allUpD[val] for val in bestSampleSelToPT ]
                            else:
                                UncSummT_allUpD = [ UncSummT_all[val] for val in bestSampleSelToPT ]
                            if( minSelCritOpt == 'MinUnc' ):    # select nbr of samples given minimum uncertainty
                                bestSampleSelTT.append( np.nanmin( np.where( (np.array(UncSummT_all) == np.nanmin(UncSummT_allUpD)) )[0] ) )
                            elif( minSelCritOpt == 'MinSamp' ): # select nbr of samples resulting in the min number of observations  # 'MinSamp'
                                bestSampleSelTT.append( np.nanmin( np.where( (np.array(UncSummT_all) == UncSummT_allUpD[0]) )[0] ) )
                            else: # MidSamp
                                bestSampleSelTT.append( np.floor( np.nanmedian( np.where( [ x in UncSummT_allUpD for x in UncSummT_all ] )[0] ) ).astype('int') ) # cahneg ceil or floor to increase or decrease nbr of samples elected

                    # use the same number of samples indicated as 'optimal' based on uncertainty - used for date specific comparison
                    else:
                        bestSampleSelTT.append( bestSampleSelF_allO[ii][jj][kk] )

            InterBooTT.append( InterBooT )
            PLSRFactTT.append( PLSRFactT )
            R2EvalTT.append( R2EvalT )
            RMSEEvalTT.append( RMSEEvalT )
            R2EEvalTT.append( R2EEvalT )
            RMSEEEvalTT.append( RMSEEEvalT )
            R2ValTT.append( R2ValT )
            RMSEValTT.append( RMSEValT )
            R2OutTT.append( R2OutT )
            RMSEOutTT.append( RMSEOutT )
            R2IndTT.append( R2IndT )
            RMSEIndTT.append( RMSEIndT )
            R2IndVTT.append( R2IndVT )
            RMSEIndVTT.append( RMSEIndVT )

            sampleBooTT.append( sampleBooT )
            selSampleRefAllTT.append( selSampleRefAllT )
        InterBooF.append( InterBooTT )
        PLSRFactF.append( PLSRFactTT )
        R2EvalF.append( R2EvalTT )
        RMSEEvalF.append( RMSEEvalTT )
        R2EEvalF.append( R2EEvalTT )
        RMSEEEvalF.append( RMSEEEvalTT )
        R2ValF.append( R2ValTT )
        RMSEValF.append( RMSEValTT )
        R2OutF.append( R2OutTT )
        RMSEOutF.append( RMSEOutTT )
        R2IndF.append( R2IndTT )
        RMSEIndF.append( RMSEIndTT )
        R2IndVF.append( R2IndVTT )
        RMSEIndVF.append( RMSEIndVTT )
        sampleBooF.append( sampleBooTT )
        bestSampleSelF.append( bestSampleSelTT )
        TestUnF.append( TestUn )
        EvalUnF.append( EvalUn )
        ETestUnF.append( ETestUn )
        ValiUnF.append( ValiUn )
        OutDUnF.append( OutDUn )
        VIPOUnF.append( VIPOUn )
        CoefUnF.append( CoefUn )
        selSampleRefAllF.append( selSampleRefAllTT )
    InterBooF_all.append(InterBooF)
    PLSRFactF_all.append(PLSRFactF)
    R2EvalF_all.append(R2EvalF)
    RMSEEvalF_all.append(RMSEEvalF)
    R2EEvalF_all.append(R2EEvalF)
    RMSEEEvalF_all.append(RMSEEEvalF)
    R2ValF_all.append(R2ValF)
    RMSEValF_all.append(RMSEValF)
    R2OutF_all.append(R2OutF)
    RMSEOutF_all.append(RMSEOutF)
    R2IndF_all.append(R2IndF)
    RMSEIndF_all.append(RMSEIndF)
    R2IndVF_all.append(R2IndVF)
    RMSEIndVF_all.append(RMSEIndVF)
    sampleBooF_all.append( sampleBooF )
    bestSampleSelF_all.append( bestSampleSelF )
    TestUnF_all.append(TestUnF)
    EvalUnF_all.append(EvalUnF)
    ETestUnF_all.append(ETestUnF)
    ValiUnF_all.append(ValiUnF)
    OutDUnF_all.append(OutDUnF)
    VIPOUnF_all.append(VIPOUnF)
    CoefUnF_all.append(CoefUnF)
    sampleValF_all.append( sampleValF )
    selSampleRefAllF_all.append( selSampleRefAllF )

### Save or load the outputs

Reads the saved outputs. Set `SAVE_OUTPUTS` to save a new run first. *(Original cell 14.)*

In [ ]:
os.chdir( r'<PROJECT_ROOT>\1_JypiterLabProcessing\5_Outputs' )

versionRef = '106'
print( (str(selThrshld) + ' | ' + str(selThrshldRMSE) + ' | ' +  criteriaOptT + ' | ' + EmodeRef + ' | v' + versionRef) )

if( SAVE_OUTPUTS ):
    with open('InterBooF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
        pickle.dump(InterBooF_all, f)
    with open('PLSRFactF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
        pickle.dump(PLSRFactF_all, f)
    with open('R2EvalF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
        pickle.dump(R2EvalF_all, f)
    with open('RMSEEvalF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
        pickle.dump(RMSEEvalF_all, f)
    with open('R2EEvalF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
        pickle.dump(R2EEvalF_all, f)
    with open('RMSEEEvalF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
        pickle.dump(RMSEEEvalF_all, f)
    with open('R2ValF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
        pickle.dump(R2ValF_all, f)
    with open('RMSEValF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
        pickle.dump(RMSEValF_all, f)
    with open('R2OutF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
        pickle.dump(R2OutF_all, f)
    with open('RMSEOutF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
        pickle.dump(RMSEOutF_all, f)
    with open('R2IndF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
        pickle.dump(R2IndF_all, f)
    with open('RMSEIndF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
        pickle.dump(RMSEIndF_all, f)
    with open('R2IndVF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
        pickle.dump(R2IndVF_all, f)
    with open('RMSEIndVF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
        pickle.dump(RMSEIndVF_all, f)
    with open('sampleBooF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
        pickle.dump(sampleBooF_all, f)
    with open('bestSampleSelF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
        pickle.dump(bestSampleSelF_all, f)
if( SAVE_OUTPUTS ):
    with open('TestUnF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
        pickle.dump(TestUnF_all, f)
    with open('EvalUnF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
        pickle.dump(EvalUnF_all, f)
    with open('ETestUnF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
        pickle.dump(ETestUnF_all, f)
    with open('ValiUnF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
        pickle.dump(ValiUnF_all, f)
    with open('OutDUnF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
        pickle.dump(OutDUnF_all, f)
    with open('VIPOUnF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
        pickle.dump(VIPOUnF_all, f)
    with open('CoefUnF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
        pickle.dump(CoefUnF_all, f)
    with open('sampleValF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
        pickle.dump(sampleValF_all, f)
    with open('selSampleRefAllF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
        pickle.dump(selSampleRefAllF_all, f)
# load
with open('InterBooF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    InterBooF_allO = pickle.load(f)

with open('PLSRFactF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    PLSRFactF_allO = pickle.load(f)
with open('R2EvalF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    R2EvalF_allO = pickle.load(f)

with open('RMSEEvalF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    RMSEEvalF_allO = pickle.load(f)

with open('R2EEvalF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    R2EEvalF_allO = pickle.load(f)

with open('RMSEEEvalF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    RMSEEEvalF_allO = pickle.load(f)

with open('R2ValF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    R2ValF_allO = pickle.load(f)

with open('RMSEValF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    RMSEValF_allO = pickle.load(f)

with open('R2OutF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    R2OutF_allO = pickle.load(f)

with open('RMSEOutF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    RMSEOutF_allO = pickle.load(f)

with open('R2IndF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    R2IndF_allO = pickle.load(f)

with open('RMSEIndF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    RMSEIndF_allO = pickle.load(f)

with open('R2IndVF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    R2IndVF_allO = pickle.load(f)

with open('RMSEIndVF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    RMSEIndVF_allO = pickle.load(f)

with open('sampleValF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    sampleValF_allO = pickle.load(f)

with open('bestSampleSelF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    bestSampleSelF_allO = pickle.load(f)

with open('TestUnF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    TestUnF_allO = pickle.load(f)

with open('EvalUnF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    EvalUnF_allO = pickle.load(f)

with open('ETestUnF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    ETestUnF_allO = pickle.load(f)

with open('ValiUnF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    ValiUnF_allO = pickle.load(f)

with open('OutDUnF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    OutDUnF_allO = pickle.load(f)

with open('VIPOUnF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    VIPOUnF_allO = pickle.load(f)

with open('CoefUnF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    CoefUnF_allO = pickle.load(f)

with open('sampleBooF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    sampleBooF_allO = pickle.load(f)

with open('selSampleRefAllF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    selSampleRefAllF_allO = pickle.load(f)

### Comparison of selection criteria

Compares the criteria tested to choose the number of samples, from the saved runs. This supports the choice described in Section 2.5. *(Original cell 17.)*

In [ ]:
os.chdir( r'<PROJECT_ROOT>\DataProcessing\1_Script_source' )
import FakeParulaPalette as FakeParulaF

# select the criteria used to select optimal nbr of samples  --  'UncO' == uncertainty only ;
# 'UncRMSETO' == uncertainty + RMSE for out-of-bag sample + uncertainty (min uncertainty) ;
# 'UncRMSEO'  == uncertainty + RMSE for out-of-bag samples (min RMSE) ;
# 'UncRMSEOO' == uncertainty + RMSE for out-of-bag samples (min nbr fo samples)
# 'UncAvgO'   == closest to average uncertainty + min nbr of samples
criteriaOptT = 'UncAvgO'

# version of analysis results
if( criteriaOptT == 'UncO' ):
    versionRefAll = [ str(val+1) if (val >= 9) else '0'+str(val+1) for val in range(0, 9, 1) ]  + ['E01']
elif( criteriaOptT == 'UncRMSETO' ):
    versionRefAll = [ str(val+1) if (val >= 9) else '0'+str(val+1) for val in range(9, 33, 1) ] + ['E02', 'E03']
elif( criteriaOptT == 'UncRMSEO' ):
    versionRefAll = [ str(val+1) if (val >= 9) else '0'+str(val+1) for val in range(33, 57, 1) ]
elif( criteriaOptT == 'UncRMSEOO' ):
    versionRefAll = [ str(val+1) if (val >= 9) else '0'+str(val+1) for val in range(57, 81, 1) ]
else:
    versionRefAll = [ str(val+1) if (val >= 9) else '0'+str(val+1) for val in range(81, 110, 1) ]

TraitRef   = ['DM_dt_ha', 'N_perc_DM', 'N_kg_ha' ]
TraitRefL  = [ 'Dry matter $[\/dt\/\/ha^{-1}\/]$',
               'N content $[\/\%\/in\/DM\/]$',
               'N total $[\/kg\/\/ha^{-1}\/]$'  ]

HarvRef  = [ '20140515', '20141015', '20170510', '20170830', '20171026' ]
col_ref  = list( range(len(center_origin)) )

modeRef = 'mean'

fontS1 = 11
fontS2 = 12
MajorTicksLength = 3.5
MinorTicksLength = 2.2
errorLineWidth   = 1.5

fontS0 = 8
fontS1 = 9
fontS2 = 10
fontS3 = 11

FontSizeG = 9
FontSizeT = 10

# distance between results for a given nbr of extra samples
distOutRef = 2

YearRef = SpecAll_sel_0_1_res_Avg.Year.unique().tolist()
DateRef = SpecAll_sel_0_1_res_Avg.Date.unique().tolist()
DateRefY = [ int(str(DateRef[val])[:4]) for val in range(len(DateRef)) ]

DateDOY = []
for elmnt in range(len(DateRef)):  # date
    dateT = pd.to_datetime(DateRef[elmnt], format='%Y%m%d')
    dateTRef = pd.Timestamp(year=dateT.year, month=1, day=1)
    DateDOY.append( (dateT - dateTRef).days + 1 )

# color - parula pallete
Parula_manual   = []
seqSelT = np.round( np.arange(0, 256, 256/len(DateRefY)) ).astype(int).tolist()
for val in range( len(seqSelT) ):
    Parula_manual.append( FakeParulaF.FakeParula_cm('FakeParula', FakeParulaF.cm_data)(seqSelT[val])[:-1] )
colorR = list( range( len(Parula_manual) ) )

idxToP   = 0
for elmnt in range( len(versionRefAll ) ):

    # best samples for each param combination
    os.chdir( r'<PROJECT_ROOT>\1_JypiterLabProcessing\5_Outputs' )
    with open('bestSampleSelF_allO_BootPLS_spike_opt_v' + versionRefAll[elmnt] + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
        bestSampleSelF_allTemp = pickle.load(f)

    # Validation - source and target data - outputs for each param combination
    with open('R2IndVF_allO_BootPLS_spike_opt_v'   + versionRefAll[elmnt] + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
        R2IndVF_allTemp = pickle.load(f)
    with open('RMSEIndVF_allO_BootPLS_spike_opt_v' + versionRefAll[elmnt] + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
        RMSEIndVF_allTemp = pickle.load(f)

    # Test - source data - out-of-bag
    with open('R2EvalF_allO_BootPLS_spike_opt_v' + versionRefAll[elmnt] + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
        R2EvalF_allTemp = pickle.load(f)

    with open('RMSEEvalF_allO_BootPLS_spike_opt_v' + versionRefAll[elmnt] + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
        RMSEEvalF_allTemp = pickle.load(f)

    # Create objects to store summary outputs
    R2RMSIndVSumm    = []
    R2RMSIndVSummIqr = []
    R2RMSEvalSumm    = []
    R2RMSEvalSummIqr = []
    for ii in range( len(R2IndVF_allTemp) ): # source dataset | year
        if( ii == 0 ):
            DateOutRefT =  HarvRef[2:6]
        else:
            DateOutRefT =  HarvRef[0:2]
        R2RMSIndVSummT    = []
        R2RMSIndVSummIqrT = []
        R2RMSEvalSummT    = []
        R2RMSEvalSummIqrT = []
        for jj in range( len(R2IndVF_allTemp[ii]) ):  # trait
            R2RMSIndVSummTT    = []
            R2RMSIndVSummIqrTT = []
            R2RMSEvalSummTT    = []
            R2RMSEvalSummIqrTT = []
            for kk in range( len(R2IndVF_allTemp[ii][jj]) ):  # date in the outside dataset - target
                R2RMSIndVSummTT.append(     np.full( (len(R2IndVF_allTemp[ii][jj][kk]),3), np.nan ) )
                R2RMSIndVSummIqrTT.append(  np.full( (len(R2IndVF_allTemp[ii][jj][kk]),3), np.nan ) )
                R2RMSEvalSummTT.append(     np.full( (len(R2EvalF_allTemp[ii][jj][kk]),3), np.nan ) )
                R2RMSEvalSummIqrTT.append(  np.full( (len(R2EvalF_allTemp[ii][jj][kk]),3), np.nan ) )
            R2RMSIndVSummT.append( R2RMSIndVSummTT )
            R2RMSIndVSummIqrT.append( R2RMSIndVSummIqrTT )
            R2RMSEvalSummT.append( R2RMSEvalSummTT )
            R2RMSEvalSummIqrT.append( R2RMSEvalSummIqrTT )
        R2RMSIndVSumm.append( R2RMSIndVSummT )
        R2RMSIndVSummIqr.append( R2RMSIndVSummIqrT )
        R2RMSEvalSumm.append( R2RMSEvalSummT )
        R2RMSEvalSummIqr.append( R2RMSEvalSummIqrT )

    # Get summary for analysis output
    for ii in range( len(R2IndVF_allTemp) ):  # dataset source
        if( ii == 0 ):
            DateOutRefT =  HarvRef[2:6]
        else:
            DateOutRefT =  HarvRef[0:2]
        for jj in range( len(R2IndVF_allTemp[ii]) ):  # trait
            for kk in range( len(R2IndVF_allTemp[ii][jj]) ):  # outside dataset | target
                for ll in range( len(R2IndVF_allTemp[ii][jj][kk]) ) :  # nbr of samples added from the target domain
                    R2RMSIndVSumm[ii][jj][kk][ll,1]     = np.nanmedian( R2IndVF_allTemp[ii][jj][kk][ll] )   if modeRef == 'med' else np.nanmean( R2IndVF_allTemp[ii][jj][kk][ll] )
                    R2RMSIndVSumm[ii][jj][kk][ll,2]     = np.nanmedian( RMSEIndVF_allTemp[ii][jj][kk][ll] ) if modeRef == 'med' else np.nanmean( RMSEIndVF_allTemp[ii][jj][kk][ll] )
                    R2RMSIndVSummIqr[ii][jj][kk][ll,1]  = scipy.stats.iqr( R2IndVF_allTemp[ii][jj][kk][ll], nan_policy='omit' )    if modeRef == 'med' else np.nanstd( R2IndVF_allTemp[ii][jj][kk][ll] )
                    R2RMSIndVSummIqr[ii][jj][kk][ll,2]  = scipy.stats.iqr( RMSEIndVF_allTemp[ii][jj][kk][ll], nan_policy='omit' )  if modeRef == 'med' else np.nanstd( RMSEIndVF_allTemp[ii][jj][kk][ll] )
                    R2RMSEvalSumm[ii][jj][kk][ll,1]     = np.nanmedian( R2EvalF_allTemp[ii][jj][kk][ll] )   if modeRef == 'med' else np.nanmean( R2EvalF_allTemp[ii][jj][kk][ll] )
                    R2RMSEvalSumm[ii][jj][kk][ll,2]     = np.nanmedian( RMSEEvalF_allTemp[ii][jj][kk][ll] ) if modeRef == 'med' else np.nanmean( RMSEEvalF_allTemp[ii][jj][kk][ll] )
                    R2RMSEvalSummIqr[ii][jj][kk][ll,1]  = scipy.stats.iqr( R2EvalF_allTemp[ii][jj][kk][ll], nan_policy='omit' )    if modeRef == 'med' else np.nanstd( R2EvalF_allTemp[ii][jj][kk][ll] )
                    R2RMSEvalSummIqr[ii][jj][kk][ll,2]  = scipy.stats.iqr( RMSEEvalF_allTemp[ii][jj][kk][ll], nan_policy='omit' )  if modeRef == 'med' else np.nanstd( RMSEEvalF_allTemp[ii][jj][kk][ll] )

    for jj in range( len(R2RMSIndVSumm[0]) ):  # trait

        # create figures
        if( elmnt == 0 ):
            if( jj == 0 ):
                fig = pylab.figure( figCount-(len(R2RMSIndVSumm[0])-jj), figsize=(12.5,  (9.5 if (criteriaOptT not in ['UncO', 'UncAvgO']) else 11.5)) )
            else:
                fig = pylab.figure( figCount-(len(R2RMSIndVSumm[0])-jj), figsize=(8,     (9.5 if (criteriaOptT not in ['UncO', 'UncAvgO']) else 11.5)) )
        else:
            fig = pylab.figure( figCount-(len(R2RMSIndVSumm[0])-jj) )

        # divide plot in parts
        gs  = matplotlib.gridspec.GridSpec(len(versionRefAll), 3, height_ratios=[1]*len(versionRefAll), width_ratios=([4,1,1] if (jj == 0) else [2,1,1]) )
        gs.update( wspace=0.025, hspace=0.05 )

        ax = plt.subplot(gs[idxToP])

        # Add point indicating final choice
        for ii in range( len(R2RMSIndVSumm) ):
            if( ii == 0 ):
                DateDOYTarget = DateDOY[2:5]
                colorRTarget  = colorR[2:5]
            else:
                DateDOYTarget = DateDOY[0:2]
                colorRTarget  = colorR[0:2]
            for kk in range( len(R2RMSIndVSumm[ii][0]) ):
                if(ii == 0):
                    posRefT = (kk+1)
                else:
                    posRefT = len(R2RMSIndVSumm[0][0])+(kk+1)
                # plot nbr of selected
                ax.plot( ((bestSampleSelF_allTemp[ii][jj][kk]*(5+distOutRef))+distOutRef+posRefT), 0.5, 'o', color=Parula_manual[ colorRTarget[kk] ],
                         mfc='none', markeredgewidth=errorLineWidth )

        # add tick marks - X-axis
        xMarksRefT = list( range( 5, (5*R2RMSIndVSumm[ii][jj][kk].shape[0])+5, 5 ) )
        xMarksRefT = [ xMarksRefT[val]+(val*distOutRef) for val in range(len(xMarksRefT)) ]
        ax.set_xticks( xMarksRefT )
        major_ticks = list( range( clstCntrlRef, (clstCntrlRef*R2RMSIndVSumm[ii][jj][kk].shape[0])+clstCntrlRef, clstCntrlRef ) )
        ax.set_xticklabels( major_ticks, fontname = "Arial",  fontsize=fontS1 )
        ax.set_xlim( xMarksRefT[0]-4, xMarksRefT[-1]+4 )
        ax.set_ylim( [0.48, 0.52] )
        labelRefT =  [ 'V' + str(versionRefAll[val]) for val in range(len(versionRefAll)) ]
        ax.set_yticks( [ 0.5] )
        ax.set_yticklabels( [ labelRefT[elmnt] ], fontname = "Arial",  fontsize=fontS1 )

        # Add division between number of samples
        xDivRefT = list( np.arange( 5, (5*R2RMSIndVSumm[ii][jj][kk].shape[0]), 5 ) )
        xDivRefT = [ xDivRefT[val]+((val+1)*2) for val in range(len(xDivRefT)) ]
        if( distOutRef % 2 == 0 ):
            xDivRefT = [ xDivRefT[val]+(distOutRef/2)+0.5 for val in range(len(xDivRefT)) ]
        else:
            xDivRefT = [ xDivRefT[val]+(distOutRef/2) for val in range(len(xDivRefT)) ]
        for val in range(len(xDivRefT)):
            ax.axvline( x=xDivRefT[val], color='k', linewidth=0.8, linestyle='dashed' )

        # add R2 and RMSE corresponding to 100 models predicting the valdation data - careful, this includes out-of-bag for the source dataset
        ax = plt.subplot(gs[idxToP+2])
        R2RMSIndVSummCompT = []
        for ii in range( len(R2RMSIndVSumm) ):
            ll = bestSampleSelF_allTemp[ii][jj][-1]  # best nbr of samples for last acquisition on target considering a given source
            if( ii == 0 ):
                xCoordAT, yCoordAT, xCoordBT, yCoordBT = 0.1, 0.65, 0.45, 0.65
            else:
                xCoordAT, yCoordAT, xCoordBT, yCoordBT = 0.1, 0.2, 0.45, 0.2
            ax.text( xCoordAT, yCoordAT, '$R^{2}$ = ' + str("%.2f" % round(R2RMSIndVSumm[ii][jj][-1][ll,1], 3)) if R2RMSIndVSumm[ii][jj][-1][ll,1] > 0 else \
                     'R-sqrd. = ' + str("%.2f" % round(0.00, 3)), va='center', rotation=0, size=FontSizeG, \
                     fontname = "Arial", fontstyle='normal', fontweight='normal', color='k')
            ax.text( xCoordBT, yCoordBT, ' ; RMSE = ' + str("%.2f" % round(R2RMSIndVSumm[ii][jj][-1][ll,2], 3)), va='center', rotation=0, size=FontSizeG, \
                     fontname = "Arial", fontstyle='normal', fontweight='normal', color='k')
            R2RMSIndVSummCompT.append( R2RMSIndVSumm[ii][jj][-1][ll,2] )
        ax.set_xticks([])
        ax.set_yticks([])

        # add R2 and RMSE corresponding to 100 models predicting the valdation data - only out-of-bag for the source dataset
        ax = plt.subplot(gs[idxToP+1])
        R2RMSEvalSummCompT = []
        for ii in range( len(R2RMSEvalSumm) ):
            ll = bestSampleSelF_allTemp[ii][jj][-1]  # best nbr of samples for last acquisition on target considering a given source
            if( ii == 0 ):
                xCoordAT, yCoordAT, xCoordBT, yCoordBT = 0.05, 0.65, 0.40, 0.65
            else:
                xCoordAT, yCoordAT, xCoordBT, yCoordBT = 0.05, 0.2,  0.40, 0.2
            ax.text( xCoordAT, yCoordAT, '$R^{2}$ = ' + str("%.2f" % round(R2RMSEvalSumm[ii][jj][-1][ll,1], 3)) if R2RMSEvalSumm[ii][jj][-1][ll,1] > 0 else \
                     '$R^{2}$ = ' + str("%.2f" % round(0.00, 3)), va='center', rotation=0, size=FontSizeG, \
                     fontname = "Arial", fontstyle='normal', fontweight='normal', color='k')
            ax.text( xCoordBT, yCoordBT, ' ; RMSE = ' + str("%.2f" % round(R2RMSEvalSumm[ii][jj][-1][ll,2], 3)), va='center', rotation=0, size=FontSizeG, \
                     fontname = "Arial", fontstyle='normal', fontweight='normal', color='k')
            R2RMSEvalSummCompT.append( R2RMSEvalSumm[ii][jj][-1][ll,2] )
        ax.set_xticks([])
        ax.set_yticks([])
    idxToP += 3
figCount += len(R2RMSIndVSumm[0])

### Summary

Calculates R2, RMSE and their spread, and summarizes the predictions and VIP. *(Original cell 18.)*

In [ ]:
# Prepare data output formats
R2RMSEvalSumm   = []
R2RMSEEvalSumm  = []
R2RMSValSumm    = []
R2RMSOutSumm    = []
R2RMSIndSumm    = []
R2RMSIndVSumm   = []
R2RMSEvalSummIqr   = []
R2RMSEEvalSummIqr  = []
R2RMSValSummIqr    = []
R2RMSOutSummIqr    = []
R2RMSIndSummIqr    = []
R2RMSIndVSummIqr   = []
PredObs     = []
PredEObs    = []
PredObsVali = []
PredObsOut  = []
PredVIP     = []
PredObsIqr     = []
PredEObsIqr    = []
PredObsValiIqr = []
PredObsOutIqr  = []
PredVIPIqr     = []
IQRSumm = []
for ii in range( len(R2EvalF_allO) ): # source dataset | year
    if( ii == 0 ):
        DateOutRefT =  HarvRef[2:6]
    else:
        DateOutRefT =  HarvRef[0:2]
    R2RMSEvalSummT   = []
    R2RMSEEvalSummT  = []
    R2RMSValSummT    = []
    R2RMSOutSummT    = []
    R2RMSIndSummT    = []
    R2RMSIndVSummT   = []
    R2RMSEvalSummIqrT   = []
    R2RMSEEvalSummIqrT  = []
    R2RMSValSummIqrT    = []
    R2RMSOutSummIqrT    = []
    R2RMSIndSummIqrT    = []
    R2RMSIndVSummIqrT   = []
    PredObsT     = []
    PredEObsT    = []
    PredObsValiT = []
    PredObsOutT  = []
    PredVIPT     = []
    PredObsIqrT     = []
    PredEObsIqrT    = []
    PredObsValiIqrT = []
    PredObsOutIqrT  = []
    PredVIPIqrT     = []
    IQRSummT = []
    for jj in range( len(R2EvalF_allO[ii]) ):  # trait
        R2RMSEvalSummTT   = []
        R2RMSEEvalSummTT  = []
        R2RMSValSummTT    = []
        R2RMSOutSummTT    = []
        R2RMSIndSummTT    = []
        R2RMSIndVSummTT   = []
        R2RMSEvalSummIqrTT   = []
        R2RMSEEvalSummIqrTT  = []
        R2RMSValSummIqrTT    = []
        R2RMSOutSummIqrTT    = []
        R2RMSIndSummIqrTT    = []
        R2RMSIndVSummIqrTT   = []
        PredObsTT     = []
        PredEObsTT    = []
        PredObsValiTT = []
        PredObsOutTT  = []
        PredVIPTT     = []
        PredObsIqrTT     = []
        PredEObsIqrTT    = []
        PredObsValiIqrTT = []
        PredObsOutIqrTT  = []
        PredVIPIqrTT     = []
        IQRSummTT = []
        for kk in range( len(R2EvalF_allO[ii][jj]) ):  # date in the outside dataset - target
# for ll in range( len(R2EvalF_allO[ii][jj][kk]) ):  # nbr of samples added from the outside domain

            R2RMSEvalSummTT.append(  np.full( (len(R2EvalF_allO[ii][jj][kk]),3), np.nan ) )   # nbr of samples added from outside acquisiion vs parameter
            R2RMSEEvalSummTT.append( np.full( (len(R2EEvalF_allO[ii][jj][kk]),3), np.nan ) )
            R2RMSValSummTT.append(  np.full( (len(R2ValF_allO[ii][jj][kk]),3), np.nan ) )
            R2RMSOutSummTT.append(  np.full( (len(R2OutF_allO[ii][jj][kk]),3), np.nan ) )
            R2RMSIndSummTT.append(  np.full( (len(R2IndF_allO[ii][jj][kk]),3), np.nan ) )
            R2RMSIndVSummTT.append(  np.full( (len(R2IndVF_allO[ii][jj][kk]),3), np.nan ) )
            R2RMSEvalSummIqrTT.append( np.full( (len(R2EvalF_allO[ii][jj][kk]),3), np.nan ) )
            R2RMSEEvalSummIqrTT.append( np.full( (len(R2EEvalF_allO[ii][jj][kk]),3), np.nan ) )
            R2RMSValSummIqrTT.append(  np.full( (len(R2ValF_allO[ii][jj][kk]),3), np.nan ) )
            R2RMSOutSummIqrTT.append(  np.full( (len(R2OutF_allO[ii][jj][kk]),3), np.nan ) )
            R2RMSIndSummIqrTT.append(  np.full( (len(R2IndF_allO[ii][jj][kk]),3), np.nan ) )
            R2RMSIndVSummIqrTT.append(  np.full( (len(R2IndVF_allO[ii][jj][kk]),3), np.nan ) )
            PredObsTT.append( [ [] for val in range(len(TestUnF_allO[ii][jj][kk])) ] )
            PredEObsTT.append( [ [] for val in range(len(ETestUnF_allO[ii][jj][kk])) ] )
            PredObsValiTT.append( [ [] for val in range(len(ValiUnF_allO[ii][jj][kk])) ] )
            PredObsOutTT.append( [ [] for val in range(len(OutDUnF_allO[ii][jj][kk])) ] )
            PredVIPTT.append( [ [] for val in range(len(VIPOUnF_allO[ii][jj][kk])) ] )
            PredObsIqrTT.append( [ [] for val in range(len(TestUnF_allO[ii][jj][kk])) ] )
            PredEObsIqrTT.append( [ [] for val in range(len(ETestUnF_allO[ii][jj][kk])) ] )
            PredObsValiIqrTT.append( [ [] for val in range(len(ETestUnF_allO[ii][jj][kk])) ] )
            PredObsOutIqrTT.append( [ [] for val in range(len(OutDUnF_allO[ii][jj][kk])) ] )
            PredVIPIqrTT.append( [ [] for val in range(len(VIPOUnF_allO[ii][jj][kk])) ] )

        R2RMSEvalSummT.append( R2RMSEvalSummTT )
        R2RMSEEvalSummT.append( R2RMSEEvalSummTT )
        R2RMSValSummT.append( R2RMSValSummTT )
        R2RMSOutSummT.append( R2RMSOutSummTT )
        R2RMSIndSummT.append( R2RMSIndSummTT )
        R2RMSIndVSummT.append( R2RMSIndVSummTT )
        R2RMSEvalSummIqrT.append( R2RMSEvalSummIqrTT )
        R2RMSEEvalSummIqrT.append( R2RMSEEvalSummIqrTT )
        R2RMSValSummIqrT.append( R2RMSValSummIqrTT )
        R2RMSOutSummIqrT.append( R2RMSOutSummIqrTT )
        R2RMSIndSummIqrT.append( R2RMSIndSummIqrTT )
        R2RMSIndVSummIqrT.append( R2RMSIndVSummIqrTT )
        PredObsT.append( PredObsTT )
        PredEObsT.append( PredEObsTT )
        PredObsValiT.append( PredObsValiTT )
        PredObsOutT.append( PredObsOutTT )
        PredVIPT.append( PredVIPTT )
        PredObsIqrT.append( PredObsIqrTT )
        PredEObsIqrT.append( PredEObsIqrTT )
        PredObsValiIqrT.append( PredObsValiIqrTT )
        PredObsOutIqrT.append( PredObsOutIqrTT )
        PredVIPIqrT.append( PredVIPIqrTT )
        IQRSummT.append( IQRSummTT )
    R2RMSEvalSumm.append( R2RMSEvalSummT )
    R2RMSEEvalSumm.append( R2RMSEEvalSummT )
    R2RMSValSumm.append( R2RMSValSummT )
    R2RMSOutSumm.append( R2RMSOutSummT )
    R2RMSIndSumm.append( R2RMSIndSummT )
    R2RMSIndVSumm.append( R2RMSIndVSummT )
    R2RMSEvalSummIqr.append( R2RMSEvalSummIqrT )
    R2RMSEEvalSummIqr.append( R2RMSEEvalSummIqrT )
    R2RMSValSummIqr.append( R2RMSValSummIqrT )
    R2RMSOutSummIqr.append( R2RMSOutSummIqrT )
    R2RMSIndSummIqr.append( R2RMSIndSummIqrT )
    R2RMSIndVSummIqr.append( R2RMSIndVSummIqrT )
    PredObs.append( PredObsT )
    PredEObs.append( PredEObsT )
    PredObsVali.append( PredObsValiT )
    PredObsOut.append( PredObsOutT )
    PredVIP.append( PredVIPT )
    PredObsIqr.append( PredObsIqrT )
    PredEObsIqr.append( PredEObsIqrT )
    PredObsValiIqr.append( PredObsValiIqrT )
    PredObsOutIqr.append( PredObsOutIqrT )
    PredVIPIqr.append( PredVIPIqrT )
    IQRSumm.append( IQRSummT )

# get summary for R2, RMSE and predictions

TraitRef  = ['DM_dt_ha', 'N_perc_DM', 'N_kg_ha' ]

HarvRef  = [ '20140515', '20141015', '20170510', '20170830', '20171026' ]
DateRefY = [ '2014', '2017' ]
col_ref  = list( range(len(center_origin)) )

modeRef = 'mean'

for ii in range( len(R2EvalF_allO) ):  # dataset source
    if( ii == 0 ):
        DateOutRefT =  HarvRef[2:6]
    else:
        DateOutRefT =  HarvRef[0:2]
    for jj in range( len(R2EvalF_allO[ii]) ):  # trait
        if( jj == 0 ): # Dry Biomass data / 49 observations per date
            DataP    = SpecAll_sel_0_1_res_Avg.loc[ ((SpecAll_sel_0_1_res_Avg.Year).astype(int) == int(DateRefY[ii])), :]  # source data
            GroupF   = [ str(DataP.Replicate.tolist()[val]) + '_' + str(DataP.Treatment.tolist()[val]) for val in range(DataP.shape[0]) ]  # reference to each plot...
            DataPOut = SpecAll_sel_0_1_res_Avg.loc[~((SpecAll_sel_0_1_res_Avg.Year).astype(int) == int(DateRefY[ii])), :]  # data outside the source domain (different season)
            DataP_PC = copy.deepcopy( PCAmOM_scoresAll )
        else:  # N data / 15 observations per date
            DataP    = SpecAll_sel_0_1_res_Avg_treat.loc[ ((SpecAll_sel_0_1_res_Avg_treat.Year).astype(int) == int(DateRefY[ii])), :]
            GroupF   = DataP.Treatment.tolist()
            DataPOut = SpecAll_sel_0_1_res_Avg_treat.loc[~((SpecAll_sel_0_1_res_Avg_treat.Year).astype(int) == int(DateRefY[ii])), :]
            DataP_PC = copy.deepcopy( PCAmOM_scoresAllRed )
        DataM_df    = DataP.loc[ :, (DataP.columns == TraitRef[jj]) | DataP.columns.isin(col_ref) ]  # select trait of interest + spectra
        DataPOut_df = DataPOut.loc[ :, (DataPOut.columns == TraitRef[jj]) | DataPOut.columns.isin(col_ref) ]  # data outside the source domain (different season)
        BRef = [ 'b' + str(range(len(col_ref))[val]) if range(len(col_ref))[val] > 9 else 'b0' + str( range(len(col_ref) )[val]) for val in range(len(col_ref)) ]
        DataM_df.columns = ['Trait'] + BRef
        DataM_df = DataM_df.reset_index(drop=True)  # not needed since reference made by index, but doing anyway
        DataPOut_df.columns = ['Trait'] + BRef
        DataPOut_df = DataPOut_df.reset_index(drop=True)  # not needed since reference made by index, but doing anyway

        for kk in range( len(R2EvalF_allO[ii][jj]) ):  # outside dataset | target
            for ll in range( len(R2EvalF_allO[ii][jj][kk]) ) :  # nbr of samples added from the target domain

                # central descriptor
                R2RMSEvalSumm[ii][jj][kk][ll,1] = np.nanmedian( R2EvalF_allO[ii][jj][kk][ll] ) if modeRef == 'med' else np.nanmean( R2EvalF_allO[ii][jj][kk][ll] )
                R2RMSEvalSumm[ii][jj][kk][ll,2] = np.nanmedian( RMSEEvalF_allO[ii][jj][kk][ll] ) if modeRef == 'med' else np.nanmean( RMSEEvalF_allO[ii][jj][kk][ll] )
                R2RMSEvalSumm[ii][jj][kk][ll,0] = scipy.stats.mode( PLSRFactF_allO[ii][jj][kk][ll] )[0].tolist()[0]
                R2RMSEEvalSumm[ii][jj][kk][ll,1] = np.nanmedian( R2EEvalF_allO[ii][jj][kk][ll] ) if modeRef == 'med' else np.nanmean( R2EEvalF_allO[ii][jj][kk][ll] )
                R2RMSEEvalSumm[ii][jj][kk][ll,2] = np.nanmedian( RMSEEEvalF_allO[ii][jj][kk][ll] ) if modeRef == 'med' else np.nanmean( RMSEEEvalF_allO[ii][jj][kk][ll] )
                R2RMSValSumm[ii][jj][kk][ll,1]  = np.nanmedian( R2ValF_allO[ii][jj][kk][ll] ) if modeRef == 'med' else np.nanmean( R2ValF_allO[ii][jj][kk][ll] )
                R2RMSValSumm[ii][jj][kk][ll,2]  = np.nanmedian( RMSEValF_allO[ii][jj][kk][ll] ) if modeRef == 'med' else np.nanmean( RMSEValF_allO[ii][jj][kk][ll] )
                R2RMSOutSumm[ii][jj][kk][ll,1]  = np.nanmedian( R2OutF_allO[ii][jj][kk][ll] ) if modeRef == 'med' else np.nanmean( R2OutF_allO[ii][jj][kk][ll] )
                R2RMSOutSumm[ii][jj][kk][ll,2]  = np.nanmedian( RMSEOutF_allO[ii][jj][kk][ll] ) if modeRef == 'med' else np.nanmean( RMSEOutF_allO[ii][jj][kk][ll] )
                R2RMSIndSumm[ii][jj][kk][ll,1]  = np.nanmedian( R2IndF_allO[ii][jj][kk][ll] ) if modeRef == 'med' else np.nanmean( R2IndF_allO[ii][jj][kk][ll] )
                R2RMSIndSumm[ii][jj][kk][ll,2]  = np.nanmedian( RMSEIndF_allO[ii][jj][kk][ll] ) if modeRef == 'med' else np.nanmean( RMSEIndF_allO[ii][jj][kk][ll] )
                R2RMSIndVSumm[ii][jj][kk][ll,1] = np.nanmedian( R2IndVF_allO[ii][jj][kk][ll] ) if modeRef == 'med' else np.nanmean( R2IndVF_allO[ii][jj][kk][ll] )
                R2RMSIndVSumm[ii][jj][kk][ll,2] = np.nanmedian( RMSEIndVF_allO[ii][jj][kk][ll] ) if modeRef == 'med' else np.nanmean( RMSEIndVF_allO[ii][jj][kk][ll] )

                # deviation descriptor
                R2RMSEvalSummIqr[ii][jj][kk][ll,1] = scipy.stats.iqr( R2EvalF_allO[ii][jj][kk][ll], nan_policy='omit' )  if modeRef == 'med' else np.nanstd( R2EvalF_allO[ii][jj][kk][ll] )
                R2RMSEvalSummIqr[ii][jj][kk][ll,2] = scipy.stats.iqr( RMSEEvalF_allO[ii][jj][kk][ll], nan_policy='omit' ) if modeRef == 'med' else np.nanstd( RMSEEvalF_allO[ii][jj][kk][ll] )
                R2RMSEEvalSummIqr[ii][jj][kk][ll,1] = scipy.stats.iqr( R2EEvalF_allO[ii][jj][kk][ll], nan_policy='omit' )   if modeRef == 'med' else np.nanstd( R2EEvalF_allO[ii][jj][kk][ll] )
                R2RMSEEvalSummIqr[ii][jj][kk][ll,2] = scipy.stats.iqr( RMSEEEvalF_allO[ii][jj][kk][ll], nan_policy='omit' ) if modeRef == 'med' else np.nanstd( RMSEEEvalF_allO[ii][jj][kk][ll] )
                R2RMSValSummIqr[ii][jj][kk][ll,1]  = scipy.stats.iqr( R2ValF_allO[ii][jj][kk][ll], nan_policy='omit' )    if modeRef == 'med' else np.nanstd( R2ValF_allO[ii][jj][kk][ll] )
                R2RMSValSummIqr[ii][jj][kk][ll,2]  = scipy.stats.iqr( RMSEValF_allO[ii][jj][kk][ll], nan_policy='omit' )  if modeRef == 'med' else np.nanstd( RMSEValF_allO[ii][jj][kk][ll] )
                R2RMSOutSummIqr[ii][jj][kk][ll,1]  = scipy.stats.iqr( R2OutF_allO[ii][jj][kk][ll], nan_policy='omit' )    if modeRef == 'med' else np.nanstd( R2OutF_allO[ii][jj][kk][ll] )
                R2RMSOutSummIqr[ii][jj][kk][ll,2]  = scipy.stats.iqr( RMSEOutF_allO[ii][jj][kk][ll], nan_policy='omit' )  if modeRef == 'med' else np.nanstd( RMSEOutF_allO[ii][jj][kk][ll] )
                R2RMSIndSummIqr[ii][jj][kk][ll,1]  = scipy.stats.iqr( R2IndF_allO[ii][jj][kk][ll], nan_policy='omit' )    if modeRef == 'med' else np.nanstd( R2IndF_allO[ii][jj][kk][ll] )
                R2RMSIndSummIqr[ii][jj][kk][ll,2]  = scipy.stats.iqr( RMSEIndF_allO[ii][jj][kk][ll], nan_policy='omit' )  if modeRef == 'med' else np.nanstd( RMSEIndF_allO[ii][jj][kk][ll] )
                R2RMSIndVSummIqr[ii][jj][kk][ll,1]  = scipy.stats.iqr( R2IndVF_allO[ii][jj][kk][ll], nan_policy='omit' )    if modeRef == 'med' else np.nanstd( R2IndVF_allO[ii][jj][kk][ll] )
                R2RMSIndVSummIqr[ii][jj][kk][ll,2]  = scipy.stats.iqr( RMSEIndVF_allO[ii][jj][kk][ll], nan_policy='omit' )  if modeRef == 'med' else np.nanstd( RMSEIndVF_allO[ii][jj][kk][ll] )

                # summary of predictions - center
                PredObs[ii][jj][kk][ll].append( np.nanmedian( TestUnF_allO[ii][jj][kk][ll], axis=1 ).tolist() if modeRef == 'med' else \
                                                  np.nanmean( TestUnF_allO[ii][jj][kk][ll], axis=1 ).tolist() )
                PredEObs[ii][jj][kk][ll].append( np.nanmedian( ETestUnF_allO[ii][jj][kk][ll], axis=1 ).tolist() if modeRef == 'med' else \
                                                   np.nanmean( ETestUnF_allO[ii][jj][kk][ll], axis=1 ).tolist() )
                PredObsVali[ii][jj][kk][ll].append( np.nanmedian( ValiUnF_allO[ii][jj][kk][ll], axis=1 ).tolist() if modeRef == 'med' else \
                                                      np.nanmean( ValiUnF_allO[ii][jj][kk][ll], axis=1 ).tolist() )
                PredObsOut[ii][jj][kk][ll].append( np.nanmedian( OutDUnF_allO[ii][jj][kk][ll], axis=1 ).tolist() if modeRef == 'med' else \
                                                     np.nanmean( OutDUnF_allO[ii][jj][kk][ll], axis=1 ).tolist() )

                # summary of predictions - deviation
                PredObsIqr[ii][jj][kk][ll].append( scipy.stats.iqr( TestUnF_allO[ii][jj][kk][ll], nan_policy='omit', axis=1 ).tolist() if modeRef == 'med' else \
                                                         np.nanstd( TestUnF_allO[ii][jj][kk][ll], axis=1 ).tolist() )
                PredEObsIqr[ii][jj][kk][ll].append( scipy.stats.iqr( ETestUnF_allO[ii][jj][kk][ll], nan_policy='omit', axis=1 ).tolist() if modeRef == 'med' else \
                                                          np.nanstd( ETestUnF_allO[ii][jj][kk][ll], axis=1 ).tolist() )
                PredObsValiIqr[ii][jj][kk][ll].append( scipy.stats.iqr( ValiUnF_allO[ii][jj][kk][ll], nan_policy='omit', axis=1 ).tolist() if modeRef == 'med' else \
                                                             np.nanstd( ValiUnF_allO[ii][jj][kk][ll], axis=1 ).tolist() )
                PredObsOutIqr[ii][jj][kk][ll].append( scipy.stats.iqr( OutDUnF_allO[ii][jj][kk][ll], nan_policy='omit', axis=1 ).tolist() if modeRef == 'med' else \
                                                            np.nanstd( OutDUnF_allO[ii][jj][kk][ll], axis=1 ).tolist() )

                # VIP values
                PredVIP[ii][jj][kk][ll].append( np.nanmedian( VIPOUnF_allO[ii][jj][kk][ll], axis=1 ).tolist() if modeRef == 'med' else \
                                                np.nanmean( VIPOUnF_allO[ii][jj][kk][ll], axis=1 ).tolist() )
                PredVIPIqr[ii][jj][kk][ll].append( scipy.stats.iqr( VIPOUnF_allO[ii][jj][kk][ll], nan_policy='omit', axis=1 ).tolist() if modeRef == 'med' else \
                                                   np.nanstd(  VIPOUnF_allO[ii][jj][kk][ll], axis=1 ).tolist() )

            # get IQR for each outside dataset - incremental
            if( kk == 0 ):
                DataPOutSelT = DataPOut_df.loc[ DataPOut.Date.isin( [ np.array(DateOutRefT[kk]).astype('str').tolist() ] ).values ]  # selected data in agiven date
                DataMAll_dfT = DataM_df.append( DataPOutSelT, ignore_index=True )
            else:
                DataPOutSelT = DataPOutSelT.append( DataPOut_df.loc[ DataPOut.Date.isin( [ np.array(DateOutRefT[kk]).astype('str').tolist() ] ).values ], ignore_index=True )  # selected data in agiven date
                DataMAll_dfT = DataM_df.append( DataPOutSelT, ignore_index=True )
            IQRSumm[ii][jj].append( scipy.stats.iqr( DataMAll_dfT.Trait, nan_policy='omit' ) )  # last IQR corresponds to the joint complete domains target and source

## 4. Figures of the transferred models

### Figure 3, RMSE

RMSE relative to the IQR, per target date. *(Original cell 20.)*

In [ ]:
os.chdir( r'<PROJECT_ROOT>\DataProcessing\1_Script_source' )
import FakeParulaPalette as FakeParulaF
import matplotlib.patches
from   matplotlib.patches import Rectangle
import matplotlib.collections
from   matplotlib.collections import PathCollection
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator, AutoLocator, MaxNLocator)
minorLocator = AutoMinorLocator()
majorLocator = MaxNLocator(4)

# color - one for each domain / source data  ;  more divisions to get better color options (using only 2 inside and outside source)
Parula_manual   = []
seqSelT = np.round( np.arange(0, 256, 256/50 ) ).astype(int).tolist()
for val in range( len(seqSelT) ):
    Parula_manual.append( FakeParulaF.FakeParula_cm('FakeParula', FakeParulaF.cm_data)(seqSelT[val])[:-1] )

TraitRef  = ['DM_dt_ha', 'N_perc_DM', 'N_kg_ha' ]

TraitRefL  = [ r'Dry matter (DM) $[\/dt\/ha^{-1}\/]$',
               r'N content $[\/\%\/in\/DM\/]$',
               r'N total $[\/kg\/ha^{-1}\/]$'  ]
TraitRefL2 = ['Biomass','N','Ntotal']

matplotlib.rcParams['axes.labelpad']  = 10.0
matplotlib.rcParams['axes.linewidth'] = 0.8
matplotlib.rcParams['legend.fontsize'] = 9
matplotlib.rcParams['legend.handleheight'] = 0.6
matplotlib.rcParams['legend.handlelength'] = 0.6
matplotlib.rcParams['legend.handletextpad'] = 0.2
matplotlib.rcParams['legend.labelspacing'] = 0.2
matplotlib.rcParams['legend.columnspacing'] = 0.65
matplotlib.rcParams['font.family'] = 'Arial'

barWid = 0.30
barSpa = 0.05
SpaCntrl = 7

lwidth = 0.8
lwidth1 = 0.8

modeRef = 'mean'

YearRef = SpecAll_sel_0_1_res_Avg.Year.unique().tolist()
YearRef = YearRef + [YearRef]  # include period with all years...

DateRef = SpecAll_sel_0_1_res_Avg.Date.unique().tolist()
DateRefY = [ int(str(DateRef[val])[:4]) for val in range(len(DateRef)) ]

DateDOY = []
for elmnt in range(len(HarvRef)):  # date
    dateT = pd.to_datetime(HarvRef[elmnt], format='%Y%m%d')
    dateTRef = pd.Timestamp(year=dateT.year, month=1, day=1)
    DateDOY.append( (dateT - dateTRef).days + 1 )

# new
fontS0=9
fontS1=10
fontS2=11
fontS3=12

saveOpt = False

refRMSEOpt = 'iqr'

for ii in range( len(TraitRef) ):  # traits

    R2PlotT_all   = []
    RMSEPlotT_all = []
    ErPlotT_all   = []
    R2PlotOT_all   = []
    RMSEPlotOT_all = []
    ErPlotOT_all   = []
    RMSEPlotpOPropT_all = []
    for jj in range( len(R2RMSEvalSumm) ):  # dataset source

        # get values to plot
        for kk in range( len(R2RMSEvalSumm[jj][ii]) ):  # outside dataset  | acq

            # get values to plot
            ll = bestSampleSelF_allO[jj][ii][kk]
            R2PlotT_all.append( R2RMSValSumm[jj][ii][kk][ll,1] )  # source
            if( refRMSEOpt == 'iqr' ):
                RMSEPlotT_all.append(  R2RMSValSumm[jj][ii][kk][ll,2]/IQRSumm[jj][ii][-1] )
                ErPlotT_all.append( R2RMSValSummIqr[jj][ii][kk][ll,2]/IQRSumm[jj][ii][-1] )
            else:
                RMSEPlotT_all.append(  R2RMSValSumm[jj][ii][kk][ll,2] )
                ErPlotT_all.append( R2RMSValSummIqr[jj][ii][kk][ll,2] )

            R2PlotOT_all.append( R2RMSIndVSumm[jj][ii][kk][ll,1] )  # all
            if( refRMSEOpt == 'iqr' ):
                RMSEPlotOT_all.append(  R2RMSIndVSumm[jj][ii][kk][ll,2]/IQRSumm[jj][ii][kk] )
                ErPlotOT_all.append( R2RMSIndVSummIqr[jj][ii][kk][ll,2]/IQRSumm[jj][ii][kk] )
                RMSEPlotpOPropT_all.append(  np.nanmin(R2RMSIndVSummIqr[jj][ii][kk][:,2])/R2RMSIndVSummIqr[jj][ii][kk][ll,2] )
            else:
                RMSEPlotOT_all.append(  R2RMSIndVSumm[jj][ii][kk][ll,2] )
                ErPlotOT_all.append( R2RMSIndVSummIqr[jj][ii][kk][ll,2] )
                RMSEPlotpOPropT_all.append(  np.nanmin(R2RMSIndVSummIqr[jj][ii][kk][:,2])/R2RMSIndVSummIqr[jj][ii][kk][ll,2] )

        if( jj == 0 ):  # inverse order since 2014 (n=2) is used to predict 2017 (n=3) and vice-versa
            major_ticks = [ str(DateDOY[2:5][val]) for val in range(len(DateDOY[2:5])) ]
        else:
            major_ticks =  major_ticks + [ str(DateDOY[0:2][val]) for val in range(len(DateDOY[0:2])) ]

    # RMSE
    fig = pylab.figure( figsize=(2.5, 4) )
    fig.subplots_adjust(left=0.10, right=0.92, bottom=0.02, top=0.8)
    ax = fig.add_subplot(1, 1, 1)

    barPosA = [ 1, 1+((barSpa+barWid)*2)+(barSpa*SpaCntrl), (1+((barSpa+barWid)*2)+(barSpa*SpaCntrl))+(((barSpa+barWid)*2)+(barSpa*SpaCntrl)), \
             ((1+((barSpa+barWid)*2)+(barSpa*SpaCntrl))+(((barSpa+barWid)*2)+(barSpa*SpaCntrl)))+(((barSpa+barWid)*2)+(barSpa*SpaCntrl)), \
             (((1+((barSpa+barWid)*2)+(barSpa*SpaCntrl))+(((barSpa+barWid)*2)+(barSpa*SpaCntrl)))+(((barSpa+barWid)*2)+(barSpa*SpaCntrl))) + \
             (((barSpa+barWid)*2)+(barSpa*SpaCntrl)) ]
    barPos1 = [ ( (barPosA[val])-(barSpa+(barWid/2)) ) for val in range(len(barPosA)) ] # 1st bar
    barPos2 = [ ( (barPosA[val])+(barSpa+(barWid/2)) ) for val in range(len(barPosA)) ] # 2nd bar

    # plot bar plot
    currentSet = ax.barh( barPos1, RMSEPlotT_all, barWid, color='lime', edgecolor='k', linewidth=lwidth, xerr=[ [0]*len(ErPlotT_all), ErPlotT_all ],
                          error_kw=dict(elinewidth=lwidth, ecolor='k', capsize=1.4) )
    OuttSet    = ax.barh( barPos2, RMSEPlotOT_all, barWid, color='deepskyblue', edgecolor='k', linewidth=lwidth, xerr=[ [0]*len(ErPlotOT_all), ErPlotOT_all ],
                          error_kw=dict(elinewidth=lwidth, ecolor='k', capsize=1.4) )  # , color='tan' ; color=Parula_manual[42]
    plt.gca().invert_yaxis()

    ax.xaxis.tick_top()
    fig.text( 0.50 if ii == 2 else 0.50 if ii == 1 else 0.50, 0.88, "Average RMSE w.r.t. IQR",
              fontsize=fontS2, fontstyle='normal', va='center', ha='center', color='k')

    ax.set_xlim( -0.02, 1.5 )
    ax.set_ylim( barPosA[-1]+(barSpa+(barWid)+(barSpa*2)), 1-(barSpa+(barWid)+(barSpa*2)) )  # add the same space that is aailabe in the begining of the plot

    majorTRefT = [0.0, 0.25, 0.50, 0.75, 1.0]
    ax.set_xticks( majorTRefT  )
    ax.set_xticks( [ round(np.arange(0.0, 1.0, 0.05).tolist()[val], 3) for val in range(np.arange(0.0, 1.0, 0.05).shape[0]) if np.arange(0.0, 1.0, 0.05).tolist()[val] not in majorTRefT ], minor=True )

    # indicate 'plot end'
    ax.axvline( x=1.00, color='k', linestyle='-', linewidth=lwidth )
    ax.axvline( x=1.05, color='k', linestyle='-', linewidth=lwidth )

    ax.set_yticks(barPosA)
    ax.set_yticklabels( major_ticks, fontname = "Arial", ha='center' )
    ax.tick_params(axis='y', which='major', length=3.5, width=lwidth1, direction="in", labelsize=fontS2, pad=12.5)
    ax.tick_params(axis='x', which='major', length=3.5, width=lwidth1, direction="in", labelsize=fontS1)
    ax.tick_params(axis='x', which='minor', length=2.0, width=lwidth1, direction="in")

    fig.text( 0.990, 0.575, '[ source - 2014 ]',
           fontsize=fontS2, fontstyle='normal', va='center', ha='right', color='k', rotation=-90 )
    fig.text( -0.0225, 0.575, '2017 - target',
           fontsize=fontS2, fontstyle='normal', va='center', ha='right', color='k', rotation=90 )
    fig.text( 0.990, 0.175, '[ source - 2017 ]',
           fontsize=fontS2, fontstyle='normal', va='center', ha='right', color='k', rotation=-90 )
    fig.text( -0.0225, 0.175, '2014 - target',
           fontsize=fontS2, fontstyle='normal', va='center', ha='right', color='k', rotation=90 )
    fig.text( -0.0960, 0.41, "Day Of the Year - DOY ",
           fontsize=fontS2, fontstyle='normal', va='center', ha='right', color='k', rotation=90 )

    fig.text( 0.50 if (ii == 2) else 0.50 if (ii == 1) else 0.50, 0.93, TraitRefL[ii],
           fontsize=fontS3, fontstyle='oblique', va='center', ha='center', color='k')

    ax.axhline( y=(1+((barSpa+barWid)*2)+(barSpa*SpaCntrl))+(((barSpa+barWid)*2)+(barSpa*SpaCntrl))+((barSpa/2)+barWid+(barSpa*(SpaCntrl/2))), \
                xmin=0, xmax=(1/1.5), linewidth=lwidth, color='k', linestyle='-')
    ax.axhline( y=(1+((barSpa+barWid)*2)+(barSpa*SpaCntrl))+(((barSpa+barWid)*2)+(barSpa*SpaCntrl))+((barSpa/2)+barWid+(barSpa*(SpaCntrl/2))), \
                xmin=(1.06/1.5), xmax=1, linewidth=lwidth, color='k', linestyle='-')

    # Add pie-chart
    jittRefT  = 0.170
    jittRef2T = 0.162
    for val in range( len(RMSEPlotpOPropT_all) ):

        # define the ratios
        r1 = RMSEPlotpOPropT_all[val]
        r2 = r1 + (100-RMSEPlotpOPropT_all[val])

        # calculate the points of the pie marker
        x1 = np.cos(2 * np.pi * np.linspace(0, r1))
        y1 = np.sin(2 * np.pi * np.linspace(0, r1))
        xy1 = np.row_stack([[0, 0], np.column_stack([x1, y1])])
        x2 = np.cos(2 * np.pi * np.linspace(r1, r2))
        y2 = np.sin(2 * np.pi * np.linspace(r1, r2))
        xy2 = np.row_stack([[0, 0], np.column_stack([x2, y2])])
        ax.scatter([1.25], barPosA[val]-jittRefT, marker=xy2, s=325, facecolor='w',       edgecolors='none')
        ax.scatter([1.25], barPosA[val]-jittRefT, marker=xy1, s=325, facecolor='crimson', edgecolors='none')
        ax.scatter([1.25], barPosA[val]-jittRefT, marker='o', s=325, facecolor='none',    edgecolors='k', linewidths=lwidth)

        # write proportions
        fmt = '.1f'
        fig.text( 0.890, 0.700-(val*jittRef2T), format(r1*100, fmt) + '%',
                  fontsize=fontS2, fontstyle='normal', va='center', ha='right', color='k', rotation=0 )

    os.chdir(r'<PROJECT_ROOT>\1_JypiterLabProcessing\4_Figures')
    if( saveOpt == True ):
        if modeRef == 'med':
            plt.savefig( 'RMSE_PLSRAllOrd_med_Spike_' + TraitRefL2[ii] + '_' + compRefT + 'png', dpi=600, facecolor='w', edgecolor='w',
                      orientation='portrait', format=None,
                      transparent=False, bbox_inches='tight', pad_inches=0.1 )
        else:
            plt.savefig( 'RMSE_PLSRAllOrd_Spike_' + TraitRefL2[ii] + '_' + compRefT + 'png', dpi=600, facecolor='w', edgecolor='w',
                      orientation='portrait', format=None,
                      transparent=False, bbox_inches='tight', pad_inches=0.1 )

### Figure 3, R2

Average R2 and number of factors, per target date. *(Original cell 21.)*

In [ ]:
matplotlib.rcParams['axes.labelpad']  = 10.0
matplotlib.rcParams['axes.linewidth'] = 0.8
matplotlib.rcParams['font.family'] = 'Arial'

SpaCntrl = 0
barWid   = 0.125
barSpa   = 0.05
SecNdGP  = 1.75

saveOpt = False

for ii in range( len(TraitRef) ):  # traits

    # get values to plot
    R2PlotT   = []
    RMSEPlotT = []
    ErPlotT   = []
    R2PlotOT   = []
    RMSEPlotOT = []
    ErPlotOT   = []
    PLSFPlotT  = []
    for jj in range( len(R2RMSEvalSumm) ):  # dataset source
        for kk in range( len(R2RMSEvalSumm[jj][ii]) ):   # outside dataset  | acq

            ll = bestSampleSelF_allO[jj][ii][kk]
            R2PlotOT.append(  R2RMSIndVSumm[jj][ii][kk][ll,1] )  # valid (source) and outside samples (not usee during model training)
            PLSFPlotT.append( R2RMSEvalSumm[jj][ii][kk][ll,0] )

        if( jj == 0 ):  # inverse order since 2014 (n=2) is used to predict 2017 (n=3) and vice-versa
            major_ticks = [ str(DateDOY[2:5][val]) + '\n[ ' + str( round(PLSFPlotT[val]) ) + ' ]' for val in range(len(DateDOY[2:5])) ]
        else:
            major_ticks =  major_ticks + [ str(DateDOY[0:2][val]) + '\n[ ' + str( round(PLSFPlotT[ len(major_ticks)+val ]) ) + ' ]' for val in range(len(DateDOY[0:2])) ]

    # R2
    fig = pylab.figure( figsize=(2.5, 4) )
    fig.subplots_adjust(left=0.23, right=0.92, bottom=0.02, top=0.8)
    ax = fig.add_subplot(1, 1, 1)

    barPosA = [ 1-(barSpa+barWid), 1, 1+(barSpa+barWid), \
                SecNdGP+(barSpa*SpaCntrl)-(barSpa+barWid), SecNdGP+(barSpa*SpaCntrl) ]

# SecNdGP+(barSpa*SpaCntrl)-(barSpa+barWid), SecNdGP+(barSpa*SpaCntrl), SecNdGP+(barSpa*SpaCntrl)+(barSpa+barWid) ]

    # plot bar plot
    R2PlotOutT = [ R2PlotOT[val] if R2PlotOT[val]>0 else -0.01 for val in range(len(R2PlotOT)) ]
    currentSet = ax.barh( barPosA, R2PlotOutT, barWid, color='deepskyblue', edgecolor='k', linewidth=lwidth)  # , yerr=None, error_kw=dict(elinewidth=lwidth,ecolor='k'), color=Parula_manual[42]
    plt.gca().invert_yaxis()
    ax.xaxis.tick_top()

    fig.text( 0.60 if ii == 2 else 0.60 if ii == 1 else 0.60, 0.87, "Average $R^{2}$",
           fontsize=fontS2, fontstyle='normal', va='center', ha='center', color='k' )
    ax.set_xlim(-0.04, 1.00)
    ax.set_ylim( (SecNdGP+(barSpa*SpaCntrl)+(barSpa+barWid)), 1-((barSpa+barWid)+(barWid/2)+(barSpa*2)) )  # add the same space that is aailabe in the begining of the plot

    ax.tick_params(axis='y', which='major', length=3.5, width=lwidth1, direction="in", labelsize=fontS1, pad=15)
    ax.tick_params(axis='x', which='major', length=3.5, width=lwidth1, direction="in", labelsize=fontS1)
    ax.tick_params(axis='x', which='minor', length=2.0, width=lwidth1, direction="in")
    ax.set_xticks( np.arange(0,1+0.25, 0.25), minor=False)
    minor_ticks = [ round( np.arange(0, 1, 0.05)[val], 2) for val in range(len(np.arange(0, 1, 0.05))) ]
    minor_ticks = [ minor_ticks[val] for val in range(len(minor_ticks)) if minor_ticks[val] not in np.arange(0,1+0.25, 0.25) ]
    ax.set_xticks( minor_ticks, minor=True)

    ax.set_yticks(barPosA)

    ax.set_yticklabels(major_ticks, fontname = "Arial",  fontsize=fontS2, ha='center' )
    fig.text( 0.990, 0.575, '[ source - 2014 ]',
           fontsize=fontS2, fontstyle='normal', va='center', ha='right', color='k', rotation=-90 )
    fig.text( 0.065, 0.575, '2017 - target',
           fontsize=fontS2, fontstyle='normal', va='center', ha='right', color='k', rotation=90 )
    fig.text( 0.990, 0.175, '[ source - 2017 ]',
           fontsize=fontS2, fontstyle='normal', va='center', ha='right', color='k', rotation=-90 )
    fig.text( 0.065, 0.175, '2014 - target',
           fontsize=fontS2, fontstyle='normal', va='center', ha='right', color='k', rotation=90 )
    fig.text( -0.040, 0.41, "DOY [ Number of PLS factors ]",
       fontsize=fontS2, fontstyle='normal', va='center', ha='center', color='k', rotation=90 )

    fig.text( 0.60 if ii == 2 else 0.60 if ii == 1 else 0.60, 0.92, TraitRefL[ii],
           fontsize=fontS3, fontstyle='oblique', va='center', ha='center', color='k' )

    ax.axhline(y=(1+(barSpa+barWid))+(SecNdGP+(barSpa*SpaCntrl)-(barSpa+barWid)-(1+(barSpa+barWid)) )/2, \
            linewidth=lwidth, color='k', linestyle='-')

    os.chdir(r'<PROJECT_ROOT>\1_JypiterLabProcessing\4_Figures')
    if( saveOpt == True ):
        if modeRef == 'med':
            plt.savefig('R2_PLSRAllOrd_med_spike_' + TraitRefL2[ii] + '_' + compRefT + 'png', dpi=600, facecolor='w', edgecolor='w',
                      orientation='portrait', format=None,
                      transparent=False, bbox_inches='tight', pad_inches=0.1 )

        else:
            plt.savefig('R2_PLSRAllOrd_spike_' + TraitRefL2[ii] + '_' + compRefT + 'png', dpi=600, facecolor='w', edgecolor='w',
                      orientation='portrait', format=None,
                      transparent=False, bbox_inches='tight', pad_inches=0.1 )

### Figure 6

Sample-wise predictions of the transferred models. *(Original cell 22.)*

In [ ]:
os.chdir( r'<PROJECT_ROOT>\DataProcessing\1_Script_source' )
import FakeParulaPalette as FakeParulaF

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.graphics.regressionplots import abline_plot
from statsmodels.sandbox.regression.predstd import wls_prediction_std
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator, AutoLocator, MaxNLocator)
minorLocator = AutoMinorLocator()
majorLocator = MaxNLocator(4)

matplotlib.rcParams['axes.labelpad']  = 10.0
matplotlib.rcParams['axes.linewidth'] = 0.8
matplotlib.rcParams['font.family'] = 'Arial'

TraitRefL  = [ r'Dry matter $[\/dt\/\/ha^{-1}\/]$',
               r'N content $[\/\%\/in\/DM\/]$',
               r'N total $[\/kg\/\/ha^{-1}\/]$'  ]

fontS0 = 9
fontS1 = 10
fontS2 = 11
fontS3 = 12

FontSizeG = 10
FontSizeT = 11

matplotlib.rcParams['axes.linewidth']  = 0.8
matplotlib.rcParams['font.family']     = 'Arial'
matplotlib.rcParams['legend.fontsize'] = 8
matplotlib.rcParams['legend.handlelength'] = 0.8

MajorTicksLength = 3.5
MinorTicksLength = 2.2
errorLineWidth   = 1.2

lwidth =  0.6

TraitRef    = [ 'DM_dt_ha', 'N_perc_DM', 'N_kg_ha' ]
TraitRefAll = [ 'DM_dt_ha', 'N_perc_DM', 'CrudeProt_perc_DM', 'N_kg_ha', 'CrudeProt_kg_ha' ]

YLimAllObs = [ [[0, 60], [0.90, 3.90], [5, 105]],
               [[0, 60], [0.90, 3.90], [5, 105]],
               [[0, 60], [0.90, 3.90], [5, 105]] ]

YLimAllObs_new = [ [[0, 60], [0.90, 3.90], [-15, 105]],
                   [[0, 60], [0.90, 3.90], [-15, 105]],
                   [[0, 60], [0.90, 3.90], [-15, 105]] ]

axisIntDistT   = [ [20, 0.75, 25], [20, 0.75, 25], [20, 0.75, 25] ]
axisIntDisLenT = [ [7, 5, 5], [7, 5, 5], [7, 5, 5] ]
axisIntRoundFT = [ ["%.1f", "%.2f", "%.1f"], ["%.1f", "%.2f", "%.1f"], ["%.1f", "%.2f", "%.1f"] ]

YearRef = SpecAll_sel_0_1_res_Avg.Year.unique().tolist()
DateRef = SpecAll_sel_0_1_res_Avg.Date.unique().tolist()
DateRefY = [ int(str(DateRef[val])[:4]) for val in range(len(DateRef)) ]

DateDOY = []
for elmnt in range(len(DateRef)):  # date
    dateT = pd.to_datetime(DateRef[elmnt], format='%Y%m%d')
    dateTRef = pd.Timestamp(year=dateT.year, month=1, day=1)
    DateDOY.append( (dateT - dateTRef).days + 1 )

cmap   = matplotlib.cm.get_cmap('BrBG')
colorR = np.arange(0, 1+0.2, 0.2)
colorR = np.arange(0, 0.3+0.15, 0.15).tolist() + colorR[4:].tolist()

Parula_manual   = [ plt.matplotlib.cm.get_cmap('Greens')(0.70), plt.matplotlib.cm.get_cmap('Greens')(0.50), \
                   'blue', 'dodgerblue', 'lightskyblue']
colorR = list( range( len(Parula_manual) ) )

validTargetRefT = 'xkcd:golden yellow'

modeRef = 'mean'

saveOpt = True

for ii in range(len(TraitRef)):  # traits
 for jj in range(len(PredObs)):  # dataset source
    if( jj == 0 ) :
        DateOutRefT =  HarvRef[2:5]
    else:
        DateOutRefT =  HarvRef[0:2]
    if( ii == 0 ):
        DataP    = SpecAll_sel_0_1_res_Avg.loc[ (SpecAll_sel_0_1_res_Avg.Year).astype(int) == int(YearRef[jj]), :]
        DataPOut = SpecAll_sel_0_1_res_Avg.loc[ ~((SpecAll_sel_0_1_res_Avg.Year).astype(int) == int(YearRef[jj])), :]
        DataDim = DataP.shape[0]/len(PredObs)
    else:
        DataP    = SpecAll_sel_0_1_res_Avg_treat.loc[ (SpecAll_sel_0_1_res_Avg_treat.Year).astype(int) == int(YearRef[jj]), :]
        DataPOut = SpecAll_sel_0_1_res_Avg_treat.loc[~((SpecAll_sel_0_1_res_Avg_treat.Year).astype(int) == int(YearRef[jj])), :]
        DataDim = DataP.shape[0]/len(PredObs)

    if( jj == 0 ):
        DateDOYSource = DateDOY[0:2]
        DateDOYTarget = DateDOY[2:5]
        colorRSource  = colorR[0:2]
    else:
        DateDOYSource = DateDOY[2:5]
        DateDOYTarget = DateDOY[0:2]
        colorRSource  = colorR[2:5]

    DataM_df    = DataP.loc[ :, (DataP.columns == TraitRef[ii]) | DataP.columns.isin(col_ref) ]
    DataPOut_df = DataPOut.loc[ :, (DataPOut.columns == TraitRef[ii]) | DataPOut.columns.isin(col_ref) ]
    BRef = [ 'b' + str(range(len(col_ref))[val]) if range(len(col_ref))[val] > 9 else 'b0' + str( range(len(col_ref))[val] ) for val in range(len(col_ref)) ]
    DataM_df.columns = ['Trait'] + BRef
    DataM_df = DataM_df.reset_index(drop=True)  # not needed since reference made by index, but doing anyway...
    DataPOut_df.columns = ['Trait'] + BRef
    DataPOut_df = DataPOut_df.reset_index(drop=True)  # not needed since reference made by index, but doing anyway...

    for kk in range( len(PredObs[jj][ii]) ):   # outside dataset | acq date

        # for ll in range(len(PredObs[jj][ii][kk])):  # nbr of samples add from the outside domain
        ll = bestSampleSelF_allO[jj][ii][kk]

        TestOT  = PredObs[jj][ii][kk][ll][0]
        ValiOT  = PredObsVali[jj][ii][kk][ll][0]
        ETestOT = PredEObs[jj][ii][kk][ll][0]
        OutOT   = PredObsOut[jj][ii][kk][ll][0]
        TestOIqrT  = PredObsIqr[jj][ii][kk][ll][0]
        ValiOIqrT  = PredObsValiIqr[jj][ii][kk][ll][0]
        ETestOIqrT = PredEObsIqr[jj][ii][kk][ll][0]
        OutOIqrT   = PredObsOutIqr[jj][ii][kk][ll][0]

        # inverse selection (masks) - based on NaN values
        sampTestT  = np.isnan( TestOT ).tolist()  # validation samples
        sampValiT  = np.isnan( ValiOT ).tolist()  # calibration samples
        sampETestT = np.logical_not(np.isnan( OutOT )).tolist()
        OutOTsub   = OutOT[ int(((kk+1)*DataDim)-DataDim):int(((kk+1)*DataDim)) ]
        sampOutT   = np.isnan( OutOTsub ).tolist()  # spike (used for calibration)

        fig, ax = plt.subplots(figsize=(2.3, 2.3) )
        fig.subplots_adjust(left=0.23, right=0.94 , bottom = 0.22, top = 0.90)

        DateRT = DataP.Date.unique().tolist()
        DataPlotColoT = []
        for val in range(len(DateRT)):
            # calib
            TestOTS    = np.array(TestOT)[ ( (DataP.Date == DateRT[val]) & sampValiT ) ]
            TestIqrOTS = np.array(TestOIqrT)[ ( (DataP.Date == DateRT[val]) & sampValiT ) ]
            TraitTTS   = DataM_df.loc[ ( DataP.reset_index(drop=True).Date == DateRT[val] ) & sampValiT, : ].Trait.tolist()

            # valid
            ValiOTS    = np.array(ValiOT)[((DataP.Date == DateRT[val]) & sampTestT)]
            ValiIqrOTS = np.array(ValiOIqrT)[((DataP.Date == DateRT[val]) & sampTestT)]
            TraitTVS   = DataM_df.loc[ (DataP.reset_index(drop=True).Date == DateRT[val]) & sampTestT, : ].Trait.tolist()

            # Compile complete dataset to be plot
            DataToPT      = np.append( np.array( [TestOTS, TestIqrOTS, TraitTTS ] ).T, np.array( [ValiOTS, ValiIqrOTS, TraitTVS] ).T, axis=0 )
            DataPlotColoT = DataPlotColoT + [ Parula_manual[colorRSource[val]] ]*len(TestOTS) + ['red']*len(ValiOTS)  # colors to be used
            if( val == 0 ):
                DataPlotCompT = copy.deepcopy(DataToPT)
            else:
                DataPlotCompT = np.append( DataPlotCompT, DataToPT, axis=0 )

        DateRT = DataPOut.Date.unique().tolist()
        for val in range(len(DateRT)):  # date - outside | only independent data
            ETestOTS    = np.array(OutOT)[ ( (DataPOut.Date == DateRT[val]) & sampETestT ) ]
            ETestIqrOTS = np.array(OutOIqrT)[ ( (DataPOut.Date == DateRT[val]) & sampETestT ) ]
            ETraitTTS   = DataPOut_df.loc[ ( DataPOut.reset_index(drop=True).Date == DateRT[val] ) & sampETestT, : ].Trait.tolist()
            DataPlotColoT = DataPlotColoT + [validTargetRefT]*len(ETestOTS)
            DataPlotCompT = np.append( DataPlotCompT, np.array( [ETestOTS, ETestIqrOTS, ETraitTTS] ).T, axis=0 )

        # plot points randonly for better visualization
        IdxToPT = list( range( DataPlotCompT.shape[0] ) )
        random.seed( [132563, 23015, 76][ii] )
        IdxToPT = random.sample(IdxToPT, k=len(IdxToPT))
        for val in range( len(IdxToPT) ):
            ax.errorbar( DataPlotCompT[IdxToPT[val],0], DataPlotCompT[IdxToPT[val],2], xerr=DataPlotCompT[IdxToPT[val],1], fmt='none', color=DataPlotColoT[IdxToPT[val]],
                         mfc='None', markersize=4, linewidth =errorLineWidth, capsize=1.40, elinewidth=errorLineWidth, markeredgewidth=errorLineWidth,
                         label= 'All data', zorder=3 )

        ax.set_ylim( YLimAllObs_new[jj][ii][0],  YLimAllObs_new[jj][ii][-1] )
        ax.set_xlim( YLimAllObs_new[jj][ii][0],  YLimAllObs_new[jj][ii][-1] )

        XPlot = [ [ np.min( list(ax.get_xlim()) + YLimAllObs[jj][ii] ), np.max( list(ax.get_xlim()) + YLimAllObs[jj][ii] )], \
                  [ np.min( list(ax.get_xlim()) + YLimAllObs[jj][ii] ), np.max( list(ax.get_xlim()) + YLimAllObs[jj][ii] )] ]
        ax.plot( XPlot[0], XPlot[1], 'k-',  linewidth=0.8, zorder=1 )  # 1:1 line
        ax.set_xlim( ax.get_xlim() )

        # Plot line passing over independent data predicted with the model (valid + outside)
        DataTL = pd.DataFrame( np.array([ np.append(DataP.loc[sampTestT, :].eval(TraitRef[ii]),  DataPOut.loc[sampETestT , :].eval(TraitRef[ii])), \
                                          np.append(np.array(ValiOT)[sampTestT], np.array(OutOT)[sampETestT]) ]).T , columns = ['Obs', 'Pred'] )
        results = smf.ols('Obs ~ Pred', data=DataTL).fit()
        MParam = results.params
        XPlot2 =  [ XPlot[0][0]-(0.1*XPlot[0][1]), XPlot[0][1]+(0.1*XPlot[0][1]) ]
        ax.plot( XPlot2, [ MParam.Intercept + MParam.Pred*XPlot2[val] for val in range(len(XPlot2))], '--', color='red', linewidth=0.8, zorder=1 )  # regression line

        # add R2 and RMSE corresponding to 100 models predicting the validation data
        fig.text( 0.2525, 0.930, '$R^{2}$ = ' + str("%.2f" % round(R2RMSIndVSumm[jj][ii][kk][ll,1], 3)) if R2RMSIndVSumm[jj][ii][kk][ll,1] > 0 else \
                               'R-sqrd. = ' + str("%.2f" % round(0.00, 3)), va='center', rotation=0, size=FontSizeG, \
                  fontname = "Arial", fontstyle='normal', fontweight='normal', color='k')
        fig.text( 0.51, 0.930, ' ; RMSE = ' + str("%.2f" % round(R2RMSIndVSumm[jj][ii][kk][ll,2], 3)), va='center', rotation=0, size=FontSizeG, \
                  fontname = "Arial", fontstyle='normal', fontweight='normal', color='k')

        axisIntT = np.arange( YLimAllObs[jj][ii][0], YLimAllObs[jj][ii][-1]+axisIntDistT[jj][ii], axisIntDistT[jj][ii] ).round( 3 ).tolist()
        ax.set_yticks( axisIntT[ 0:axisIntDisLenT[jj][ii] ] )
        ax.set_yticklabels( [ str( axisIntRoundFT[jj][ii] % axisIntT[val]) for val in range(len( axisIntT[0:axisIntDisLenT[jj][ii]] )) ], fontname = "Arial",  fontsize=fontS2, ha='center' )
        ax.set_xticks(  axisIntT[0:axisIntDisLenT[jj][ii]] )
        ax.set_xticklabels( [ str( axisIntRoundFT[jj][ii] % axisIntT[val]) for val in range(len( axisIntT[0:axisIntDisLenT[jj][ii]] )) ], fontname = "Arial",  fontsize=fontS2, ha='center', rotation=45 )

        minorLocator1 = AutoMinorLocator()
        ax.plot( YLimAllObs[jj][ii], YLimAllObs[jj][ii], 'none' )
        ax.yaxis.set_minor_locator(minorLocator1)
        minorLocator2 = AutoMinorLocator()
        ax.plot( YLimAllObs[jj][ii], YLimAllObs[jj][ii], 'none' )
        ax.xaxis.set_minor_locator(minorLocator2)

        ax.tick_params(axis='y', which='major', width=0.8, length=MajorTicksLength, direction ='in', labelsize=FontSizeG, pad=13)
        ax.tick_params(axis='y', which='minor', width=0.8, length=MinorTicksLength, direction ='in', labelsize=FontSizeG, pad=13)
        ax.tick_params(axis='x', which='major', width=0.8, length=MajorTicksLength, direction ='in', labelsize=FontSizeG, pad=2)
        ax.tick_params(axis='x', which='minor', width=0.8, length=MinorTicksLength, direction ='in', labelsize=FontSizeG, pad=2)

        ax.grid(b=1, which='major', axis='both', color='gray', alpha=0.4, linestyle='dotted', linewidth=0.8, zorder=0)
        ax.grid(b=1, which='minor', axis='both', color='gray', alpha=0.2, linestyle='dotted', linewidth=0.8, zorder=0)

        fig.text( 0.01, 0.55, TraitRefL[ii] + ' - ' + 'Observ.', va='center', rotation=90, size=FontSizeG, fontname = "Arial", fontstyle='normal', fontweight='normal')
        fig.text( 0.55, 0.04, TraitRefL[ii] + ' - ' + 'Pred.', ha='center', va='center', rotation=0, \
                  size=FontSizeG, fontname = "Arial", fontstyle='normal', fontweight='normal')

        if( saveOpt == True ):
            os.chdir( r'<PROJECT_ROOT>\1_JypiterLabProcessing\4_Figures' )
            plt.savefig("VI_ScatterYearAll_CalVal_ER_PLSR_spike_" + str([2014, 2017, 'All'][jj]) + "_Best_" + TraitRef[ii] + '_' + DateOutRefT[kk] + '_' + compRefT + "png", dpi=600, facecolor='w', edgecolor='w', \
                        orientation='portrait', format=None, \
                        transparent=False, bbox_inches='tight', pad_inches=0.05) # , bbox_inches='tight'

### Figure 4

RMSE according to the number of samples added from the target date. *(Original cell 24.)*

In [ ]:
os.chdir( r'<PROJECT_ROOT>\DataProcessing\1_Script_source' )
import FakeParulaPalette as FakeParulaF
import matplotlib.patches as patches
from matplotlib.collections import PatchCollection

matplotlib.rcParams['axes.labelpad']  = 10.0
matplotlib.rcParams['axes.linewidth'] = 0.8
matplotlib.rcParams['font.family'] = 'Arial'

fontS1 = 12
fontS2 = 13
MajorTicksLength = 3.5
MinorTicksLength = 2.2
errorLineWidth   = 1.75

# distance between results for a given nbr of extra samples
distOutRef = 2

TraitRef   = [ 'DM_dt_ha', 'N_perc_DM', 'N_kg_ha' ]
TraitRefL  = [ 'Dry matter $[\/dt\/\/ha^{-1}\/]$',
               'N content $[\/\%\/in\/DM\/]$',
               'N total $[\/kg\/\/ha^{-1}\/]$'  ]

YearRef = SpecAll_sel_0_1_res_Avg.Year.unique().tolist()
DateRef = SpecAll_sel_0_1_res_Avg.Date.unique().tolist()
DateRefY = [ int(str(DateRef[val])[:4]) for val in range(len(DateRef)) ]

DateDOY = []
for elmnt in range(len(DateRef)):  # date
    dateT = pd.to_datetime(DateRef[elmnt], format='%Y%m%d')
    dateTRef = pd.Timestamp(year=dateT.year, month=1, day=1)
    DateDOY.append( (dateT - dateTRef).days + 1 )

Parula_manual   = [ plt.matplotlib.cm.get_cmap('Greens')(0.70), plt.matplotlib.cm.get_cmap('Greens')(0.50), \
                   'blue', 'dodgerblue', 'lightskyblue']
colorR = list( range( len(Parula_manual) ) )

pltType = 'Other'  # for detailed DM plot
if( pltType == 'fullExt' ):
    yAxis_LimRef = [ [2, 30, 4], [0.1, 0.7, 0.15], [2, 16, 2] ]
else:
    yAxis_LimRef = [ [2, 8, 2], [0.1, 0.7, 0.15],  [2, 16, 2] ]

saveOpt = False

# threshold for selecting best case = (min + ((max-min)*threshold))
thrsholdRMSEOpt = 0.0

for jj in range( len(R2RMSOutSumm[0]) ):  # trait

    if( jj==0 ):
        fig = pylab.figure( figsize=(10, 2.5) )
    else:
        fig = pylab.figure( figsize=(3.3, 2.5) )
    fig.subplots_adjust(left=0.1, right=0.95, bottom=0.1, top=0.95)
    gs  = matplotlib.gridspec.GridSpec(3, 1, height_ratios=[1, 1, 6])
    gs.update( wspace=0.0, hspace=0.085 )
    ax0 = plt.subplot(gs[2])
    ax1 = plt.subplot(gs[1])
    ax2 = plt.subplot(gs[0])

    xAxis_valRefT = 0
    for ll in range( R2RMSOutSumm[0][jj][kk].shape[0] ):  # nbr of samples added to the training data
        for ii in range( len(R2RMSOutSumm) ):
            if( ii == 0 ):
                DateDOYTarget = DateDOY[2:5]
                colorRTarget  = colorR[2:5]
            else:
                DateDOYTarget = DateDOY[0:2]
                colorRTarget  = colorR[0:2]

            for kk in range( len(R2RMSOutSumm[ii][0]) ):

                # add indication of best nbr of samples for transfer
                idealSelT = np.nanmin( R2RMSIndVSumm[ii][jj][kk][:,-1] ) + ((np.nanmax( R2RMSIndVSumm[ii][jj][kk][:,-1] ) - np.nanmin( R2RMSIndVSumm[ii][jj][kk][:,-1] ))*thrsholdRMSEOpt)
                idealSelT = np.where( np.abs(R2RMSIndVSumm[ii][jj][kk][:,-1]-idealSelT) == np.nanmin(np.abs(R2RMSIndVSumm[ii][jj][kk][:,-1]-idealSelT)) )[0][0]
                if(ii == 0):
                    posRefT = (kk+0.5)
                else:
                    posRefT = len(R2RMSOutSumm[0][0])+(kk+0.5)
                rect1 = patches.Rectangle( (((idealSelT*(5+distOutRef))+distOutRef+posRefT), 0), width=1, height=1 )
                rectA  = [rect1]
                pc1 = PatchCollection(rectA, linewidth=errorLineWidth, edgecolor='none', facecolor=Parula_manual[ colorRTarget[kk] ], alpha=1, zorder=1 )
                ax2.add_collection(pc1)

                # add error bar
                if( all([ (ii == 0), (kk == 0) ]) ):
                    xAxis_valRefT += distOutRef+1
                else:
                    xAxis_valRefT += 1
                dataToPT = R2RMSIndVSumm[ii][jj][kk][ll,-1]
                devToPT  = R2RMSIndVSummIqr[ii][jj][kk][ll,-1]
                ax0.errorbar( xAxis_valRefT, dataToPT, yerr=devToPT, fmt='none', color = Parula_manual[ colorRTarget[kk] ],
                             mfc='None', markersize=4, linewidth =errorLineWidth, capsize=3.0, elinewidth=errorLineWidth, markeredgewidth=errorLineWidth,
                             label= 'DOY [' + str(YearRef[ii]) + '] = ' + str(DateDOYTarget[kk]), zorder=2 )

    # Add indication of final choice - point or patch
    for ii in range( len(R2RMSOutSumm) ):
        if( ii == 0 ):
            DateDOYTarget = DateDOY[2:5]
            colorRTarget  = colorR[2:5]
        else:
            DateDOYTarget = DateDOY[0:2]
            colorRTarget  = colorR[0:2]
        for kk in range( len(R2RMSOutSumm[ii][0]) ):
            if(ii == 0):
                posRefT = (kk+0.5)
            else:
                posRefT = len(R2RMSOutSumm[0][0])+(kk+0.5)
            rect1 = patches.Rectangle( (((bestSampleSelF_allO[ii][jj][kk]*(5+distOutRef))+distOutRef+posRefT), 0), width=1, height=1 )
            rectA  = [rect1]
            pc1 = PatchCollection(rectA, linewidth=errorLineWidth, edgecolor='none', facecolor=Parula_manual[ colorRTarget[kk] ], alpha=1 )
            ax1.add_collection(pc1)

    # color of axis auxiliary plots
    topColRefT    = 'xkcd:yellowish orange'
    bottomColRefT = 'red'

    ax1.spines['left'].set_color(bottomColRefT)
    ax1.spines['right'].set_color(bottomColRefT)
    ax1.spines['top'].set_color(bottomColRefT)
    ax1.spines['bottom'].set_color(bottomColRefT)
    ax2.spines['left'].set_color(topColRefT)
    ax2.spines['right'].set_color(topColRefT)
    ax2.spines['top'].set_color(topColRefT)
    ax2.spines['bottom'].set_color(topColRefT)

    # add tick marks - X-axis
    xMarksRefT = list( range( 5, (5*R2RMSOutSumm[ii][jj][kk].shape[0])+5, 5 ) )
    xMarksRefT = [ xMarksRefT[val]+(val*distOutRef) for val in range(len(xMarksRefT)) ]
    ax0.set_xticks( xMarksRefT )
    major_ticks = list( range( clstCntrlRef, (clstCntrlRef*R2RMSOutSumm[ii][jj][kk].shape[0])+clstCntrlRef, clstCntrlRef ) )
    ax0.set_xticklabels( major_ticks, fontname = "Arial",  fontsize=fontS1 )
    if( jj==0 ):
        ax0.set_xlim( xMarksRefT[0]-4, xMarksRefT[-1]+4 )
    ax1.set_xlim( ax0.get_xlim() )
    ax1.set_xticks([])
    ax2.set_xlim( ax0.get_xlim() )
    ax2.set_xticks([])

    # add tick marks - Y-axis
    fmt = '.2f'
    ax0.set_yticks( list( np.arange( yAxis_LimRef[jj][0], yAxis_LimRef[jj][1]+yAxis_LimRef[jj][2], yAxis_LimRef[jj][2] ) ) )
    major_ticks = list( np.arange( yAxis_LimRef[jj][0], (yAxis_LimRef[jj][1]+yAxis_LimRef[jj][2]), yAxis_LimRef[jj][2] ) )
    if( jj == 1 ):
        fmt = '.2f'
    else:
        fmt = '.1f'
    major_ticks = [ format(major_ticks[val], fmt) for val in range(len(major_ticks)) ]
    ax0.set_yticklabels( major_ticks, fontname = "Arial",  fontsize=fontS1 )

    if( all([ (jj == 0), (pltType == 'fullExt') ]) ):
        ax0.set_ylim( yAxis_LimRef[jj][0:2][0]-1, yAxis_LimRef[jj][0:2][1]+1 )
    elif( (jj == 0) ):
        ax0.set_ylim( yAxis_LimRef[jj][0:2] )
    ax1.set_yticks([])
    ax2.set_yticks([])

    ax0.tick_params(axis='y', which='major', width=1.0, length=MajorTicksLength, direction ='in', labelsize=fontS1, pad=5)
    ax0.tick_params(axis='x', which='major', width=1.0, length=MajorTicksLength, direction ='in', labelsize=fontS1, pad=5)

    # Add X and Y-Axis titles
    if( jj == 0 ):
        if( pltType == 'fullExt' ):
            xLoc_yAxis, yLoc_yAxis = 0.045, 0.5
            xLoc_xAxis, yLoc_xAxis = 0.5, -0.085
        else:
            xLoc_yAxis, yLoc_yAxis = 0.050, 0.5
            xLoc_xAxis, yLoc_xAxis = 0.5, -0.085
    else:
        xLoc_yAxis, yLoc_yAxis = -0.065, 0.5
        xLoc_xAxis, yLoc_xAxis = 0.5, -0.085
    fig.text( xLoc_yAxis, yLoc_yAxis, 'RMSE - ' + TraitRefL[jj],                           va='center', ha='center',
              rotation=90, size=fontS2, fontname = "Arial", fontstyle='normal', fontweight='normal')
    fig.text( xLoc_xAxis, yLoc_xAxis, 'Number of Samples from the Target\nDomain Added to the Training Dataset', va='center', ha='center',
              rotation=0,  size=fontS2, fontname = "Arial", fontstyle='normal', fontweight='normal')

    # Add division between number of samples
    xDivRefT = list( np.arange( 5, (5*R2RMSOutSumm[ii][jj][kk].shape[0]), 5 ) )
    xDivRefT = [ xDivRefT[val]+((val+1)*2) for val in range(len(xDivRefT)) ]
    if( distOutRef % 2 == 0 ):
        xDivRefT = [ xDivRefT[val]+(distOutRef/2)+0.5 for val in range(len(xDivRefT)) ]
    else:
        xDivRefT = [ xDivRefT[val]+(distOutRef/2) for val in range(len(xDivRefT)) ]
    for val in range(len(xDivRefT)):
        ax0.axvline( x=xDivRefT[val], color='k', linewidth=1.0, linestyle='dashed' )
        ax1.axvline( x=xDivRefT[val], color=bottomColRefT, linewidth=1.0, linestyle='dashed' )
        ax2.axvline( x=xDivRefT[val], color=topColRefT, linewidth=1.0, linestyle='dashed' )

    # plot detail for illustrating lower part of DM plot
    if( all([ (pltType == 'fullExt'), (jj==0) ]) ):
        ax0.axhline(y=8, color='k', linestyle='-', linewidth=0.8)

    # change axis line width
    for sideT in ax0.spines.keys():
        ax0.spines[sideT].set_linewidth(1.0)
        ax1.spines[sideT].set_linewidth(1.0)
        ax2.spines[sideT].set_linewidth(1.0)

    if( saveOpt == True ):
        os.chdir(r'<PROJECT_ROOT>\1_JypiterLabProcessing\4_Figures')
        plt.savefig('RMSE_AddSamples_' + TraitRef[jj] + '_' + compRefT + 'png', dpi=600, facecolor='w', edgecolor='w',
                     orientation='portrait', format=None,
                     transparent=False, bbox_inches='tight', pad_inches=0.1)

### Figure 5

RMSE when selecting samples with and without prediction uncertainty. *(Original cell 25.)*

In [ ]:
os.chdir( r'<PROJECT_ROOT>\DataProcessing\1_Script_source' )
import FakeParulaPalette as FakeParulaF
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator, AutoLocator, MaxNLocator)
minorLocator = AutoMinorLocator()
majorLocator = MaxNLocator(4)

matplotlib.rcParams['axes.linewidth'] = 0.8
matplotlib.rcParams['font.family']    = 'Arial'

# load RMSE for assessing the performance of the sample selection methods
os.chdir( r'<PROJECT_ROOT>\1_JypiterLabProcessing\5_Outputs' )

versionRef = '106'
EmodeRef   = 'centroid'  # attention (latest version should be run in both 'modes')
with open('RMSEIndVF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    RMSEIndVF_allO_cent = pickle.load(f)
EmodeRef   = 'stdS'
with open('RMSEIndVF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    RMSEIndVF_allO_stdS = pickle.load(f)
with open('bestSampleSelF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
    bestSampleSelF_allO_stdS = pickle.load(f)

# Prepare data output formats
R2RMSIndVSumm_stdS = []
R2RMSIndVSumm_cent = []
R2RMSIndVSummIqr_stdS = []
R2RMSIndVSummIqr_cent = []
for ii in range( len(RMSEIndVF_allO_stdS) ):  # source dataset | year
    R2RMSIndVSumm_stdST = []
    R2RMSIndVSumm_centT = []
    R2RMSIndVSummIqr_stdST = []
    R2RMSIndVSummIqr_centT = []
    for jj in range( len(RMSEIndVF_allO_stdS[ii]) ):  # trait
        R2RMSIndVSumm_stdSTT = []
        R2RMSIndVSumm_centTT = []
        R2RMSIndVSummIqr_stdSTT = []
        R2RMSIndVSummIqr_centTT = []
        for kk in range( len(RMSEIndVF_allO_stdS[ii][jj]) ):  # date in the outside dataset - target
            R2RMSIndVSumm_stdSTT.append(  np.full( (len(RMSEIndVF_allO_stdS[ii][jj][kk]),3), np.nan ) )
            R2RMSIndVSumm_centTT.append(  np.full( (len(RMSEIndVF_allO_cent[ii][jj][kk]),3), np.nan ) )
            R2RMSIndVSummIqr_stdSTT.append(  np.full( (len(RMSEIndVF_allO_stdS[ii][jj][kk]),3), np.nan ) )
            R2RMSIndVSummIqr_centTT.append(  np.full( (len(RMSEIndVF_allO_cent[ii][jj][kk]),3), np.nan ) )
        R2RMSIndVSumm_stdST.append( R2RMSIndVSumm_stdSTT )
        R2RMSIndVSumm_centT.append( R2RMSIndVSumm_centTT )
        R2RMSIndVSummIqr_stdST.append( R2RMSIndVSummIqr_stdSTT )
        R2RMSIndVSummIqr_centT.append( R2RMSIndVSummIqr_centTT )
    R2RMSIndVSumm_stdS.append( R2RMSIndVSumm_stdST )
    R2RMSIndVSumm_cent.append( R2RMSIndVSumm_centT )
    R2RMSIndVSummIqr_stdS.append( R2RMSIndVSummIqr_stdST )
    R2RMSIndVSummIqr_cent.append( R2RMSIndVSummIqr_centT )

# get summary for R2, RMSE and predictions
TraitRef  = ['DM_dt_ha', 'N_perc_DM', 'N_kg_ha' ]
HarvRef  = [ '20140515', '20141015', '20170510', '20170830', '20171026' ]
DateRefY = [ '2014', '2017' ]
col_ref  = list( range(len(center_origin)) )
modeRef = 'mean'

for ii in range( len(RMSEIndVF_allO_stdS) ):  # dataset source
    for jj in range( len(RMSEIndVF_allO_stdS[ii]) ):  # trait
        for kk in range( len(RMSEIndVF_allO_stdS[ii][jj]) ):  # outside dataset | target
            for ll in range( len(RMSEIndVF_allO_stdS[ii][jj][kk]) ) :  # nbr of samples added from the target domain

                # central descriptor - only for RMSE for the moment
                R2RMSIndVSumm_stdS[ii][jj][kk][ll,2] = np.nanmedian( RMSEIndVF_allO_stdS[ii][jj][kk][ll] ) if modeRef == 'med' else np.nanmean( RMSEIndVF_allO_stdS[ii][jj][kk][ll] )
                R2RMSIndVSumm_cent[ii][jj][kk][ll,2] = np.nanmedian( RMSEIndVF_allO_cent[ii][jj][kk][ll] ) if modeRef == 'med' else np.nanmean( RMSEIndVF_allO_cent[ii][jj][kk][ll] )

                # deviation descriptor
                R2RMSIndVSummIqr_stdS[ii][jj][kk][ll,2] = scipy.stats.iqr( RMSEIndVF_allO_stdS[ii][jj][kk][ll], nan_policy='omit' )  if modeRef == 'med' else \
                                                                np.nanstd( RMSEIndVF_allO_stdS[ii][jj][kk][ll] )

                R2RMSIndVSummIqr_cent[ii][jj][kk][ll,2] = scipy.stats.iqr( RMSEIndVF_allO_cent[ii][jj][kk][ll], nan_policy='omit' )  if modeRef == 'med' else \
                                                                np.nanstd( RMSEIndVF_allO_cent[ii][jj][kk][ll] )

# Plot results for (cluster-based + uncertainty driven) selection in contrast to (cluster-based) only
fontS1 = 10
fontS2 = 11
MajorTicksLength = 3.5
MinorTicksLength = 2.2
errorLineWidth   = 1.75

TraitRef   = [ 'DM_dt_ha', 'N_perc_DM', 'N_kg_ha' ]
TraitRefL  = [ 'Dry matter $[\/dt\/\/ha^{-1}\/]$',
               'N content $[\/\%\/in\/DM\/]$',
               'N total $[\/kg\/\/ha^{-1}\/]$'  ]

YearRef = SpecAll_sel_0_1_res_Avg.Year.unique().tolist()
DateRef = SpecAll_sel_0_1_res_Avg.Date.unique().tolist()
DateRefY = [ int(str(DateRef[val])[:4]) for val in range(len(DateRef)) ]

DateDOY = []
for elmnt in range(len(DateRef)):  # date
    dateT = pd.to_datetime(DateRef[elmnt], format='%Y%m%d')
    dateTRef = pd.Timestamp(year=dateT.year, month=1, day=1)
    DateDOY.append( (dateT - dateTRef).days + 1 )

Parula_manual   = [ plt.matplotlib.cm.get_cmap('Greens')(0.70), plt.matplotlib.cm.get_cmap('Greens')(0.50), \
                   'blue', 'dodgerblue', 'lightskyblue']
colorR = list( range( len(Parula_manual) ) )

pltType = 'fullExt'
if( pltType == 'fullExt' ):
    xyAxis_LimRef = [ [2, 6.4, 1], [0.1, 0.5, 0.1], [3, 12, 2] ]
else:
    xyAxis_LimRef = [ [2, 4.2, 1], [0.1, 0.5, 0.1], [3, 12, 2] ]

saveOpt = True

for jj in range( len(TraitRef) ):  # trait
    fig, ax = plt.subplots(figsize=(2.3, 2.3) )
    fig.subplots_adjust(left=0.23, right=0.94 , bottom = 0.22, top = 0.90)
    for ii in range( len(R2RMSIndVSumm_stdS) ):  # source dataset
        if( ii == 0 ):
            DateDOYTarget = DateDOY[2:5]
            colorRTarget  = colorR[2:5]
        else:
            DateDOYTarget = DateDOY[0:2]
            colorRTarget  = colorR[0:2]
        for kk in range( len(R2RMSIndVSumm_stdS[ii][jj]) ):  # target dataset

            ll=bestSampleSelF_allO_stdS[ii][jj][kk]

            dataToPT_1 = copy.deepcopy( R2RMSIndVSumm_stdS[ii][jj][kk][ll,-1] )
            devToPT_1  = copy.deepcopy( R2RMSIndVSummIqr_stdS[ii][jj][kk][ll,-1] )
            dataToPT_2 = copy.deepcopy( R2RMSIndVSumm_cent[ii][jj][kk][ll,-1] )
            devToPT_2  = copy.deepcopy( R2RMSIndVSummIqr_cent[ii][jj][kk][ll,-1] )
            ax.errorbar( dataToPT_2, dataToPT_1, xerr=devToPT_2, yerr=devToPT_1, fmt='none', color = Parula_manual[ colorRTarget[kk] ],
                         mfc='None', markersize=4, linewidth =errorLineWidth, capsize=3.0, elinewidth=errorLineWidth, markeredgewidth=errorLineWidth,
                         label= 'DOY [' + str(YearRef[ii]) + '] = ' + str(DateDOYTarget[kk]), zorder=2 )

    ax.plot( xyAxis_LimRef[jj][0:2], xyAxis_LimRef[jj][0:2], linestyle='-', color='k', linewidth=0.8, zorder=1 )

    # add tick marks - X- and Y-axis
    ax.set_xticks( list( np.arange( xyAxis_LimRef[jj][0], xyAxis_LimRef[jj][1]+xyAxis_LimRef[jj][2], xyAxis_LimRef[jj][2] ) ) )
    ax.set_yticks( list( np.arange( xyAxis_LimRef[jj][0], xyAxis_LimRef[jj][1]+xyAxis_LimRef[jj][2], xyAxis_LimRef[jj][2] ) ) )
    fmt = '.2f'
    major_ticks = list( np.arange( xyAxis_LimRef[jj][0], (xyAxis_LimRef[jj][1]+xyAxis_LimRef[jj][2]), xyAxis_LimRef[jj][2] ) )
    if( jj == 1 ):
        fmt = '.2f'
    else:
        fmt = '.1f'
    major_ticks = [ format(major_ticks[val], fmt) for val in range(len(major_ticks)) ]
    ax.set_xticklabels( major_ticks, fontname = "Arial",  fontsize=fontS1 )
    ax.set_yticklabels( major_ticks, fontname = "Arial",  fontsize=fontS1 )

    minorLocator1 = AutoMinorLocator()
    ax.xaxis.set_minor_locator(minorLocator1)
    ax.yaxis.set_minor_locator(minorLocator1)

    ax.set_xlim( xyAxis_LimRef[jj][0:2] )
    ax.set_ylim( xyAxis_LimRef[jj][0:2] )

    ax.tick_params(axis='y', which='major', width=0.8, length=MajorTicksLength, direction ='in', labelsize=fontS1, pad=5)
    ax.tick_params(axis='x', which='major', width=0.8, length=MajorTicksLength, direction ='in', labelsize=fontS1, pad=5)
    ax.tick_params(axis='y', which='minor', width=0.8, length=MinorTicksLength, direction ='in', labelsize=fontS1, pad=5)
    ax.tick_params(axis='x', which='minor', width=0.8, length=MinorTicksLength, direction ='in', labelsize=fontS1, pad=5)

    # Add X and Y-Axis titles
    xLoc_yAxis, yLoc_yAxis = 0.000, 0.55
    xLoc_xAxis, yLoc_xAxis = 0.575, 0.05
    fig.text( xLoc_yAxis, yLoc_yAxis, 'RMSE - ' + TraitRefL[jj] + '\n(Diversity and Uncertainty)', va='center', ha='center',
              rotation=90, size=fontS2, fontname = "Arial", fontstyle='normal', fontweight='normal')
    fig.text( xLoc_xAxis, yLoc_xAxis, 'RMSE - ' + TraitRefL[jj] + '\n(Diversity)', va='center', ha='center',
              rotation=0,  size=fontS2, fontname = "Arial", fontstyle='normal', fontweight='normal')

    ax.grid(b=1, which='major', axis='both', color='gray', alpha=0.4, linestyle='dotted', linewidth=0.8)
    ax.grid(b=1, which='minor', axis='both', color='gray', alpha=0.2, linestyle='dotted', linewidth=0.8)

    # plot detail for illustrating lower part of DM plot
    if( all([ (pltType == 'fullExt'), (jj==0) ]) ):
        ax.axhline(y=4.2, xmax=(4.2-2)/(6.4-2), color='k', linestyle='-', linewidth=0.8)
        ax.axvline(x=4.2, ymax=(4.2-2)/(6.4-2), color='k', linestyle='-', linewidth=0.8)

    # change axis line width
    for sideT in ax0.spines.keys():
        ax.spines[sideT].set_linewidth(1.0)

    if( saveOpt == True ):
        os.chdir(r'<PROJECT_ROOT>\1_JypiterLabProcessing\4_Figures')
        plt.savefig('RMSE_TransfPerf_' + TraitRef[jj] + '_' + compRefT + 'png', dpi=600, facecolor='w', edgecolor='w',
                     orientation='portrait', format=None,
                     transparent=False, bbox_inches='tight', pad_inches=0.1)

## 5. Feature selection

### Parameters

Settings of the VIP confidence interval. *(Original cell 30.)*

In [ ]:
DateRef  = SpecAll_sel_0_1_res_Avg.Date.unique().tolist()
DateRefY = np.unique( [ int(str(DateRef[val])[:4]) for val in range(len(DateRef)) ] ).tolist()

# indicate reference for the calculation of residuals
VIP_refMod = 'mean'  # 'full'  >>> 'mean' = average of boot VIP values  ;  'full' =  VIP for model with all samples

# indicate which value to use for N in the std formula (bootstrap repetitions or nbr of samples)
Nref = 'BootMod'

# significance level for Martens Uncertainty Test (MUT; Forina et al., 2004) - CI | t-test
coefSelThrsld = 0.1

### Models with the selected bands

Retains the bands whose lower VIP confidence limit is above one and refits the transferred models with them. *(Original cell 31.)*

In [ ]:
R2EvalF_varSel_all       = []
R2EEvalF_varSel_all      = []
R2ValF_varSel_all        = []
R2OutF_varSel_all        = []
R2IndF_varSel_all        = []
R2IndVF_varSel_all       = []
RMSEEvalF_varSel_all     = []
RMSEEEvalF_varSel_all    = []
RMSEValF_varSel_all      = []
RMSEOutF_varSel_all      = []
RMSEIndF_varSel_all      = []
RMSEIndVF_varSel_all     = []
TestUnF_varSel_all       = []
ETestUnF_varSel_all      = []
EvalUnF_varSel_all       = []
ValiUnF_varSel_all       = []
OutDUnF_varSel_all       = []
CoefUnF_varSel_all       = []
VIPUnF_varSel_all        = []
InterBooF_varSel_all     = []
PLSRFactF_varSel_all     = []
CIselVIPRefF_varSel_all  = []
VIPRetainedF_varSel_all  = []
CIOutF_varSel_all        = []
for ii in range( len(VIPOUnF_allO) ):  # dataset source |  year
    if( ii == 0 ) :
        DateOutRefT =  HarvRef[2:5]
    else:
        DateOutRefT =  HarvRef[0:2]

    R2EvalF        = []
    R2EEvalF       = []
    R2ValF         = []
    R2OutF         = []
    R2IndF         = []
    R2IndVF        = []
    RMSEEvalF      = []
    RMSEEEvalF     = []
    RMSEValF       = []
    RMSEOutF       = []
    RMSEIndF       = []
    RMSEIndVF      = []
    TestUnF        = []
    ETestUnF       = []
    EvalUnF        = []
    ValiUnF        = []
    OutDUnF        = []
    CoefUnF        = []
    VIPOUnF        = []
    InterBooF      = []
    PLSRFactF      = []
    CIselVIPRefF   = []
    VIPRetainedF   = []
    CIOutF         = []
    for jj in range( len(VIPOUnF_allO[ii]) ):  # trait
        if( jj == 0 ): # Dry Biomass data / 49 observations per date
            DataP    = SpecAll_sel_0_1_res_Avg.loc[ (SpecAll_sel_0_1_res_Avg.Year).astype(int) == int(DateRefY[ii]), :]  # source data
            GroupF   = [ str(DataP.Replicate.tolist()[val]) + '_' + str(DataP.Treatment.tolist()[val]) for val in range(DataP.shape[0]) ]  # reference to each plot...
            DataPOut = SpecAll_sel_0_1_res_Avg.loc[ ~((SpecAll_sel_0_1_res_Avg.Year).astype(int) == int(DateRefY[ii])), :]  # data outside the source domain (different season)
        else:  # N data / 15 observations per date
            DataP    = SpecAll_sel_0_1_res_Avg_treat.loc[ (SpecAll_sel_0_1_res_Avg_treat.Year).astype(int) == int(DateRefY[ii]), :]
            GroupF   = DataP.Treatment.tolist()
            DataPOut = SpecAll_sel_0_1_res_Avg_treat.loc[~((SpecAll_sel_0_1_res_Avg_treat.Year).astype(int) == int(DateRefY[ii])), :]
        DataM_df    = DataP.loc[ :, (DataP.columns == TraitRef[jj]) | DataP.columns.isin(col_ref) ]  # select trait of interest + spectra
        DataPOut_df = DataPOut.loc[ :, (DataPOut.columns == TraitRef[jj]) | DataPOut.columns.isin(col_ref) ]  # data outside the source domain (different season/year)
        BRef = [ 'b' + str(range(len(col_ref))[val]) if range(len(col_ref))[val] > 9 else 'b0' + str(range(len(col_ref))[val]) for val in range(len(col_ref)) ]
        DataM_df.columns = ['Trait'] + BRef
        DataM_df = DataM_df.reset_index(drop=True)
        DataPOut_df.columns = ['Trait'] + BRef
        DataPOut_df = DataPOut_df.reset_index(drop=True)

        # select calibration and validation datasets - source domain | valid = 0.3*n
        sampleValF   = sampleValF_allO[ii]
        testIF_Oall  = sampleValF_allO[ii][jj]
        trainIF_Oall = list( set( list(range( DataM_df.shape[0] )) ) - set( testIF_Oall ) )

        # get calibration and validation sets
        Cal_df   = DataM_df.loc[ trainIF_Oall,: ]
        GroupFTr = [ GroupF[val] for val in range(len(GroupF)) if val in trainIF_Oall ]
        Val_df   = DataM_df.loc[ testIF_Oall,: ]
        GroupFTt = [ GroupF[val] for val in range(len(GroupF)) if val in testIF_Oall ]

        # Create objects to store predictions
        TestUn  = []
        EvalUn  = []
        ValiUn  = []
        ETestUn = []
        OutDUn  = []
        VIPOUn  = []
        CoefUn  = []
        for elmnt in range( len(DateOutRefT) ):   # date in the outside dataset (target)
            TestUn.append(  np.full( (DataM_df.shape[0], len(seedRepTmF[jj])), np.nan ) )      # samples in the calib x repetitions (boot)
            EvalUn.append(  np.full( (DataM_df.shape[0], len(seedRepTmF[jj])), np.nan ) )
            ValiUn.append(  np.full( (DataM_df.shape[0], len(seedRepTmF[jj])), np.nan ) )
            ETestUn.append( np.full( (DataPOut_df.shape[0], len(seedRepTmF[jj])), np.nan ) )
            OutDUn.append(  np.full( (DataPOut_df.shape[0], len(seedRepTmF[jj])), np.nan ) )  # extra samples in the calib x repetitions (boot) - using full array dimensions (standard dimensions)
            VIPOUn.append(  np.full( (len(BRef), len(seedRepTmF[jj])), np.nan ) )
            CoefUn.append(  np.full( (len(BRef), len(seedRepTmF[jj])), np.nan ) )

        R2EvalT        = []
        R2EEvalT       = []
        R2ValT         = []
        R2OutT         = []
        R2IndT         = []
        R2IndVT        = []
        RMSEEvalT      = []
        RMSEEEvalT     = []
        RMSEValT       = []
        RMSEOutT       = []
        RMSEIndT       = []
        RMSEIndVT      = []
        PLSRFactT      = []
        InterBooT      = []
        CIselVIPRefT   = []
        VIPRetainedT   = []
        CIOutT         = []
        Cal_dfUpInT = []  # progressive - samples added for each date (kk)
        for kk in pbar.progressbar( range( len(DateOutRefT) ) ):   # date in the outside dataset (target)
            DataPOutSelT = DataPOut_df.loc[ DataPOut.reset_index(drop=True).Date.isin( [ np.array(DateOutRefT[kk]).astype('str').tolist() ] ) ]  # selected dataset for a given target date

            # nbr of samples to add from the external dataset (target domain)
            ll = bestSampleSelF_allO[ii][jj][kk]
            bestSampleSelTT  = bestSampleSelF_allO[ii][jj]

            # samples selected based on clustering analysis + uncertainty
            selSampleRefAllT = selSampleRefAllF_allO[ii][jj][kk]

            # add samples from the external dataset - incremental up to the optimal number of samples
            for val in range( (ll+1) ):
                if( all([ (val==0) ]) ):
                    Cal_dfUpInT     = Cal_dfUpInT + [ DataPOutSelT.loc[ selSampleRefAllT[val], :] ]
                else:
                    Cal_dfUpInTemp  = DataPOutSelT.loc[ selSampleRefAllT[val], :]
                    Cal_dfUpInT[kk] = Cal_dfUpInT[kk].append( Cal_dfUpInTemp )  # not ignoring indices >> keep original row numbers  -  used for reference

            # VIP values from model derived using all samples - use as reference instead of VIP_meanT

            # add samples to the training data - in this case no out of bag samples - full dataset used
            Cal_dfUpT_full      = Cal_df.copy(deep=True)  # updated dataset
            DataPOut_dfUpT_full = []
            selSampleRefAllTT   = selSampleRefAllF_allO[ii][jj]
            for elmnt in range( len(Cal_dfUpInT) ):  # date in the outside dataset (target)
                if( elmnt != kk ):  # previous dates
                    selSampleRefAllTT_sel = itertools.chain(*selSampleRefAllTT[elmnt][:(bestSampleSelTT[elmnt]+1) ])
                    selSampleRefAllTT_sel = [ True if Cal_dfUpInT[elmnt].index.tolist()[val] in selSampleRefAllTT_sel else False for val in range( Cal_dfUpInT[elmnt].shape[0] ) ]
                    Cal_dfUpT_full = Cal_dfUpT_full.append( Cal_dfUpInT[elmnt].loc[ selSampleRefAllTT_sel,: ], ignore_index=True )  # only optimal nbr of samples (for previous dates) - added at the end of the DataFrame
                else:  # current date
                    Cal_dfUpT_full = Cal_dfUpT_full.append( Cal_dfUpInT[elmnt], ignore_index=True )   # all samples selected until now (for current date)

            # VIP values - full dataset
            CoefUn_full  = np.full( (len(BRef), len(seedRepTmF[jj])), np.nan )
            VIPOUn_full  = np.full( (len(BRef), len(seedRepTmF[jj])), np.nan )
            PLSRFactTemp = PLSRFactF_allO[ii][jj][kk][ll]
            for elmnt in range( len(PLSRFactTemp) ):  # bootstrap repetition  ;  only difference is the number of factors used in PLSR model
                plsM_full = PLSRegression( n_components=PLSRFactTemp[elmnt], scale=ScaleOpt )
                plsM_full.fit( Cal_dfUpT_full.loc[:,BRef].values,  Cal_dfUpT_full.loc[:,'Trait'].values.reshape( (Cal_dfUpT_full.loc[:,'Trait'].shape[0],1) ) )
                CoefUn_full[ :, elmnt] = [ plsM_full.coef_.tolist()[val][0] for val in range(len(plsM_full.coef_.tolist())) ]
                VIPOUn_full[ :, elmnt] = vip(plsM_full)

            # feature selection - VIP Conf. Intervals
            VIPUn_sel = VIPOUnF_allO[ii][jj][kk][ll]
            if( VIP_refMod == 'mean' ):  # mean for all boot repetitions
                VIP_meanT = np.nanmean( VIPUn_sel, axis=1 )
                VIP_meanT = np.repeat( np.expand_dims(VIP_meanT,1), VIPUn_sel.shape[1], axis=1 )
            else:  # using actual coefficients for complete dataset (changes due to nbr of PLSR factors...)
                VIP_meanT = copy.deepcopy( VIPOUn_full )
            VIP_residualT = ( VIPUn_sel - VIP_meanT )
            VIP_sqrResidualT = np.nansum( (VIP_residualT*VIP_residualT), axis=1 )
            if( Nref == 'BootMod' ):
                VIP_sdv_boot = np.sqrt( (VIP_sqrResidualT / (BootSNbr - 1)) )  # bootstrap
            else:
                VIP_sdv_boot = np.sqrt( (VIP_sqrResidualT / (Cal_dfUpT.shape[0] - 1)) )

            # use conf. interval to evaluate coefficient-wise significance - Afanador et al. (2014)
            CIselVIPRefTemp =  ['reject']*len(BRef)   # indication if conf. interval for the hypotehesis of averages equal 0 was accepted or rejected (significant coef.)
            CIOutTemp  = []
            quatileT   = 1-( (coefSelThrsld)/2 )
            for elmnt in range( VIPUn_sel.shape[0] ):  # feature
                if(  Nref == 'BootMod' ):
                    tQuantOT   = scipy.stats.t.ppf( q=quatileT, df=(BootSNbr-1) )  # t-statistic student
                    CI_paramOT = ( np.nanmean( VIPUn_sel[elmnt, :] ) - tQuantOT*VIP_sdv_boot[elmnt], np.nanmean( VIPUn_sel[elmnt, :] ) + tQuantOT*VIP_sdv_boot[elmnt] )
                else:
                    tQuantOT   = scipy.stats.t.ppf( q=quatileT, df=(Cal_dfUpT.shape[0]-1) )  # t-statistic student
                    CI_paramOT = ( np.nanmean( VIPUn_sel[elmnt, :] ) - tQuantOT*VIP_sdv_boot[elmnt], np.nanmean( VIPUn_sel[elmnt, :] ) + tQuantOT*VIP_sdv_boot[elmnt] )
                CIselVIPRefTemp[ elmnt ] = ( 'reject' if (CI_paramOT[0] > 1) else 'accept' )
# indicate if hypotehsis of VIP equal 0 coef. should be rejected (significant coef) or accepted (insignificant coef)
                CIOutTemp.append( CI_paramOT )
            VIPRetainedT.append( np.where( (np.array(CIselVIPRefTemp) == 'reject') )[0].tolist() )

            # currently active features - update
            BRef_sel = np.array(BRef)[ VIPRetainedT[-1] ].tolist()

            # create array to store the outputs - Part II - one array per iteration (storing model coefficients, ...)
            R2EvalTemp  = []
            R2EEvalTemp = []
            R2ValTemp   = []
            R2OutTemp   = []
            R2IndTemp   = []
            R2IndVTemp  = []
            RMSEEvalTemp  = []
            RMSEEEvalTemp = []
            RMSEValTemp   = []
            RMSEOutTemp   = []
            RMSEIndTemp   = []
            RMSEIndVTemp  = []
            PLSRFactTemp  = []
            InterBooTemp  = []
            CoefUn_curr   = np.full( (len(BRef), len(seedRepTmF[jj])), np.nan )  # used for calculations within each iteration - even for the first one, which is not saved
            for mm in range( len(sampleBooF_allO[ii][jj][kk][ll]) ):  # repetition for bootstrap (according to trait)

                # get model fitting and test sets >> in and out of bag samples - only source data
                Cal_dfT   = Cal_df.copy(deep=True)
                GroupFTrT = copy.deepcopy(GroupFTr)
                sampleBooTemp = sampleBooF_allO[ii][jj][kk][ll][mm]   # reference to test set within each boot repetition - attention: based on row nbr not index
                testIFidx = Cal_dfT.iloc[sampleBooTemp,:].index.tolist()
                Cal_dfT      = Cal_df.drop( labels = testIFidx )
                GroupFTrTidx = [ val for val in range(Cal_df.shape[0]) if Cal_df.index.tolist()[val] not in testIFidx ]
                GroupFTrT    = [ GroupFTr[val] for val in GroupFTrTidx ]
                Eval_dfT     = Cal_df.loc[ testIFidx, :]

                # add samples to the training data
                Cal_dfUpT      = Cal_dfT.copy(deep=True)  # updated dataset
                DataPOut_dfUpT = []
                selSampleRefAllTT = selSampleRefAllF_allO[ii][jj]
                for elmnt in range( len(Cal_dfUpInT) ):  # date in the outside dataset (target)
                    DataPOutSelT = DataPOut_df.loc[ DataPOut.reset_index(drop=True).Date.isin( [ np.array(DateOutRefT[elmnt]).astype('str').tolist() ] ) ]  # selected data for a given target date - outside samples
                    if( elmnt != kk ):  # previous dates
                        selSampleRefAllTT_sel = itertools.chain(*selSampleRefAllTT[elmnt][:(bestSampleSelTT[elmnt]+1) ])
                        selSampleRefAllTT_sel = [ True if Cal_dfUpInT[elmnt].index.tolist()[val] in selSampleRefAllTT_sel else False for val in range( Cal_dfUpInT[elmnt].shape[0] ) ]
                        Cal_dfUpT = Cal_dfUpT.append( Cal_dfUpInT[elmnt].loc[ selSampleRefAllTT_sel,: ], ignore_index=True )  # only optimal nbr of samples (for previous dates) - added at the end of the DataFrame
                        OutSampleIdxRefT = Cal_dfUpInT[elmnt].loc[ selSampleRefAllTT_sel,: ].index.tolist()  # samples included in the calibration model
                    else:  # current date
                        Cal_dfUpT = Cal_dfUpT.append( Cal_dfUpInT[elmnt], ignore_index=True )   # all samples selected until now (for current date)
                        OutSampleIdxRefT = Cal_dfUpInT[elmnt].index.tolist()  # samples included in the calibration model
                    # remove selected samples from available pool - incremental (date by date)
                    if( len(DataPOut_dfUpT) == 0 ):
                        DataPOut_dfUpT = DataPOutSelT.loc[ [ True if DataPOutSelT.index.tolist()[val] not in OutSampleIdxRefT else False for val in range(DataPOutSelT.shape[0]) ],: ]
                    else:
                        DataPOut_dfUpT = DataPOut_dfUpT.append( DataPOutSelT.loc[ [ True if DataPOutSelT.index.tolist()[val] not in OutSampleIdxRefT else False for val in range(DataPOutSelT.shape[0]) ],: ], \
                                                                ignore_index=False )

                # outputs according to number of PLS factors - not saved for posterior use , only outputs for best model are retained
                RMSETO     = []
                R2TO       = []
                RMSETO_Out = []
                R2TO_Out   = []
                nbrFactMax = np.min( [ min(Cal_dfUpT.shape), nbrFactMax_thrshld, len(BRef_sel) ] )  # maximun nbr of PLS factors should be <= nbr of samples
                for elmnt in range( (nbrFactMax-nbrFactMin)+1 ):  # nbr of PLS factors to try
                    plsM = PLSRegression( n_components=range(nbrFactMin, nbrFactMax+1, 1)[elmnt], scale=ScaleOpt )
                    plsM.fit( Cal_dfUpT.loc[:,BRef_sel].values,  Cal_dfUpT.loc[:,'Trait'].values.reshape( (Cal_dfUpT.loc[:,'Trait'].shape[0],1) ) )
                    # evaluation over the source domain
                    PredEvalO = [ plsM.predict(Eval_dfT.loc[:,BRef_sel].values).tolist()[val][0] for val in range(len(plsM.predict(Eval_dfT.loc[:,BRef_sel].values).tolist())) ]
                    ResT =  Eval_dfT.Trait - np.array(PredEvalO)
                    uVal = (ResT**2).sum()
                    vVal = ((Eval_dfT.Trait.values.reshape(-1, 1) - Eval_dfT.Trait.values.reshape(-1, 1).mean())**2).sum()
                    R2TO.append( (1-(uVal/vVal)) )
                    RMSETO.append( np.sqrt( uVal/Eval_dfT.shape[0] ) )
                    # evaluation over the outside data in the source domain (calibration set - source)
                    DataPOut_df_EvalT = Cal_dfUpT.iloc[ (Cal_dfT.shape[0]):(Cal_dfUpT.shape[0]), :]  # only samples outside dataset - incremental
                    PredEvalO = plsM.predict(DataPOut_df_EvalT.loc[:,BRef_sel].values).tolist()
                    PredEvalO = [ PredEvalO[val][0] for val in range( len(PredEvalO) ) ]
                    ResT =  DataPOut_df_EvalT.Trait - np.array(PredEvalO)
                    uVal = (ResT**2).sum()
                    vVal = ((DataPOut_df_EvalT.Trait.values.reshape(-1, 1) - DataPOut_df_EvalT.Trait.values.reshape(-1, 1).mean())**2).sum()
                    R2TO_Out.append( (1-(uVal/vVal)) )
                    RMSETO_Out.append( np.sqrt( uVal/DataPOut_df_EvalT.shape[0] ) )

                # select factors | compromise between source and target accuracy
                RMSETO_Out_sortedT = list( enumerate( RMSETO_Out ) )
                RMSETO_Out_sortedT = sorted( RMSETO_Out_sortedT, key=lambda x:x[1], reverse=True )
                RMSETO_Out_sortedTRef = [ RMSETO_Out_sortedT[val][0] for val in range(len(RMSETO_Out_sortedT)) ]
                RMSETO_Out_sortedT    = [ RMSETO_Out_sortedT[val][1] for val in range(len(RMSETO_Out_sortedT)) ]
                RMSETO_sortedT = [ RMSETO[ RMSETO_Out_sortedTRef[val] ] for val in range(len(RMSETO_Out_sortedTRef)) ]
                RMSETO_joinT   = [ RMSETO_Out_sortedT[val]+RMSETO_sortedT[val] for val in range(len(RMSETO_Out_sortedT)) ]      # sum of RMSE for source and target domains
                nbrFactSel = range(nbrFactMin, nbrFactMax+1, 1)[ np.where( (RMSETO_joinT == np.nanmin(RMSETO_joinT)) )[0][0] ]
                PLSRFactTemp.append( nbrFactSel )
                plsM = PLSRegression( n_components=nbrFactSel, scale=ScaleOpt )
                plsM.fit( Cal_dfUpT.loc[:,BRef_sel].values,  Cal_dfUpT.loc[:,'Trait'].values.reshape( (Cal_dfUpT.loc[:,'Trait'].shape[0],1) ) )

                # save predictions for selected samples

                # source data - complete calibration set
                TestUn[kk][ Cal_dfT.index.tolist(), mm] = plsM.predict( Cal_dfT.loc[:,BRef_sel].values ).reshape(len(Cal_dfT.index.tolist()),)

                # outside samples added to the training set
                for elmnt in range( len(Cal_dfUpInT) ):  # date in the outside dataset (target)
                    # results from previous dates and current date are stored together - best set of samples for previous dates (need to be selected) + incremental dataset for current date
                    if( elmnt != kk ):  # previous dates
                        selSampleRefAllTT_sel = itertools.chain(*selSampleRefAllTT[elmnt][:(bestSampleSelTT[elmnt]+1) ])
                        selSampleRefAllTT_sel = [ True if Cal_dfUpInT[elmnt].index.tolist()[val] in selSampleRefAllTT_sel else False for val in range( Cal_dfUpInT[elmnt].shape[0] ) ]  # reference to best set of samples for previous dates
                        ETestUn[kk][ Cal_dfUpInT[elmnt].loc[ selSampleRefAllTT_sel, :].index.tolist(), mm] = \
                            plsM.predict( Cal_dfUpInT[elmnt].loc[ selSampleRefAllTT_sel, BRef_sel ].values ).reshape( Cal_dfUpInT[elmnt].loc[ selSampleRefAllTT_sel, :].shape[0], )
                    else:   # current date
                        ETestUn[kk][ Cal_dfUpInT[elmnt].index.tolist(), mm] = plsM.predict( Cal_dfUpInT[elmnt].loc[:,BRef_sel].values ).reshape( Cal_dfUpInT[elmnt].shape[0], )
                EvalUn[kk][ testIFidx, mm] = plsM.predict( Eval_dfT.loc[:,BRef_sel].values).reshape(len(testIFidx),)                                         # test samples - source dataset - boot repetition
                ValiUn[kk][ Val_df.index.tolist(), mm] = plsM.predict(Val_df.loc[:,BRef_sel].values).reshape(len(Val_df.index.tolist()),)                    # valid samples - source dataset
                OutDUn[kk][ DataPOut_dfUpT.index.tolist(), mm] = plsM.predict( DataPOut_dfUpT.loc[:,BRef_sel].values ).reshape( DataPOut_dfUpT.shape[0], )

                # get model parameters
                VIPOUn[kk][ VIPRetainedT[-1], mm] = vip(plsM)
                CoefUn[kk][ VIPRetainedT[-1], mm] = [ plsM.coef_.tolist()[val][0] for val in range(len(plsM.coef_.tolist())) ]
                InterBooTemp.append( (plsM._y_mean - np.dot(plsM._x_mean, plsM.coef_)).tolist()[0] )
                # https://stackoverflow.com/questions/41188410/how-obtain-the-intercept-of-the-pls-regression-sklearn

                # R2 and RMSE for calib set - Eval (source) - out-of-bag samples
                PredT = copy.deepcopy( EvalUn[kk][testIFidx, mm] )
                ResT = Eval_dfT.Trait.values - np.array( PredT )
                uVal = (ResT**2).sum()
                vVal = ((Eval_dfT.Trait.values.reshape(-1, 1) - Eval_dfT.Trait.values.reshape(-1, 1).mean())**2).sum()
                R2EvalTemp.append( (1-(uVal/vVal)) )
                RMSEEvalTemp.append( np.sqrt( uVal/Eval_dfT.shape[0] ) )

                # R2 and RMSE for external dataset - added to the training data - EEval
                PredT = copy.deepcopy( ETestUn[kk][ :, mm] )
                ECal_dfUpT    = np.empty( ETestUn[kk].shape[0], )  # same as ETestUn[kk][ll][:,mm] but for traits not predictions
                ECal_dfUpT[:] = np.nan
                for elmnt in range( len(Cal_dfUpInT) ):  # date in the outside dataset (target)
                    # samples from previous dates and current date are stored together - best set of samples for previous dates (need to be selected) + incremental dataset (for current date)
                    if( elmnt != kk ):  # previous dates
                        selSampleRefAllTT_sel = itertools.chain(*selSampleRefAllTT[elmnt][:(bestSampleSelTT[elmnt]+1) ])
                        selSampleRefAllTT_sel = [ True if Cal_dfUpInT[elmnt].index.tolist()[val] in selSampleRefAllTT_sel else False for val in range( Cal_dfUpInT[elmnt].shape[0] ) ]  # reference to best set of samples for previous dates
                        ECal_dfUpT[ Cal_dfUpInT[elmnt].loc[ selSampleRefAllTT_sel, :].index.tolist(), ] = \
                            Cal_dfUpInT[elmnt].loc[ selSampleRefAllTT_sel,'Trait'].values.reshape( Cal_dfUpInT[elmnt].loc[ selSampleRefAllTT_sel, :].shape[0], )
                    else:   # current date
                        ECal_dfUpT[ Cal_dfUpInT[elmnt].index.tolist(), ] = Cal_dfUpInT[elmnt].loc[:,'Trait'].values.reshape( Cal_dfUpInT[elmnt].shape[0], )
                ResT = ECal_dfUpT - np.array( PredT )
                uVal = np.nansum( (ResT**2) )
                vVal = np.nansum( ((ECal_dfUpT.reshape(-1, 1) - np.nanmean(ECal_dfUpT.reshape(-1, 1)))**2) )
                R2EEvalTemp.append( (1-(uVal/vVal)) )
                RMSEEEvalTemp.append( np.sqrt( uVal/(ECal_dfUpT.shape[0]-np.isnan(ECal_dfUpT).sum()) ) )

                # R2 and RMSE - validation set - source
                PredT = copy.deepcopy( ValiUn[kk][ Val_df.index.tolist(), mm] )
                ResT = Val_df.Trait.values - np.array( PredT )
                uVal = (ResT**2).sum()
                vVal = ((Val_df.Trait.values.reshape(-1, 1) - Val_df.Trait.values.reshape(-1, 1).mean())**2).sum()
                R2ValTemp.append( (1-(uVal/vVal)) )
                RMSEValTemp.append( np.sqrt( uVal/Val_df.shape[0] ) )

                # R2 and RMSE - outside dataset - not included in the training data
                PredT = copy.deepcopy( OutDUn[kk][ DataPOut_dfUpT.index.tolist(), mm] )
                ResT = DataPOut_dfUpT.Trait.values - np.array( PredT )
                uVal = (ResT**2).sum()
                vVal = ((DataPOut_dfUpT.Trait.values.reshape(-1, 1) - DataPOut_dfUpT.Trait.values.reshape(-1, 1).mean())**2).sum()
                R2OutTemp.append( (1-(uVal/vVal)) )
                RMSEOutTemp.append( np.sqrt( uVal/ DataPOut_dfUpT.shape[0] ) )

                # R2 and RMSE - all independent data (source + target) - careful source samples are out-of-bag not validation
                ResT = np.append( (DataPOut_dfUpT.Trait.values - OutDUn[kk][ DataPOut_dfUpT.index.tolist(), mm]), \
                                  (Eval_dfT.Trait.values - EvalUn[kk][ testIFidx, mm]) )
                uVal = (ResT**2).sum()
                vVal = (( np.append(DataPOut_dfUpT.Trait.values.reshape(-1, 1), Eval_dfT.Trait.values.reshape(-1, 1)) - \
                          np.append(DataPOut_dfUpT.Trait.values.reshape(-1, 1), Eval_dfT.Trait.values.reshape(-1, 1)).mean() )**2).sum()
                R2IndTemp.append( (1-(uVal/vVal)) )
                RMSEIndTemp.append( np.sqrt( uVal/ (DataPOut_dfUpT.shape[0]+Eval_dfT.shape[0]) ) )

                # R2 and RMSE - all independent data (source + target) >> validation (source ; stable) + samples not included in the training (target ; dynamic)
                ResT = np.append( (DataPOut_dfUpT.Trait.values - OutDUn[kk][ DataPOut_dfUpT.index.tolist(), mm]), \
                                  (Val_df.Trait.values - ValiUn[kk][ Val_df.index.tolist(), mm]) )
                uVal = (ResT**2).sum()
                vVal = (( np.append(DataPOut_dfUpT.Trait.values.reshape(-1, 1), Val_df.Trait.values.reshape(-1, 1)) - \
                          np.append(DataPOut_dfUpT.Trait.values.reshape(-1, 1), Val_df.Trait.values.reshape(-1, 1)).mean() )**2).sum()
                R2IndVTemp.append( (1-(uVal/vVal)) )
                RMSEIndVTemp.append( np.sqrt( uVal/ (DataPOut_dfUpT.shape[0]+Val_df.shape[0]) ) )

            # save outputs
            R2EvalT.append( R2EvalTemp )
            R2EEvalT.append( R2EEvalTemp )
            R2ValT.append( R2ValTemp )
            R2OutT.append( R2OutTemp )
            R2IndT.append( R2IndTemp )
            R2IndVT.append( R2IndVTemp )
            RMSEEvalT.append( RMSEEvalTemp )
            RMSEEEvalT.append( RMSEEEvalTemp )
            RMSEValT.append( RMSEValTemp )
            RMSEOutT.append( RMSEOutTemp )
            RMSEIndT.append( RMSEIndTemp )
            RMSEIndVT.append( RMSEIndVTemp )
            PLSRFactT.append( PLSRFactTemp )
            InterBooT.append( InterBooTemp )
            CIselVIPRefT.append( CIselVIPRefTemp )
            CIOutT.append(  CIOutTemp )
        R2EvalF.append( R2EvalT )
        R2EEvalF.append( R2EEvalT )
        R2ValF.append( R2ValT )
        R2OutF.append( R2OutT )
        R2IndF.append( R2IndT )
        R2IndVF.append( R2IndVT )
        RMSEEvalF.append( RMSEEvalT )
        RMSEEEvalF.append( RMSEEEvalT )
        RMSEValF.append( RMSEValT )
        RMSEOutF.append( RMSEOutT )
        RMSEIndF.append( RMSEIndT )
        RMSEIndVF.append( RMSEIndVT )
        PLSRFactF.append( PLSRFactT )
        InterBooF.append( InterBooT )
        CIselVIPRefF.append( CIselVIPRefT )
        VIPRetainedF.append( VIPRetainedT )
        CIOutF.append( CIOutT )
        TestUnF.append( TestUn )
        ETestUnF.append( ETestUn )
        EvalUnF.append( EvalUn )
        ValiUnF.append( ValiUn )
        OutDUnF.append( OutDUn )
        CoefUnF.append( CoefUn )
        VIPOUnF.append( VIPOUn )
    R2EvalF_varSel_all.append( R2EvalF )
    R2EEvalF_varSel_all.append( R2EEvalF )
    R2ValF_varSel_all.append( R2ValF )
    R2OutF_varSel_all.append( R2OutF )
    R2IndF_varSel_all.append( R2IndF )
    R2IndVF_varSel_all.append( R2IndVF )
    RMSEEvalF_varSel_all.append( RMSEEvalF )
    RMSEEEvalF_varSel_all.append( RMSEEEvalF )
    RMSEValF_varSel_all.append( RMSEValF )
    RMSEOutF_varSel_all.append( RMSEOutF )
    RMSEIndF_varSel_all.append( RMSEIndF )
    RMSEIndVF_varSel_all.append( RMSEIndVF )
    InterBooF_varSel_all.append( InterBooF )
    PLSRFactF_varSel_all.append( PLSRFactF )
    CIselVIPRefF_varSel_all.append( CIselVIPRefF )
    VIPRetainedF_varSel_all.append( VIPRetainedF )
    CIOutF_varSel_all.append( CIOutF )
    TestUnF_varSel_all.append( TestUnF )
    ETestUnF_varSel_all.append( ETestUnF )
    EvalUnF_varSel_all.append( EvalUnF )
    ValiUnF_varSel_all.append( ValiUnF )
    OutDUnF_varSel_all.append( OutDUnF )
    CoefUnF_varSel_all.append( CoefUnF )
    VIPUnF_varSel_all.append(  VIPOUnF )

### Save or load the outputs

Reads the saved outputs. Set `SAVE_OUTPUTS` to save a new run first. *(Original cell 32.)*

In [ ]:
os.chdir( r'<PROJECT_ROOT>\1_JypiterLabProcessing\5_Outputs' )

versionRef = '01'

if( SAVE_OUTPUTS ):
    with open('InterBooF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'wb') as f:
        pickle.dump(InterBooF_varSel_all, f)
    with open('PLSRFactF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'wb') as f:
        pickle.dump(PLSRFactF_varSel_all, f)
    with open('R2EvalF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'wb') as f:
        pickle.dump(R2EvalF_varSel_all, f)
    with open('RMSEEvalF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'wb') as f:
        pickle.dump(RMSEEvalF_varSel_all, f)
    with open('R2EEvalF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'wb') as f:
        pickle.dump(R2EEvalF_varSel_all, f)
    with open('RMSEEEvalF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'wb') as f:
        pickle.dump(RMSEEEvalF_varSel_all, f)
    with open('R2ValF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'wb') as f:
        pickle.dump(R2ValF_varSel_all, f)
    with open('RMSEValF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'wb') as f:
        pickle.dump(RMSEValF_varSel_all, f)
    with open('R2OutF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'wb') as f:
        pickle.dump(R2OutF_varSel_all, f)
    with open('RMSEOutF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'wb') as f:
        pickle.dump(RMSEOutF_varSel_all, f)
    with open('R2IndF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'wb') as f:
        pickle.dump(R2IndF_varSel_all, f)
    with open('RMSEIndF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'wb') as f:
        pickle.dump(RMSEIndF_varSel_all, f)
    with open('R2IndVF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'wb') as f:
        pickle.dump(R2IndVF_varSel_all, f)
    with open('RMSEIndVF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'wb') as f:
        pickle.dump(RMSEIndVF_varSel_all, f)
if( SAVE_OUTPUTS ):
    with open('CIselVIPRefF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'wb') as f:
        pickle.dump(CIselVIPRefF_varSel_all, f)
    with open('VIPRetainedF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'wb') as f:
        pickle.dump(VIPRetainedF_varSel_all, f)
    with open('CIOutF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'wb') as f:
        pickle.dump(CIOutF_varSel_all, f)
if( SAVE_OUTPUTS ):
    with open('TestUnF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'wb') as f:
        pickle.dump(TestUnF_varSel_all, f)
    with open('EvalUnF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'wb') as f:
        pickle.dump(EvalUnF_varSel_all, f)
    with open('ETestUnF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'wb') as f:
        pickle.dump(ETestUnF_varSel_all, f)
    with open('ValiUnF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'wb') as f:
        pickle.dump(ValiUnF_varSel_all, f)
    with open('OutDUnF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'wb') as f:
        pickle.dump(OutDUnF_varSel_all, f)
    with open('CoefUnF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'wb') as f:
        pickle.dump(CoefUnF_varSel_all, f)
    with open('VIPUnF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'wb') as f:
        pickle.dump(VIPUnF_varSel_all, f)
# load
with open('InterBooF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'rb') as f:
    InterBooF_varSel_allO = pickle.load(f)

with open('PLSRFactF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'rb') as f:
    PLSRFactF_varSel_allO = pickle.load(f)

with open('R2EvalF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'rb') as f:
    R2EvalF_varSel_allO = pickle.load(f)

with open('RMSEEvalF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'rb') as f:
    RMSEEvalF_varSel_allO = pickle.load(f)

with open('R2EEvalF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'rb') as f:
    R2EEvalF_varSel_allO = pickle.load(f)

with open('RMSEEEvalF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'rb') as f:
    RMSEEEvalF_varSel_allO = pickle.load(f)

with open('R2ValF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'rb') as f:
    R2ValF_varSel_allO = pickle.load(f)

with open('RMSEValF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'rb') as f:
    RMSEValF_varSel_allO = pickle.load(f)

with open('R2OutF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'rb') as f:
    R2OutF_varSel_allO = pickle.load(f)

with open('RMSEOutF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'rb') as f:
    RMSEOutF_varSel_allO = pickle.load(f)

with open('R2IndF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'rb') as f:
    R2IndF_varSel_allO = pickle.load(f)

with open('RMSEIndF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'rb') as f:
    RMSEIndF_varSel_allO = pickle.load(f)

with open('R2IndVF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'rb') as f:
    R2IndVF_varSel_allO = pickle.load(f)

with open('RMSEIndVF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'rb') as f:
    RMSEIndVF_varSel_allO = pickle.load(f)

with open('CIselVIPRefF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'rb') as f:
    CIselVIPRefF_varSel_allO = pickle.load(f)

with open('VIPRetainedF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'rb') as f:
    VIPRetainedF_varSel_allO = pickle.load(f)

with open('CIOutF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'rb') as f:
    CIOutF_varSel_allO = pickle.load(f)

with open('TestUnF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'rb') as f:
    TestUnF_varSel_allO = pickle.load(f)

with open('EvalUnF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'rb') as f:
    EvalUnF_varSel_allO = pickle.load(f)

with open('ETestUnF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'rb') as f:
    ETestUnF_varSel_allO = pickle.load(f)

with open('ValiUnF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'rb') as f:
    ValiUnF_varSel_allO = pickle.load(f)

with open('OutDUnF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'rb') as f:
    OutDUnF_varSel_allO = pickle.load(f)

with open('CoefUnF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'rb') as f:
    CoefUnF_varSel_allO = pickle.load(f)

with open('VIPUnF_varSel_allO_BootPLS_spike_VIP_opt_v' + versionRef + '_' + str(VIP_refMod) + '.pkl', 'rb') as f:
    VIPUnF_varSel_allO = pickle.load(f)

### Summary

Calculates R2, RMSE and their spread after feature selection. *(Original cell 33.)*

In [ ]:
DateRefY = [ '2014', '2017' ]
HarvRef  = [ '20140515', '20141015', '20170510', '20170830', '20171026' ]
col_ref  = list( range(len(center_origin)) )

modeRef = 'mean'

TraitRef   = [ 'DM_dt_ha', 'N_perc_DM', 'N_kg_ha' ]

PredVarSelSumm_mean_all = []
PredVarSelSumm_std_all  = []
for ii in range( len(R2EvalF_varSel_allO) ): # source dataset | year
    if( ii == 0 ):
        DateOutRefT =  HarvRef[2:6]
    else:
        DateOutRefT =  HarvRef[0:2]

    PredVarSelSumm_mean_TT = []
    PredVarSelSumm_std_TT  = []
    for jj in range( len(R2EvalF_varSel_allO[ii]) ):  # trait
        if( jj == 0 ): # Dry Biomass data / 49 observations per date
            DataP    = SpecAll_sel_0_1_res_Avg.loc[ ((SpecAll_sel_0_1_res_Avg.Year).astype(int) == int(DateRefY[ii])), :]  # source data
            GroupF   = [ str(DataP.Replicate.tolist()[val]) + '_' + str(DataP.Treatment.tolist()[val]) for val in range(DataP.shape[0]) ]  # reference to each plot...
            DataPOut = SpecAll_sel_0_1_res_Avg.loc[~((SpecAll_sel_0_1_res_Avg.Year).astype(int) == int(DateRefY[ii])), :]  # data outside the source domain (different season)
            DataP_PC = copy.deepcopy( PCAmOM_scoresAll )
        else:  # N data / 15 observations per date
            DataP    = SpecAll_sel_0_1_res_Avg_treat.loc[ ((SpecAll_sel_0_1_res_Avg_treat.Year).astype(int) == int(DateRefY[ii])), :]
            GroupF   = DataP.Treatment.tolist()
            DataPOut = SpecAll_sel_0_1_res_Avg_treat.loc[~((SpecAll_sel_0_1_res_Avg_treat.Year).astype(int) == int(DateRefY[ii])), :]
            DataP_PC = copy.deepcopy( PCAmOM_scoresAllRed )
        DataM_df    = DataP.loc[ :, (DataP.columns == TraitRef[jj]) | DataP.columns.isin(col_ref) ]  # select trait of interest + spectra
        DataPOut_df = DataPOut.loc[ :, (DataPOut.columns == TraitRef[jj]) | DataPOut.columns.isin(col_ref) ]  # data outside the source domain (different season)
        BRef = [ 'b' + str(range(len(col_ref))[val]) if range(len(col_ref))[val] > 9 else 'b0' + str(range(len(col_ref))[val]) for val in range(len(col_ref)) ]
        DataM_df.columns = ['Trait'] + BRef
        DataM_df = DataM_df.reset_index(drop=True)  # not needed since reference made by index, but doing anyway
        DataPOut_df.columns = ['Trait'] + BRef
        DataPOut_df = DataPOut_df.reset_index(drop=True)  # not needed since reference made by index, but doing anyway

        PredVarSelSumm_mean_T = []
        PredVarSelSumm_std_T  = []
        for kk in range( len(R2EvalF_varSel_allO[ii][jj]) ):  # date in the outside dataset - target

            R2EvalF_summO  = [ np.nanmedian( R2EvalF_varSel_allO[ii][jj][kk] ).tolist()  if modeRef == 'med' else np.nanmean( R2EvalF_varSel_allO[ii][jj][kk] ).tolist() ]   # calib set (source) - out-of-bag samples - Eval
            R2EEvalF_summO = [ np.nanmedian( R2EEvalF_varSel_allO[ii][jj][kk] ).tolist() if modeRef == 'med' else np.nanmean( R2EEvalF_varSel_allO[ii][jj][kk] ).tolist() ]  # external dataset - added to the training data - EEval
            R2ValF_summO   = [ np.nanmedian( R2ValF_varSel_allO[ii][jj][kk] ).tolist()   if modeRef == 'med' else np.nanmean( R2ValF_varSel_allO[ii][jj][kk] ).tolist() ]    # validation set (source)
            R2OutF_summO   = [ np.nanmedian( R2OutF_varSel_allO[ii][jj][kk] ).tolist()   if modeRef == 'med' else np.nanmean( R2OutF_varSel_allO[ii][jj][kk] ).tolist() ]    # outside dataset - not included in the training data - Out
            R2IndF_summO   = [ np.nanmedian( R2IndF_varSel_allO[ii][jj][kk] ).tolist()   if modeRef == 'med' else np.nanmean( R2IndF_varSel_allO[ii][jj][kk] ).tolist() ]    # all independent data (source + target) - careful source samples are out-of-bag not validation - Ind
            R2IndVF_summO  = [ np.nanmedian( R2IndVF_varSel_allO[ii][jj][kk] ).tolist()  if modeRef == 'med' else np.nanmean( R2IndVF_varSel_allO[ii][jj][kk] ).tolist() ]   # all independent data (source + target) - validation (source ; stable) + samples not included in the training (target ; dynamic) - IndV

            R2EvalF_summO  = R2EvalF_summO  + [ scipy.stats.iqr( R2EvalF_varSel_allO[ii][jj][kk], nan_policy='omit' )  if modeRef == 'med' else np.nanstd( R2EvalF_varSel_allO[ii][jj][kk] ) ]
            R2EEvalF_summO = R2EEvalF_summO + [ scipy.stats.iqr( R2EEvalF_varSel_allO[ii][jj][kk], nan_policy='omit' ) if modeRef == 'med' else np.nanstd( R2EEvalF_varSel_allO[ii][jj][kk] ) ]
            R2ValF_summO   = R2ValF_summO   + [ scipy.stats.iqr( R2ValF_varSel_allO[ii][jj][kk], nan_policy='omit' )   if modeRef == 'med' else np.nanstd( R2ValF_varSel_allO[ii][jj][kk] ) ]
            R2OutF_summO   = R2OutF_summO   + [ scipy.stats.iqr( R2OutF_varSel_allO[ii][jj][kk], nan_policy='omit' )   if modeRef == 'med' else np.nanstd( R2OutF_varSel_allO[ii][jj][kk] ) ]
            R2IndF_summO   = R2IndF_summO   + [ scipy.stats.iqr( R2IndF_varSel_allO[ii][jj][kk], nan_policy='omit' )   if modeRef == 'med' else np.nanstd( R2IndF_varSel_allO[ii][jj][kk] ) ]
            R2IndVF_summO  = R2IndVF_summO  + [ scipy.stats.iqr( R2IndVF_varSel_allO[ii][jj][kk], nan_policy='omit' )  if modeRef == 'med' else np.nanstd( R2IndVF_varSel_allO[ii][jj][kk] ) ]

            RMSEEvalF_summO  = [ np.nanmedian( RMSEEvalF_varSel_allO[ii][jj][kk] ).tolist()  if modeRef == 'med' else np.nanmean( RMSEEvalF_varSel_allO[ii][jj][kk] ).tolist() ]
            RMSEEEvalF_summO = [ np.nanmedian( RMSEEEvalF_varSel_allO[ii][jj][kk] ).tolist() if modeRef == 'med' else np.nanmean( RMSEEEvalF_varSel_allO[ii][jj][kk] ).tolist() ]
            RMSEValF_summO   = [ np.nanmedian( RMSEValF_varSel_allO[ii][jj][kk] ).tolist()   if modeRef == 'med' else np.nanmean( RMSEValF_varSel_allO[ii][jj][kk] ).tolist() ]
            RMSEOutF_summO   = [ np.nanmedian( RMSEOutF_varSel_allO[ii][jj][kk] ).tolist()   if modeRef == 'med' else np.nanmean( RMSEOutF_varSel_allO[ii][jj][kk] ).tolist() ]
            RMSEIndF_summO   = [ np.nanmedian( RMSEIndF_varSel_allO[ii][jj][kk] ).tolist()   if modeRef == 'med' else np.nanmean( RMSEIndF_varSel_allO[ii][jj][kk] ).tolist() ]
            RMSEIndVF_summO  = [ np.nanmedian( RMSEIndVF_varSel_allO[ii][jj][kk] ).tolist()  if modeRef == 'med' else np.nanmean( RMSEIndVF_varSel_allO[ii][jj][kk] ).tolist() ]

            RMSEEvalF_summO  = RMSEEvalF_summO  + [ scipy.stats.iqr( RMSEEvalF_varSel_allO[ii][jj][kk], nan_policy='omit' )  if modeRef == 'med' else np.nanstd( RMSEEvalF_varSel_allO[ii][jj][kk] ) ]
            RMSEEEvalF_summO = RMSEEEvalF_summO + [ scipy.stats.iqr( RMSEEEvalF_varSel_allO[ii][jj][kk], nan_policy='omit' ) if modeRef == 'med' else np.nanstd( RMSEEEvalF_varSel_allO[ii][jj][kk] ) ]
            RMSEValF_summO   = RMSEValF_summO   + [ scipy.stats.iqr( RMSEValF_varSel_allO[ii][jj][kk], nan_policy='omit' )   if modeRef == 'med' else np.nanstd( RMSEValF_varSel_allO[ii][jj][kk] ) ]
            RMSEOutF_summO   = RMSEOutF_summO   + [ scipy.stats.iqr( RMSEOutF_varSel_allO[ii][jj][kk], nan_policy='omit' )   if modeRef == 'med' else np.nanstd( RMSEOutF_varSel_allO[ii][jj][kk] ) ]
            RMSEIndF_summO   = RMSEIndF_summO   + [ scipy.stats.iqr( RMSEIndF_varSel_allO[ii][jj][kk], nan_policy='omit' )   if modeRef == 'med' else np.nanstd( RMSEIndF_varSel_allO[ii][jj][kk] ) ]
            RMSEIndVF_summO  = RMSEIndVF_summO  + [ scipy.stats.iqr( RMSEIndVF_varSel_allO[ii][jj][kk], nan_policy='omit' )  if modeRef == 'med' else np.nanstd( RMSEIndVF_varSel_allO[ii][jj][kk] ) ]

            PredEval_summO   = [ np.nanmedian( TestUnF_varSel_allO[ii][jj][kk], axis=1 ).tolist()  if modeRef == 'med' else np.nanmean( TestUnF_varSel_allO[ii][jj][kk], axis=1 ).tolist()  ]  # calib set (source) - out-of-bag samples - Eval
            PredEEval_summO  = [ np.nanmedian( EvalUnF_varSel_allO[ii][jj][kk], axis=1 ).tolist()  if modeRef == 'med' else np.nanmean( EvalUnF_varSel_allO[ii][jj][kk], axis=1 ).tolist()  ]  # test samples - source dataset - boot repetition
            PredETest_summO  = [ np.nanmedian( ETestUnF_varSel_allO[ii][jj][kk], axis=1 ).tolist() if modeRef == 'med' else np.nanmean( ETestUnF_varSel_allO[ii][jj][kk], axis=1 ).tolist() ]  # external dataset - added to the training data - EEval
            PredVali_summO   = [ np.nanmedian( ValiUnF_varSel_allO[ii][jj][kk], axis=1 ).tolist()  if modeRef == 'med' else np.nanmean( ValiUnF_varSel_allO[ii][jj][kk], axis=1 ).tolist()  ]  # validation set (source)
            PredOut_summO    = [ np.nanmedian( OutDUnF_varSel_allO[ii][jj][kk], axis=1 ).tolist()  if modeRef == 'med' else np.nanmean( OutDUnF_varSel_allO[ii][jj][kk], axis=1 ).tolist()  ]  # outside dataset - not included in the training data - Out
            PredCoef_summO   = [ np.nanmedian( CoefUnF_varSel_allO[ii][jj][kk], axis=1 ).tolist()  if modeRef == 'med' else np.nanmean( CoefUnF_varSel_allO[ii][jj][kk], axis=1 ).tolist()  ]
            PredVIP_summO    = [ np.nanmedian( VIPUnF_varSel_allO[ii][jj][kk], axis=1 ).tolist()   if modeRef == 'med' else np.nanmean( VIPUnF_varSel_allO[ii][jj][kk], axis=1 ).tolist()   ]

            PredEval_summO  = PredEval_summO  + [ scipy.stats.iqr( TestUnF_varSel_allO[ii][jj][kk], axis=1, nan_policy='omit' )  if modeRef == 'med' else np.nanstd( TestUnF_varSel_allO[ii][jj][kk], axis=1 )  ]
            PredEEval_summO = PredEEval_summO + [ scipy.stats.iqr( EvalUnF_varSel_allO[ii][jj][kk], axis=1, nan_policy='omit' )  if modeRef == 'med' else np.nanstd( EvalUnF_varSel_allO[ii][jj][kk], axis=1 )  ]
            PredETest_summO = PredETest_summO + [ scipy.stats.iqr( ETestUnF_varSel_allO[ii][jj][kk], axis=1, nan_policy='omit' ) if modeRef == 'med' else np.nanstd( ETestUnF_varSel_allO[ii][jj][kk], axis=1 ) ]
            PredVali_summO  = PredVali_summO  + [ scipy.stats.iqr( ValiUnF_varSel_allO[ii][jj][kk], axis=1, nan_policy='omit' )  if modeRef == 'med' else np.nanstd( ValiUnF_varSel_allO[ii][jj][kk], axis=1 )  ]
            PredOut_summO   = PredOut_summO   + [ scipy.stats.iqr( OutDUnF_varSel_allO[ii][jj][kk], axis=1, nan_policy='omit' )  if modeRef == 'med' else np.nanstd( OutDUnF_varSel_allO[ii][jj][kk], axis=1 )  ]
            PredCoef_summO  = PredCoef_summO  + [ scipy.stats.iqr( CoefUnF_varSel_allO[ii][jj][kk], axis=1, nan_policy='omit' )  if modeRef == 'med' else np.nanstd( CoefUnF_varSel_allO[ii][jj][kk], axis=1 )  ]
            PredVIP_summO   = PredVIP_summO   + [ scipy.stats.iqr( VIPUnF_varSel_allO[ii][jj][kk], axis=1, nan_policy='omit' )   if modeRef == 'med' else np.nanstd( VIPUnF_varSel_allO[ii][jj][kk], axis=1 )  ]

            # get IQR for each outside dataset - incremental
            if( kk == 0 ):
                DataPOutSelT = DataPOut_df.loc[ DataPOut.Date.isin( [ np.array(DateOutRefT[kk]).astype('str').tolist() ] ).values ]  # selected data in agiven date
                DataMAll_dfT = DataM_df.append( DataPOutSelT, ignore_index=True )
            else:
                DataPOutSelT = DataPOutSelT.append( DataPOut_df.loc[ DataPOut.Date.isin( [ np.array(DateOutRefT[kk]).astype('str').tolist() ] ).values ], ignore_index=True )  # selected data in agiven date
                DataMAll_dfT = DataM_df.append( DataPOutSelT, ignore_index=True )
            IQR_summO = scipy.stats.iqr( DataMAll_dfT.Trait, nan_policy='omit' )  # last IQR corresponds to the joint complete domains target and source

            # organize outputs
            if( all([ (ii == 0), (jj == 0), (kk == 0) ]) ):
                R2VarSelSumm_mean_all   = pd.DataFrame( np.expand_dims( np.array([R2EvalF_summO[0], R2EEvalF_summO[0], R2ValF_summO[0], R2OutF_summO[0], R2IndF_summO[0], R2IndVF_summO[0]]), axis=0 ), \
                                                        columns=['Eval', 'EEval', 'Val', 'Out', 'Ind', 'IndV'] )
                R2VarSelSumm_mean_all[ 'Year' ]    = DateRefY[ii]
                R2VarSelSumm_mean_all[ 'Trait' ]   = TraitRef[jj]
                R2VarSelSumm_mean_all[ 'DateOut' ] = DateOutRefT[kk]
                R2VarSelSumm_std_all    = pd.DataFrame( np.expand_dims( np.array([R2EvalF_summO[1], R2EEvalF_summO[1], R2ValF_summO[1], R2OutF_summO[1], R2IndF_summO[1], R2IndVF_summO[1]]), axis=0 ), \
                                                        columns=['Eval', 'EEval', 'Val', 'Out', 'Ind', 'IndV'] )
                R2VarSelSumm_std_all[ 'Year' ]    = DateRefY[ii]
                R2VarSelSumm_std_all[ 'Trait' ]   = TraitRef[jj]
                R2VarSelSumm_std_all[ 'DateOut' ] = DateOutRefT[kk]

                RMSEVarSelSumm_mean_all = pd.DataFrame( np.expand_dims( np.array([RMSEEvalF_summO[0], RMSEEEvalF_summO[0], RMSEValF_summO[0], RMSEOutF_summO[0], RMSEIndF_summO[0], RMSEIndVF_summO[0]]), axis=0 ), \
                                                        columns=['Eval', 'EEval', 'Val', 'Out', 'Ind', 'IndV'] )
                RMSEVarSelSumm_mean_all[ 'Year' ]    = DateRefY[ii]
                RMSEVarSelSumm_mean_all[ 'Trait' ]   = TraitRef[jj]
                RMSEVarSelSumm_mean_all[ 'DateOut' ] = DateOutRefT[kk]
                RMSEVarSelSumm_std_all  = pd.DataFrame( np.expand_dims( np.array([RMSEEvalF_summO[1], RMSEEEvalF_summO[1], RMSEValF_summO[1], RMSEOutF_summO[1], RMSEIndF_summO[1], RMSEIndVF_summO[1]]), axis=0 ), \
                                                        columns=['Eval', 'EEval', 'Val', 'Out', 'Ind', 'IndV'] )
                RMSEVarSelSumm_std_all[ 'Year' ]    = DateRefY[ii]
                RMSEVarSelSumm_std_all[ 'Trait' ]   = TraitRef[jj]
                RMSEVarSelSumm_std_all[ 'DateOut' ] = DateOutRefT[kk]
            else:
                R2VarSelSumm_mean_allT = pd.DataFrame( np.expand_dims( np.array([R2EvalF_summO[0], R2EEvalF_summO[0], R2ValF_summO[0], R2OutF_summO[0], R2IndF_summO[0], R2IndVF_summO[0]]), axis=0 ), columns=['Eval', 'EEval', 'Val', 'Out', 'Ind', 'IndV'])
                R2VarSelSumm_mean_allT[ 'Year' ]    = DateRefY[ii]
                R2VarSelSumm_mean_allT[ 'Trait' ]   = TraitRef[jj]
                R2VarSelSumm_mean_allT[ 'DateOut' ] = DateOutRefT[kk]
                R2VarSelSumm_mean_all  = R2VarSelSumm_mean_all.append( R2VarSelSumm_mean_allT, ignore_index=True)
                R2VarSelSumm_std_allT = pd.DataFrame( np.expand_dims( np.array([R2EvalF_summO[1], R2EEvalF_summO[1], R2ValF_summO[1], R2OutF_summO[1], R2IndF_summO[1], R2IndVF_summO[1]]), axis=0 ), columns=['Eval', 'EEval', 'Val', 'Out', 'Ind', 'IndV'])
                R2VarSelSumm_std_allT[ 'Year' ]    = DateRefY[ii]
                R2VarSelSumm_std_allT[ 'Trait' ]   = TraitRef[jj]
                R2VarSelSumm_std_allT[ 'DateOut' ] = DateOutRefT[kk]
                R2VarSelSumm_std_all  = R2VarSelSumm_std_all.append( R2VarSelSumm_std_allT, ignore_index=True)
                RMSEVarSelSumm_mean_allT = pd.DataFrame( np.expand_dims( np.array([RMSEEvalF_summO[0], RMSEEEvalF_summO[0], RMSEValF_summO[0], RMSEOutF_summO[0], RMSEIndF_summO[0], RMSEIndVF_summO[0]]), axis=0 ), \
                                                         columns=['Eval', 'EEval', 'Val', 'Out', 'Ind', 'IndV'] )
                RMSEVarSelSumm_mean_allT[ 'Year' ]    = DateRefY[ii]
                RMSEVarSelSumm_mean_allT[ 'Trait' ]   = TraitRef[jj]
                RMSEVarSelSumm_mean_allT[ 'DateOut' ] = DateOutRefT[kk]
                RMSEVarSelSumm_mean_all = RMSEVarSelSumm_mean_all.append(RMSEVarSelSumm_mean_allT, ignore_index=True)
                RMSEVarSelSumm_std_allT  = pd.DataFrame( np.expand_dims( np.array([RMSEEvalF_summO[1], RMSEEEvalF_summO[1], RMSEValF_summO[1], RMSEOutF_summO[1], RMSEIndF_summO[1], RMSEIndVF_summO[1]]), axis=0 ), \
                                                         columns=['Eval', 'EEval', 'Val', 'Out', 'Ind', 'IndV'] )
                RMSEVarSelSumm_std_allT[ 'Year' ]    = DateRefY[ii]
                RMSEVarSelSumm_std_allT[ 'Trait' ]   = TraitRef[jj]
                RMSEVarSelSumm_std_allT[ 'DateOut' ] = DateOutRefT[kk]
                RMSEVarSelSumm_std_all  = RMSEVarSelSumm_std_all.append( RMSEVarSelSumm_std_allT, ignore_index=True)
            PredVarSelSumm_mean_T.append( [PredEval_summO[0], PredEEval_summO[0], PredETest_summO[0], PredVali_summO[0], PredOut_summO[0], PredCoef_summO[0], PredVIP_summO[0]] )
            PredVarSelSumm_std_T.append(  [PredEval_summO[1], PredEEval_summO[1], PredETest_summO[1], PredVali_summO[1], PredOut_summO[1], PredCoef_summO[1], PredVIP_summO[1]] )
        PredVarSelSumm_mean_TT.append( PredVarSelSumm_mean_T )
        PredVarSelSumm_std_TT.append(  PredVarSelSumm_std_T )
    PredVarSelSumm_mean_all.append( PredVarSelSumm_mean_TT )
    PredVarSelSumm_std_all.append(  PredVarSelSumm_std_TT )

### Figure 8

Sample-wise predictions of the transferred models, using the selected bands. *(Original cell 34.)*

In [ ]:
os.chdir( r'<PROJECT_ROOT>\DataProcessing\1_Script_source' )
import FakeParulaPalette as FakeParulaF
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.graphics.regressionplots import abline_plot
from statsmodels.sandbox.regression.predstd import wls_prediction_std
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator, AutoLocator, MaxNLocator)
minorLocator = AutoMinorLocator()
majorLocator = MaxNLocator(4)

TraitRefL  = [ r'Dry matter $[\/dt\/\/ha^{-1}\/]$',
               r'N content $[\/\%\/in\/DM\/]$',
               r'N total $[\/kg\/\/ha^{-1}\/]$'  ]

fontS0 = 9
fontS1 = 10
fontS2 = 11
fontS3 = 12
FontSizeG = 10
FontSizeT = 11

matplotlib.rcParams['axes.linewidth']  = 0.8
matplotlib.rcParams['font.family']     = 'Arial'
matplotlib.rcParams['legend.fontsize'] = 8
matplotlib.rcParams['legend.handlelength']   = 0.8

MajorTicksLength = 3.5
MinorTicksLength = 2.2
errorLineWidth   = 1.2

lwidth =  0.6

TraitRef    = [ 'DM_dt_ha', 'N_perc_DM', 'N_kg_ha' ]
TraitRefAll = [ 'DM_dt_ha', 'N_perc_DM', 'CrudeProt_perc_DM', 'N_kg_ha', 'CrudeProt_kg_ha' ]

YLimAllObs = [ [[0, 60], [0.90, 3.90], [5, 105]],
               [[0, 60], [0.90, 3.90], [5, 105]],
               [[0, 60], [0.90, 3.90], [5, 105]] ]
YLimAllObs_new = [ [[0, 60], [0.90, 3.90], [-15, 105]],
                   [[0, 60], [0.90, 3.90], [-15, 105]],
                   [[0, 60], [0.90, 3.90], [-15, 105]] ]

axisIntDistT   = [ [20, 0.75, 25], [20, 0.75, 25], [20, 0.75, 25] ]
axisIntDisLenT = [ [7, 5, 5], [7, 5, 5], [7, 5, 5] ]
axisIntRoundFT = [ ["%.1f", "%.2f", "%.1f"], ["%.1f", "%.2f", "%.1f"], ["%.1f", "%.2f", "%.1f"] ]

YearRef = SpecAll_sel_0_1_res_Avg.Year.unique().tolist()
DateRef = SpecAll_sel_0_1_res_Avg.Date.unique().tolist()
DateRefY = [ int(str(DateRef[val])[:4]) for val in range(len(DateRef)) ]

DateDOY = []
for elmnt in range(len(DateRef)):  # date
    dateT = pd.to_datetime(DateRef[elmnt], format='%Y%m%d')
    dateTRef = pd.Timestamp(year=dateT.year, month=1, day=1)
    DateDOY.append( (dateT - dateTRef).days + 1 )

cmap   = matplotlib.cm.get_cmap('BrBG')
colorR = np.arange(0, 1+0.2, 0.2)
colorR = np.arange(0, 0.3+0.15, 0.15).tolist() + colorR[4:].tolist()

Parula_manual   = [ plt.matplotlib.cm.get_cmap('Greens')(0.70), plt.matplotlib.cm.get_cmap('Greens')(0.50), \
                   'blue', 'dodgerblue', 'lightskyblue']
colorR = list( range( len(Parula_manual) ) )

validTargetRefT = 'xkcd:golden yellow'

modeRef = 'mean'

saveOpt = True

for ii in range( len(TraitRef) ):  # traits
 for jj in range( len(PredVarSelSumm_mean_all) ):  # dataset source
    if( jj == 0 ) :
        DateOutRefT =  HarvRef[2:5]
    else:
        DateOutRefT =  HarvRef[0:2]
    if ii == 0:
        DataP    = SpecAll_sel_0_1_res_Avg.loc[ (SpecAll_sel_0_1_res_Avg.Year).astype(int) == int(YearRef[jj]), :]
        DataPOut = SpecAll_sel_0_1_res_Avg.loc[ ~((SpecAll_sel_0_1_res_Avg.Year).astype(int) == int(YearRef[jj])), :]
        DataDim = DataP.shape[0]/len(PredObs)
    else:
        DataP    = SpecAll_sel_0_1_res_Avg_treat.loc[ (SpecAll_sel_0_1_res_Avg_treat.Year).astype(int) == int(YearRef[jj]), :]
        DataPOut = SpecAll_sel_0_1_res_Avg_treat.loc[~((SpecAll_sel_0_1_res_Avg_treat.Year).astype(int) == int(YearRef[jj])), :]
        DataDim = DataP.shape[0]/len(PredObs)

    if( jj == 0 ):
        DateDOYSource = DateDOY[0:2]
        DateDOYTarget = DateDOY[2:5]
        colorRSource  = colorR[0:2]
    else:
        DateDOYSource = DateDOY[2:5]
        DateDOYTarget = DateDOY[0:2]
        colorRSource  = colorR[2:5]

    DataM_df    = DataP.loc[ :, (DataP.columns == TraitRef[ii]) | DataP.columns.isin(col_ref) ]
    DataPOut_df = DataPOut.loc[ :, (DataPOut.columns == TraitRef[ii]) | DataPOut.columns.isin(col_ref) ]
    BRef = [ 'b' + str(range(len(col_ref))[val]) if range(len(col_ref))[val] > 9 else 'b0' + str(range(len(col_ref))[val]) for val in range(len(col_ref)) ]
    DataM_df.columns = ['Trait'] + BRef
    DataM_df = DataM_df.reset_index(drop=True)  # not needed since reference made by index, but doing anyway...
    DataPOut_df.columns = ['Trait'] + BRef
    DataPOut_df = DataPOut_df.reset_index(drop=True)  # not needed since reference made by index, but doing anyway...

    for kk in range( len(PredVarSelSumm_mean_all[jj][ii]) ):   # outside dataset | acq date

        TestOT  = PredVarSelSumm_mean_all[jj][ii][kk][0]  # source, source, target, source, target, [...]  ;  calibrarion set - source
        EEvalOT = PredVarSelSumm_mean_all[jj][ii][kk][1]  # test set - source (bootstrap within bag)
        ETestOT = PredVarSelSumm_mean_all[jj][ii][kk][2]  # calibration/test set - target (samples added to the training data)
        ValiOT  = PredVarSelSumm_mean_all[jj][ii][kk][3]  # validation set  - source
        OutOT   = PredVarSelSumm_mean_all[jj][ii][kk][4]  # validation set  - target

        TestOT_std  = PredVarSelSumm_std_all[jj][ii][kk][0]
        EEvalOT_std = PredVarSelSumm_std_all[jj][ii][kk][1]
        ETestOT_std = PredVarSelSumm_std_all[jj][ii][kk][2]
        ValiOT_std  = PredVarSelSumm_std_all[jj][ii][kk][3]
        OutOT_std   = PredVarSelSumm_std_all[jj][ii][kk][4]

        # inverse selection (masks) - based on NaN values
        sampTestT  = np.isnan( TestOT ).tolist()                                   # validation samples
        sampValiT  = np.isnan( ValiOT ).tolist()                                   # calibration samples
        sampETestT = np.logical_not(np.isnan( OutOT )).tolist()
        OutOTsub   = OutOT[ int(((kk+1)*DataDim)-DataDim):int(((kk+1)*DataDim)) ]
        sampOutT   = np.isnan( OutOTsub ).tolist()                                 # spike (used for calibration)

        # remove sample with large uncertainty - only one for validation set
        if( all([ (ii==1), (jj==1) ]) ):
            sampTestT[0] = False

        fig, ax = plt.subplots(figsize=(2.3, 2.3) )
        fig.subplots_adjust(left=0.23, right=0.94 , bottom = 0.22, top = 0.90)

        DateRT = DataP.Date.unique().tolist()
        DataPlotColoT = []
        for val in range( len(DateRT) ):
            # calib
            TestOTS     = np.array(TestOT)[ ( (DataP.Date == DateRT[val]) & sampValiT ) ]
            TestOTS_std = np.array(TestOT_std)[ ( (DataP.Date == DateRT[val]) & sampValiT ) ]
            TraitTTS    = DataM_df.loc[ ( DataP.reset_index(drop=True).Date == DateRT[val] ) & sampValiT, : ].Trait.tolist()

            # valid
            ValiOTS     = np.array(ValiOT)[((DataP.Date == DateRT[val]) & sampTestT)]
            ValiOTS_std = np.array(ValiOT_std)[((DataP.Date == DateRT[val]) & sampTestT)]
            TraitTVS    = DataM_df.loc[ (DataP.reset_index(drop=True).Date == DateRT[val]) & sampTestT, : ].Trait.tolist()

            # Compile complete dataset to be plot
            DataToPT      = np.append( np.array( [TestOTS, TestOTS_std, TraitTTS] ).T, np.array( [ValiOTS, ValiOTS_std, TraitTVS] ).T, axis=0 )
            DataPlotColoT = DataPlotColoT + [Parula_manual[ colorRSource[val] ]]*len(TestOTS) + ['red']*len(ValiOTS)  # colors to be used
            if( val == 0 ):
                DataPlotCompT = copy.deepcopy(DataToPT)
            else:
                DataPlotCompT = np.append( DataPlotCompT, DataToPT, axis=0 )

        # valid - target dataset
        DateRT = DataPOut.Date.unique().tolist()
        for val in range(len(DateRT)):  # date - outside | only independent data
            ETestOTS     = np.array(OutOT)[ ( (DataPOut.Date == DateRT[val]) & sampETestT ) ]
            ETestOTS_std = np.array(OutOT_std)[ ( (DataPOut.Date == DateRT[val]) & sampETestT ) ]
            ETraitTTS    = DataPOut_df.loc[ ( DataPOut.reset_index(drop=True).Date == DateRT[val] ) & sampETestT, : ].Trait.tolist()
            DataPlotColoT = DataPlotColoT + [validTargetRefT]*len(ETestOTS)
            DataPlotCompT = np.append( DataPlotCompT, np.array( [ETestOTS, ETestOTS_std, ETraitTTS] ).T, axis=0 )

        # plot points randonly for better visualization
        IdxToPT = list( range( DataPlotCompT.shape[0] ) )
        random.seed( [132563, 23015, 76][ii] )
        IdxToPT = random.sample(IdxToPT, k=len(IdxToPT))
        for val in range( len(IdxToPT) ):
            ax.errorbar( DataPlotCompT[IdxToPT[val],0], DataPlotCompT[IdxToPT[val],2], xerr=DataPlotCompT[IdxToPT[val],1], fmt='none', color=DataPlotColoT[IdxToPT[val]],
                         mfc='None', markersize=4, linewidth =errorLineWidth, capsize=1.40, elinewidth=errorLineWidth, markeredgewidth=errorLineWidth,
                         label= 'All data', zorder=3 )

        ax.set_ylim( YLimAllObs_new[jj][ii][0],  YLimAllObs_new[jj][ii][-1] )
        ax.set_xlim( YLimAllObs_new[jj][ii][0],  YLimAllObs_new[jj][ii][-1] )

        XPlot = [ [ np.min( list(ax.get_xlim()) + YLimAllObs[jj][ii] ), np.max( list(ax.get_xlim()) + YLimAllObs[jj][ii] )], \
                  [ np.min( list(ax.get_xlim()) + YLimAllObs[jj][ii] ), np.max( list(ax.get_xlim()) + YLimAllObs[jj][ii] )] ]
        ax.plot( XPlot[0], XPlot[1], 'k-',  linewidth=0.8 )  # 1:1 line
        ax.set_xlim( ax.get_xlim() )

        # Plot line passing over independent data predicted with the model (valid + outside)
        DataTL = pd.DataFrame( np.array([ np.append(DataP.loc[sampTestT, :].eval(TraitRef[ii]),  DataPOut.loc[sampETestT , :].eval(TraitRef[ii])), \
                                          np.append(np.array(ValiOT)[sampTestT], np.array(OutOT)[sampETestT]) ]).T , columns = ['Obs', 'Pred'] )
        results = smf.ols('Obs ~ Pred', data=DataTL).fit()
        MParam = results.params

        XPlot2 =  [ XPlot[0][0]-(0.1*XPlot[0][1]), XPlot[0][1]+(0.1*XPlot[0][1]) ]
        ax.plot( XPlot2, [ MParam.Intercept + MParam.Pred*XPlot2[val] for val in range(len(XPlot2))], '--', color='red', linewidth=0.8, zorder=2 )  # regression line

        # add R2 and RMSE corresponding to 100 models predicting the valdation data - careful, this includes out-of-bag for the source dataset
        R2valSelT   = R2VarSelSumm_mean_all.loc[     (R2VarSelSumm_mean_all.Trait == TraitRef[ii]) &   (R2VarSelSumm_mean_all.DateOut == DateOutRefT[kk]), 'IndV'].values[0]
        RMSEvalSelT = RMSEVarSelSumm_mean_all.loc[ (RMSEVarSelSumm_mean_all.Trait == TraitRef[ii]) & (RMSEVarSelSumm_mean_all.DateOut == DateOutRefT[kk]), 'IndV'].values[0]
        fig.text( 0.2525, 0.930, '$R^{2}$ = ' + str("%.2f" % round(R2valSelT, 3)) if R2valSelT > 0 else \
                                '$R^{2}$ = ' + str("%.2f" % round(0.00, 3)), va='center', rotation=0, size=FontSizeG, \
                  fontname = "Arial", fontstyle='normal', fontweight='normal', color='k')
        fig.text( 0.51, 0.930, ' ; RMSE = ' + str("%.2f" % round(RMSEvalSelT, 3)), va='center', rotation=0, size=FontSizeG, \
                  fontname = "Arial", fontstyle='normal', fontweight='normal', color='k')

        axisIntT = np.arange( YLimAllObs[jj][ii][0], YLimAllObs[jj][ii][-1]+axisIntDistT[jj][ii], axisIntDistT[jj][ii] ).round( 3 ).tolist()
        ax.set_yticks( axisIntT[ 0:axisIntDisLenT[jj][ii] ] )
        ax.set_yticklabels( [ str( axisIntRoundFT[jj][ii] % axisIntT[val]) for val in range(len( axisIntT[0:axisIntDisLenT[jj][ii]] )) ], fontname = "Arial",  fontsize=fontS2, ha='center' )
        ax.set_xticks(  axisIntT[0:axisIntDisLenT[jj][ii]] )
        ax.set_xticklabels( [ str( axisIntRoundFT[jj][ii] % axisIntT[val]) for val in range(len( axisIntT[0:axisIntDisLenT[jj][ii]] )) ], fontname = "Arial",  fontsize=fontS2, ha='center', rotation=45 )

        minorLocator1 = AutoMinorLocator()
        ax.plot( YLimAllObs[jj][ii], YLimAllObs[jj][ii], 'none' )
        ax.yaxis.set_minor_locator(minorLocator1)
        minorLocator2 = AutoMinorLocator()
        ax.plot( YLimAllObs[jj][ii], YLimAllObs[jj][ii], 'none' )
        ax.xaxis.set_minor_locator(minorLocator2)

        ax.tick_params(axis='y', which='major', width=0.8, length=MajorTicksLength, direction ='in', labelsize=FontSizeG, pad=13)
        ax.tick_params(axis='y', which='minor', width=0.8, length=MinorTicksLength, direction ='in', labelsize=FontSizeG, pad=13)
        ax.tick_params(axis='x', which='major', width=0.8, length=MajorTicksLength, direction ='in', labelsize=FontSizeG, pad=2)
        ax.tick_params(axis='x', which='minor', width=0.8, length=MinorTicksLength, direction ='in', labelsize=FontSizeG, pad=2)

        ax.grid(b=1, which='major', axis='both', color='gray', alpha=0.4, linestyle='dotted', linewidth=0.8)
        ax.grid(b=1, which='minor', axis='both', color='gray', alpha=0.2, linestyle='dotted', linewidth=0.8)

        fig.text( 0.01, 0.55, TraitRefL[ii] + ' - ' + 'Observ.', va='center', rotation=90, size=FontSizeG, fontname = "Arial", fontstyle='normal', fontweight='normal')
        fig.text( 0.55, 0.04, TraitRefL[ii] + ' - ' + 'Pred.', ha='center', va='center', rotation=0, \
                  size=FontSizeG, fontname = "Arial", fontstyle='normal', fontweight='normal')

        if( saveOpt == True ):
            os.chdir( r'<PROJECT_ROOT>\1_JypiterLabProcessing\4_Figures' )
            plt.savefig("VI_ScatterYearAll_CalVal_ER_PLSR_spike_VarSel_" + str([2014, 2017, 'All'][jj]) + "_Best_" + TraitRef[ii] + '_' + DateOutRefT[kk] + "_new.png", dpi=600, facecolor='w', edgecolor='w', \
                        orientation='portrait', format=None, \
                        transparent=False, bbox_inches='tight', pad_inches=0.05) # , bbox_inches='tight'

### Color scale

Color normalization centered on a chosen value, used by the next figure. *(Original cell 35.)*

In [ ]:
class MidpointNormalize(matplotlib.colors.Normalize):
    def __init__(self, vmin, vmax, midpoint=1, clip=False):
        self.midpoint = midpoint
        matplotlib.colors.Normalize.__init__(self, vmin, vmax, clip)

    def __call__(self, value, clip=None):
        normalized_min = max(0, 1 / 2 * (1 - abs((self.midpoint - self.vmin) / (self.midpoint - self.vmax))))
        normalized_max = min(1, 1 / 2 * (1 + abs((self.vmax - self.midpoint) / (self.midpoint - self.vmin))))
        normalized_mid = 0.5
        x, y = [self.vmin, self.midpoint, self.vmax], [normalized_min, normalized_mid, normalized_max]
        return np.ma.masked_array(np.interp(value, x, y))

### Figure 7

Average VIP per band and bands retained, for all training datasets. *(Original cell 38.)*

In [ ]:
import seaborn as sns
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator, AutoLocator, MaxNLocator)
import matplotlib.patches as patches
from matplotlib.collections import PatchCollection

matplotlib.rcParams['axes.linewidth'] = 1.0
matplotlib.rcParams['font.family']    = 'Arial'

# load VIP values - scenarios I to III - first evaluation
os.chdir( r'<PROJECT_ROOT>\1_JypiterLabProcessing\5_Outputs' )
versionRef = '01'
with open('VIPOUnF_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    VIPOUnF_allO_orig = pickle.load(f)
with open('VIPOUnF_varSel_allO_BootPLS_v' + versionRef + '.pkl', 'rb') as f:
    VIPOUnF_varSel_allO_orig = pickle.load(f)

# Extract info Eval and Valid - R2 and RMSE + var
TraitRef  = ['DM_dt_ha', 'N_perc_DM', 'N_kg_ha' ]  # reduced version of the traits - since crude protein is estimated from N content
YearRefL   = [ '2014', '2017', 'All data' ]
PredVIP_orig        = []
PredVIPIqr_orig     = []
PredVIP_varSel_orig = []
for val in range( len(YearRefL) ): # dataset
    PredVIP_orig.append( [] )
    PredVIPIqr_orig.append( [] )
    PredVIP_varSel_orig.append( [] )
modeRef = 'mean'
for ii in range(len(YearRefL)):  # dataset
    for jj in range( len(TraitRef) ):  # properties
        PredVIP_orig[ii].append( np.nanmedian( VIPOUnF_allO_orig[ii][jj], axis=1 ).tolist() if modeRef == 'med' else np.nanmean( VIPOUnF_allO_orig[ii][jj], axis=1 ).tolist() )
        PredVIPIqr_orig[ii].append( scipy.stats.iqr( VIPOUnF_allO_orig[ii][jj], nan_policy='omit', axis=1 ).tolist() if modeRef == 'med' else np.nanstd( VIPOUnF_allO_orig[ii][jj], axis=1 ).tolist() )
        PredVIP_varSel_orig[ii].append( np.nanmedian( VIPOUnF_varSel_allO_orig[ii][jj], axis=1 ).tolist() if modeRef == 'med' else np.nanmean( VIPOUnF_varSel_allO_orig[ii][jj], axis=1 ).tolist() )

TraitRefL  = [ 'Dry matter (DM)',
               'N% in DM',
               'N-uptake'  ]  # 1 line

YearRefL     = [ '2014', '2017', 'All data' ]
DataLabeRefT = [ '2014', '2017', 'All data', '2014+', '2017+' ]

cmapSel   = matplotlib.cm.get_cmap('PuOr_r')

fontS1 = 10
fontS2 = 11
fontS3 = 12
lwidth0 = 0.6
lwidth1 = 0.8
lwidth2 = 1.2

pattRepRefT    = 8
pattEmpRepRefT = 5

saveOpt = True

plotLegOptT = True

VIP_valuesCheck = [0.30, 2.30]

for jj in range( len(TraitRef) ):  # trait
    for ii in range( int(len(PredVIP_orig)+len(PredVIP)) ):  # dataset source |  year
        # create img to plot
        if( ii == 0 ):
            MetaToP = []
            EmpRefT    = np.empty( (1, len(PredVIP[0][0][0][0][0]) ) )
            EmpRefT[:] = np.nan
            DataToPT   = copy.deepcopy(EmpRefT)
            for val in range( pattEmpRepRefT-1 ):
                DataToPT   = np.append( DataToPT, EmpRefT, axis=0 )
        else:
            for val in range( pattEmpRepRefT ):  # add empty space before new data
                DataToPT   = np.append( DataToPT, EmpRefT, axis=0 )

        # add info to plot
        if( ii < len(PredVIP_orig) ): # original scenarios
            for val in range( pattRepRefT ): # number of repeated patterns
                DataToPT = np.append( DataToPT, \
                                      np.expand_dims( PredVIP_orig[ii][jj], axis=0 ), axis=0 )
            MetaToP.append( [ np.nanmin( PredVIP_orig[ii][jj]), np.nanmax(PredVIP_orig[ii][jj]), \
                              np.nanmean(PredVIP_orig[ii][jj]), np.nanstd(PredVIP_orig[ii][jj]) ] )
        else:  # after transfer
            kk = len(PredVIP[int(ii-len(PredVIP_orig))][jj])-1   # date outside source
            ll = bestSampleSelF_allO[int(ii-len(PredVIP_orig))][jj][kk]
            for val in range( pattRepRefT ): # number of repeated patterns
                DataToPT = np.append( DataToPT, \
                                      np.expand_dims( PredVIP[int(ii-len(PredVIP_orig))][jj][kk][ll][0], axis=0 ), axis=0 )
            MetaToP = [ [ np.nanmin( PredVIP[int(ii-len(PredVIP_orig))][jj][kk][ll][0]), np.nanmax(PredVIP[int(ii-len(PredVIP_orig))][jj][kk][ll][0]), \
                          np.nanmean(PredVIP[int(ii-len(PredVIP_orig))][jj][kk][ll][0]), np.nanstd(PredVIP[int(ii-len(PredVIP_orig))][jj][kk][ll][0]) ] ]

        # plot matrix
        if( ii == 0 ):
            fig = pylab.figure( figsize=(3.0, 5.5) )
            fig.subplots_adjust(left=0.15, right=0.85, bottom=0.05, top=0.925)
            ax = fig.add_subplot(1,1,1)

        # spectrogram with coeff. values
        normT      = MidpointNormalize( vmin=VIP_valuesCheck[0], vmax=VIP_valuesCheck[1], midpoint=1 )
        currentSet = ax.imshow( np.flip(DataToPT, axis=1).T, cmap=cmapSel, vmin=VIP_valuesCheck[0], vmax=VIP_valuesCheck[1], norm=normT )

    # indicate bands selected in each case
    selMinRefT   = (pattEmpRepRefT)
    for ii in range( int(len(PredVIP_orig)+len(PredVIP)) ):
       # Retained features according to VIP criteria
        if( ii < len(PredVIP_orig) ):  # original scenarios
            VIPRetainRefT = np.where( np.logical_not(np.isnan(PredVIP_varSel_orig[ii][jj])) )[0].tolist()
        else:  # after transfer
            kk = len(PredVIP[int(ii-len(PredVIP_orig))][jj])-1   # date outside source
            ll = bestSampleSelF_allO[int(ii-len(PredVIP_orig))][jj][kk]
            VIPRetainRefT = np.where( np.logical_not(np.isnan(PredVarSelSumm_mean_all[int(ii-len(PredVIP_orig))][jj][kk][6])) )[0].tolist()
        # reverse order
        VIPRetainRefT = [ (DataToPT.shape[1]-1)-VIPRetainRefT[val] for val in range(len(VIPRetainRefT)) ]
        # horizontal lines
        ax.plot( [(selMinRefT-2)]*len(VIPRetainRefT), VIPRetainRefT, \
                 color='blue', marker="_", mfc='blue', linewidth=0, markersize=7, markeredgewidth=lwidth2 )
        selMinRefT += (pattEmpRepRefT+pattRepRefT)

    ax.set_ylim( len(center_origin), -1 )
    fig.text( 0.023, 0.5, "Wavelength [ nm ]", \
              fontsize=fontS3, fontstyle='normal', va='center', ha='right', fontname = "Arial", color='k', rotation=90 )

    ticksPosRefT = list(range(0, DataToPT.shape[1], int(50/5)))
    ticksPosRefT = [ (len(center_origin)-1)-ticksPosRefT[val] for val in range(len(ticksPosRefT)) ]
    ax.set_yticks( ticksPosRefT )
    major_ticks = np.arange(450, 950, 50).tolist()
    ax.set_yticklabels(major_ticks, fontname = "Arial",  fontsize=fontS2 )

    ax.set_xlim( ax.get_xlim()[0], ax.get_xlim()[1] )
    fig.text( 0.820, 0.0650, "Training dataset [ Source ]", \
              fontsize=fontS3, fontstyle='normal', va='center', ha='right', color='k', fontname = "Arial", rotation=0 )

    xTickPosT = [ pattRepRefT ] * (len(PredVIP_orig)+len(PredVIP))
    xTickPosT = [ ((xTickPosT[val]/2)+(1*pattEmpRepRefT)-0.5) if (val == 0) else ((xTickPosT[val]/2)+(val*pattRepRefT)+(val*pattEmpRepRefT)+(1*pattEmpRepRefT)-0.5) for val in range(len(xTickPosT)) ]
    ax.set_xticks( xTickPosT )
    ax.set_xticklabels( ['']*len(DataLabeRefT), fontname = "Arial",  fontsize=fontS2, ha='center' )

    iniPosT   = 0.280
    deltaPosT = 0.140
    for val in range(len(DataLabeRefT)):
        fig.text( iniPosT+(val*deltaPosT), 0.175, DataLabeRefT[val],
                  fontsize=fontS2, fontstyle='normal', va='top', ha='right', color='k', fontname = "Arial", rotation=45)

    ax.tick_params(axis='y', which='major', length=3.5, width=lwidth1, direction="out", labelsize=fontS2, pad=2.5)
    ax.tick_params(axis='y', which='minor', length=1.5, width=lwidth1, direction="out")
    ax.tick_params(axis='x', which='major', length=3.5, width=lwidth1, direction="out", labelsize=fontS2)  # , pad=27
    ax.tick_params(axis='x', which='minor', length=1.5, width=lwidth1, direction="out")

    minorLocator1 = AutoMinorLocator()
    ax.yaxis.set_minor_locator(minorLocator1)

    # add title
    posTitleRefT = (0.545 if (jj == 0) else 0.3975 if (jj == 1) else 0.370)
    fig.text( posTitleRefT, 0.800, TraitRefL[jj],
              fontsize=fontS3, fontstyle='normal', va='center', ha='right', fontname = "Arial", color='k')

    # add patch
    rect1 = patches.Rectangle( (xTickPosT[2]-(pattRepRefT/2)-(pattEmpRepRefT/2)-0.5, -1), width=(pattEmpRepRefT+pattRepRefT)-(pattEmpRepRefT/2)+0.5, height=ax.get_ylim()[0]+1 )
    rectA  = [rect1]
    pc1 = PatchCollection(rectA, linewidth=1.40, edgecolor='magenta', facecolor='none', alpha=1, zorder=3 )
    ax.add_collection(pc1)

    # add legend
    if( plotLegOptT == True ):
        cax  = ax.figure.add_axes([0.880, 0.200, 0.035, 0.175])
        cbar = ax.figure.colorbar(currentSet, cax=cax, orientation='vertical')
        cbar.ax.tick_params( direction='in', labelsize=fontS1, labelcolor='k', width=0.8, color='k')   # , length=lineW1
        cbar.set_ticks( [ VIP_valuesCheck[0], 1, VIP_valuesCheck[1] ] )
        cbar.set_ticklabels( [VIP_valuesCheck[0],'1.0',VIP_valuesCheck[1]] )
        cbar.outline.set_edgecolor('k')
        cbar.outline.set_linewidth(0.8)
        for val in range(len(cbar.ax.get_yticklabels())):
            cbar.ax.get_yticklabels()[val].set_fontproperties('Arial')

        fig.text( 0.920, 0.410, 'VIP',
                  fontsize=fontS2, fontstyle='normal', va='center', ha='center', color='k', fontname = "Arial")
    # save
    if( saveOpt == True ):
        os.chdir(r'<PROJECT_ROOT>\1_JypiterLabProcessing\4_Figures')
        plt.savefig('VIP_PLSRAll_spike_FeatSel_SummAll_' + TraitRef[jj] + '_legOpt0' + str(0 if (plotLegOptT == False) else 1) + \
                    'v2.png', dpi=600, facecolor='w', edgecolor='w',
                     orientation='portrait', format=None,
                     transparent=False, bbox_inches='tight', pad_inches=0.1)

## 6. Maps

### Parameters

Settings of the pixel-level predictions. *(Original cell 39.)*

In [ ]:
# Variable Importance in Projection - formulation for VIP calculation   # https://github.com/scikit-learn/scikit-learn/issues/7050
def vip(model):
    t = model.x_scores_
    w = model.x_weights_
    q = model.y_loadings_
    p, h = w.shape
    vips = np.zeros((p,))
    s = np.diag(t.T @ t @ q.T @ q).reshape(h, -1)
    total_s = np.sum(s)
    for i in range(p):
        weight  = np.array([ (w[i,j] / np.linalg.norm(w[:,j]))**2 for j in range(h) ])
        vips[i] = np.sqrt(p*(s.T @ weight)/total_s)
    return vips

# max and min nbr of factors to use
nbrFactMax_thrshld = 10
nbrFactMin         = 3

TraitRef  = ['DM_dt_ha', 'N_perc_DM', 'N_kg_ha' ]  # reduced version of the traits - since crude protein seems to be estimated from N content
TraitRefL = ['Dry matter [dt/ha]', 'N [%]', 'N [kg/ha]']

DateRef  = SpecAll_sel_0_1_res_Avg.Date.unique().tolist()
DateRefY = np.unique( [ int(str(DateRef[val])[:4]) for val in range(len(DateRef)) ] ).tolist()

col_ref = list( range(len(center_origin)) )

# seed for calib and valid
seedAll = [325685, 73256841, 568]

# seed for CV repetition
BootSNbr   = 100 # nbr of resampling repetitions
seedRepTmF = []
np.random.seed(seed=23577)
for elmnt in range( len(TraitRef) ):  # traits
    seedRepTmF.append( np.random.uniform(low=0, high=1000000000, size=BootSNbr) )
    seedRepTmF[-1] = [ int(round(seedRepTmF[-1][val])) for val in range(len(seedRepTmF[-1])) ]

# random seed for k-means - part I
seedKmeanCombAll = []
np.random.seed(seed=27581425)
for elmnt in range( len(DateRefY) ):  # dataset source |  year
    if( elmnt == 0) :
        DateOutRefT =  HarvRef[2:5]
    else:
        DateOutRefT =  HarvRef[0:2]
    seedKmeanCombAllT = []
    for elmnt1 in range( len(TraitRef) ):  # trait  # nbr of samples available for a given trait | large or small set 0:2
        seedKmeanCombAllT.append( np.random.randint(low=0, high=1000000000, size=len(DateOutRefT)) )
    seedKmeanCombAll.append( seedKmeanCombAllT )

ScaleOpt = True
# indicate how 'uncertainty' is estimated while selecting the best number of samples to add from target domain
modeRef  = 'mean'
# attention - this parameter also affects feature selection - use 'stdS' in this case
EmodeRef = 'stdS'  # 'centroid' = select external samples according to cluster applied to the spectral data - closest sample to centroid ;
                   # 'stdS' = select samples according to 'uncertainty' criterion - largest uncertainty within the cluster
                   # obs.: run first 'stdS', because best sample selection is only performed based on this approach
if( EmodeRef == 'centroid' ):  # load best number of samples selected based on stdS (to compare outputs)
    os.chdir( r'<PROJECT_ROOT>\1_JypiterLabProcessing\5_Outputs' )
    versionRef = '106'  # latest version in which 'stdS' was saved (run 'stdS' first)
    # load
    with open('bestSampleSelF_allO_BootPLS_spike_opt_v' + versionRef + '_' + 'stdS' + '.pkl', 'rb') as f:
        bestSampleSelF_allO = pickle.load(f)

# maximum nbr of samples to add to the training data ( proportion from external data )
maxPropSamples = 1/2

# number of samples to add at the time - indicate how many clusters are adopted in k-means - starting from 2:
clstCntrlRef = 2

# threshold for selecting dataset providing satisfactory 'uncertainty' reduction
selThrshld   = 0.200

# threshold - 2nd criteria RMSE
selThrshldRMSE   = 0.300

# threshold for selecting PCA factors
selThrshldPCA   = 0.995

# select the criteria used to select optimal nbr of samples  --  'UncO' == uncertainty only ;
# 'UncRMSETO' == uncertainty + RMSE for out-of-bag sample + uncertainty (min uncertainty) ;
# 'UncRMSEO'  == uncertainty + RMSE for out-of-bag samples (min RMSE) ;
# 'UncRMSEOO' == uncertainty + RMSE for out-of-bag samples (min nbr fo samples)
# 'UncAvgO'   == closest to average uncertainty + min nbr of samples/uncertainty
criteriaOptT = 'UncAvgO'
if( criteriaOptT == 'UncAvgO' ): # select nbr of samples given minimum nbr of observations or uncertainty
    minSelCritOpt = 'MidSamp'

# indicate which data to use for mapping
SpecDataSelOpt = 'Buffer'

### Pixel-level predictions

Repeats the transfer with the selected number of samples and predicts every pixel. *(Original cell 40.)*

In [ ]:
InterBooF_all        = []
PLSRFactF_all        = []
R2EvalF_all          = []
RMSEEvalF_all        = []
R2EEvalF_all         = []
RMSEEEvalF_all       = []
R2ValF_all           = []
RMSEValF_all         = []
R2OutF_all           = []
RMSEOutF_all         = []
R2IndF_all           = []
RMSEIndF_all         = []
R2IndVF_all          = []
RMSEIndVF_all        = []
bestSampleSelF_all   = []
TestUnF_all          = []
ETestUnF_all         = []
EvalUnF_all          = []
ValiUnF_all          = []
OutDUnF_all          = []
VIPOUnF_all          = []
CoefUnF_all          = []
sampleValF_all       = []
selSampleRefAllF_all = []
sampleBooF_all       = []
PredMapUnF_all       = []
for ii in pbar.progressbar( range( len(DateRefY) ) ):  # dataset source |  year
    if( ii == 0 ) :
        DateOutRefT =  HarvRef[2:5]
    else:
        DateOutRefT =  HarvRef[0:2]
    InterBooF        = []
    PLSRFactF        = []
    R2EvalF          = []
    RMSEEvalF        = []
    R2EEvalF         = []
    RMSEEEvalF       = []
    R2ValF           = []
    RMSEValF         = []
    R2OutF           = []
    RMSEOutF         = []
    R2IndF           = []
    RMSEIndF         = []
    R2IndVF          = []
    RMSEIndVF        = []
    bestSampleSelF   = []
    TestUnF          = []
    ETestUnF         = []
    EvalUnF          = []
    ValiUnF          = []
    OutDUnF          = []
    VIPOUnF          = []
    CoefUnF          = []
    sampleValF       = []
    selSampleRefAllF = []
    sampleBooF       = []
    PredMapUnF       = []
    for jj in range( len(TraitRef) ):  # trait
        if( jj == 0 ): # Dry Biomass data / 49 observations per date
            DataP    = SpecAll_sel_0_1_res_Avg.loc[ (SpecAll_sel_0_1_res_Avg.Year).astype(int) == int(DateRefY[ii]), :]  # source data
            GroupF   = [ str(DataP.Replicate.tolist()[val]) + '_' + str(DataP.Treatment.tolist()[val]) for val in range(DataP.shape[0]) ]  # reference to each plot...
            DataPOut = SpecAll_sel_0_1_res_Avg.loc[ ~((SpecAll_sel_0_1_res_Avg.Year).astype(int) == int(DateRefY[ii])), :]  # data outside the source domain (different season)
            DataP_PC = copy.deepcopy( PCAmOM_scoresAll )
        else:  # N data / 15 observations per date
            DataP    = SpecAll_sel_0_1_res_Avg_treat.loc[ (SpecAll_sel_0_1_res_Avg_treat.Year).astype(int) == int(DateRefY[ii]), :]
            GroupF   = DataP.Treatment.tolist()
            DataPOut = SpecAll_sel_0_1_res_Avg_treat.loc[~((SpecAll_sel_0_1_res_Avg_treat.Year).astype(int) == int(DateRefY[ii])), :]
            DataP_PC = copy.deepcopy( PCAmOM_scoresAllRed )
        DataM_df    = DataP.loc[ :, (DataP.columns == TraitRef[jj]) | DataP.columns.isin(col_ref) ]  # select trait of interest + spectra
        DataPOut_df = DataPOut.loc[ :, (DataPOut.columns == TraitRef[jj]) | DataPOut.columns.isin(col_ref) ]  # data outside the source domain (different season/year)
        BRef = [ 'b' + str(range(len(col_ref))[val]) if range(len(col_ref))[val] > 9 else 'b0' + str( range(len(col_ref) )[val]) for val in range(len(col_ref)) ]
        DataM_df.columns = ['Trait'] + BRef
        DataM_df = DataM_df.reset_index(drop=True)
        DataPOut_df.columns = ['Trait'] + BRef
        DataPOut_df = DataPOut_df.reset_index(drop=True)

        # select calibration and validation datasets - source domain | valid = 0.3*n
        gsTV = sklearn.model_selection.GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seedAll[ii])
        trainIF_Oall = []
        testIF_Oall  = []
        for trainI, testI in gsTV.split(DataM_df.loc[:,BRef], DataM_df.Trait, GroupF):
            trainIF_Oall.append( trainI )
            testIF_Oall.append( testI )
        trainIF_Oall = trainIF_Oall[0].tolist()
        testIF_Oall  = testIF_Oall[0].tolist()

        # save reference to validation set - attention: based on row nbr not index
        sampleValF.append( testIF_Oall )

        # get calibration and validation sets
        Cal_df   = DataM_df.iloc[ trainIF_Oall,: ]
        GroupFTr = [ GroupF[val] for val in range(len(GroupF)) if val in trainIF_Oall ]
        Val_df   = DataM_df.iloc[ testIF_Oall,: ]
        GroupFTt = [ GroupF[val] for val in range(len(GroupF)) if val in testIF_Oall ]

        # Create arrays to store predictions - focus on avrg at the plot level | maps done later on (after optimal parameters are selected)
        TestUn  = []
        ETestUn = []
        EvalUn  = []
        ValiUn  = []
        OutDUn  = []
        VIPOUn  = []
        CoefUn  = []
        PredMapUn = []
        for val in range( len(DateOutRefT) ):   # date in the outside dataset (target)
            # Nbr of observations - selected data for a given target date
            DataPOutSelDimT = DataPOut_df.loc[ DataPOut.Date.isin( [ np.array(DateOutRefT[val]).astype('str').tolist() ] ).values ].shape[0]
            # max nbr of samples - use even or odd nbr to allow selection of 2: observation at the time (combine represetativity + uncertainty as sample selection criteria)
            MaxNbrSmplsOutRefTemp = int( np.ceil( DataPOutSelDimT*maxPropSamples ) )
            if( clstCntrlRef % 2 == 0 ):   # if nbr of samples to add at the time is even
                MaxNbrSmplsOutRefTemp = MaxNbrSmplsOutRefTemp if (MaxNbrSmplsOutRefTemp % 2 == 0) else (MaxNbrSmplsOutRefTemp-1)
            else:  # if nbr of samples to add at the time is odd
                MaxNbrSmplsOutRefTemp = (MaxNbrSmplsOutRefTemp-1) if (MaxNbrSmplsOutRefTemp % 2 == 0) else MaxNbrSmplsOutRefTemp
            TestUnT  = []
            ETestUnT = []
            EvalUnT  = []
            ValiUnT  = []
            OutDUnT  = []
            VIPOUnT  = []
            CoefUnT  = []
            PredMapUnT = []
            for val1 in range( int(MaxNbrSmplsOutRefTemp/clstCntrlRef) ):   # nbr of samples to add from the external dataset (target domain)  -  progressive increase
                                                                            # depending on: acq | dataset size
                TestUnT.append( np.full( (DataM_df.shape[0], len(seedRepTmF[jj])), np.nan ) )  # samples in the calib x repetitions (boot)
                EvalUnT.append( np.full( (DataM_df.shape[0], len(seedRepTmF[jj])), np.nan ) )
                ValiUnT.append( np.full( (DataM_df.shape[0], len(seedRepTmF[jj])), np.nan ) )
                ETestUnT.append( np.full( (DataPOut_df.shape[0], len(seedRepTmF[jj])), np.nan ) )   # extra samples in the calib x repetitions (boot) - using full array dimensions (standard dimensions)
                OutDUnT.append( np.full(  (DataPOut_df.shape[0], len(seedRepTmF[jj])), np.nan ) )
                VIPOUnT.append( np.full( (len(BRef), len(seedRepTmF[jj])), np.nan ) )
                CoefUnT.append( np.full( (len(BRef), len(seedRepTmF[jj])), np.nan ) )

                if( SpecDataSelOpt == 'Full' ):
                    FullDataSel   = SpecAllFull_sel_0_1_res[ HarvRef.index(DateOutRefT[val]) ]
                else:
                    FullDataSel   = SpecAll_sel_0_1_res[ HarvRef.index(DateOutRefT[val]) ]
                PredMapUnT.append( np.full( (FullDataSel.shape[1], len(seedRepTmF[jj])), np.nan ) )
            TestUn.append( TestUnT  )
            ETestUn.append( ETestUnT )
            EvalUn.append( EvalUnT )
            ValiUn.append( ValiUnT )
            OutDUn.append( OutDUnT )
            VIPOUn.append( VIPOUnT )
            CoefUn.append( CoefUnT )
            PredMapUn.append( PredMapUnT )

        InterBooTT      = []
        PLSRFactTT      = []
        R2EvalTT        = []
        RMSEEvalTT      = []
        R2EEvalTT       = []
        RMSEEEvalTT     = []
        R2ValTT         = []
        RMSEValTT       = []
        R2OutTT         = []
        RMSEOutTT       = []
        R2IndTT         = []
        RMSEIndTT       = []
        R2IndVTT        = []
        RMSEIndVTT      = []
        sampleBooTT     = []
        bestSampleSelTT = []
        Cal_dfUpInT       = []
        selSampleRefAllTT = []
        for kk in range( len(DateOutRefT) ):   # date in the outside dataset (target)
            DataPOutSelT = DataPOut_df.loc[ DataPOut.Date.isin( [ np.array(DateOutRefT[kk]).astype('str').tolist() ] ).values ]  # selected dataset for a given target date
            # max nbr of samples - use nbr to allow selection of 2: observation at the time (combine represetativity + uncertainty as sample selection criteria)
            MaxNbrSmplsOutRefTemp = int( np.ceil( DataPOutSelT.shape[0]*maxPropSamples ) )
            if( clstCntrlRef % 2 == 0 ):   # if nbr of samples to add at the time is even
                MaxNbrSmplsOutRefTemp = MaxNbrSmplsOutRefTemp if (MaxNbrSmplsOutRefTemp % 2 == 0) else (MaxNbrSmplsOutRefTemp-1)
            else:  # if nbr of samples to add at the time is odd
                MaxNbrSmplsOutRefTemp = (MaxNbrSmplsOutRefTemp-1) if (MaxNbrSmplsOutRefTemp % 2 == 0) else MaxNbrSmplsOutRefTemp
            if( jj == 0 ):  # larger dataset - DM
                CompKeep     = np.where( np.cumsum(PCAsumm[ HarvRef.index(DateOutRefT[kk]) ][:,0]) >= selThrshldPCA )[0].min()
            else:   # smaller dataset - N
                CompKeep     = np.where( np.cumsum(PCAsummRed[ HarvRef.index(DateOutRefT[kk]) ][:,0]) >= selThrshldPCA )[0].min()
            CompSel      = CompKeep+1 if CompKeep+1 > 3 else 3  # select at least 3 components
            DataP_PCSelT = DataP_PC[ HarvRef.index(DateOutRefT[kk]) ][ :, 0:CompSel ]

            # random seed for k-means - part II
            np.random.seed(seed=seedKmeanCombAll[ii][jj][kk])
            seedKmeanCombAllF = np.random.randint(low=0, high=1000000000, size=int(MaxNbrSmplsOutRefTemp/clstCntrlRef)).astype(int)

            InterBooT  = []
            PLSRFactT  = []
            R2EvalT    = []
            RMSEEvalT  = []
            R2EEvalT   = []
            RMSEEEvalT = []
            R2ValT     = []
            RMSEValT   = []
            R2OutT     = []
            RMSEOutT   = []
            R2IndT     = []
            RMSEIndT   = []
            R2IndVT    = []
            RMSEIndVT  = []
            sampleBooT = []
            selSampleRefAllT = []
            for ll in range( int(MaxNbrSmplsOutRefTemp/clstCntrlRef) ):   # nbr of samples to add from the external dataset (target domain) >> progressive seletion
                                                                                              # depending on: acq / dataset size (proportional)

                # uncertainty for samples from the outside dataset >> already modelled previously (PLSR part I) or based on previous selection loop (ll)
                if( ll == 0 ):  # all samples
                    DataPOutSelTT = DataPOutSelT.copy(deep=True)
                    DataP_PCSelTT = copy.deepcopy( DataP_PCSelT )
                    # uncertainty for samples available
                    stdOutAllT    = scipy.stats.iqr( OutDUnF_allO_PLS[ii][jj], nan_policy='omit', axis=1 ).tolist() if modeRef == 'med' else np.nanstd( OutDUnF_allO_PLS[ii][jj], axis=1 ).tolist()
                    stdOutSelTT   = np.array(stdOutAllT)[ DataPOut.Date.isin( [ np.array(DateOutRefT[kk]).astype('str').tolist() ] ).values ]
                else:
                    # remove samples selected previously - incremental dataset
                    DataPOutSelTT = DataPOutSelT[ ~DataPOutSelT.index.isin( list(itertools.chain(*selSampleRefAllT)) ) ]  # samples still available
                    DataP_PCSelTT = DataP_PCSelT[ ~DataPOutSelT.index.isin( list(itertools.chain(*selSampleRefAllT)) ) ]  # PCs for samples still available ;
                                                                                                                          # using representation derived for the complete dataset
# for elmnt in range(DataPOutSelT.shape[0]) ]
                    # uncertainty for samples still available
                    stdOutSelT  = OutDUn[kk][int(ll-1)][ DataPOut.Date.isin( [ np.array(DateOutRefT[kk]).astype('str').tolist() ] ).values ]
                    stdOutSelT  = scipy.stats.iqr( stdOutSelT, nan_policy='omit', axis=1 ).tolist() if modeRef == 'med' else np.nanstd( stdOutSelT, axis=1 ).tolist()
                    stdOutSelTT = np.array(stdOutSelT)[ ~DataPOutSelT.index.isin( list(itertools.chain(*selSampleRefAllT)) ) ]

                # fit k-means and assign each sample to a cluster (predict)
                KmeansM = sklearn.cluster.KMeans(n_clusters=clstCntrlRef, init='k-means++', n_init=10, max_iter=300, tol=0.0001,
                                                 verbose=0, copy_x=True, algorithm='auto', random_state=seedKmeanCombAllF[ll] )
                KmeansMO    = KmeansM.fit( DataP_PCSelTT )
                KmeansMPred = KmeansMO.predict( DataP_PCSelTT )

                selSampleRefAllTemp = []
                for elmnt in range( clstCntrlRef ):  # number of samples to select at the time - n:
                    DataPOutSelTTT = DataPOutSelTT.loc[ (KmeansMPred == elmnt) ]  # select elmnts in a given cluster
                    DataP_PCSelTTT = DataP_PCSelTT[     (KmeansMPred == elmnt) ]
                    stdOutSelTTT   = stdOutSelTT[       (KmeansMPred == elmnt) ]  # uncertainty for samples in the cluster
                    clstCntrT      = DataP_PCSelTTT.mean(axis=0).reshape(1, DataP_PCSelTTT.shape[1])  # cluster centroid
                    if( EmodeRef == 'centroid' ):   # select sample closest to the cluster centroid - opt1
                        distClstrOT     = scipy.spatial.distance.cdist( DataP_PCSelTTT, clstCntrT )
                        selSampleClstrT = np.where(distClstrOT == np.nanmin(distClstrOT))[0][0]
                        selSampleRefAllTemp.append( DataPOutSelTTT.index.tolist()[ selSampleClstrT ] )  # final sample selected in the outside dataset | target
                    else:   # select sample with largest uncertainty - opt2
                        selSampleStdT   = np.where( stdOutSelTTT == np.nanmax(stdOutSelTTT) )[0][0]
                        selSampleRefAllTemp.append( DataPOutSelTTT.index.tolist()[ selSampleStdT ] )
                selSampleRefAllT.append( selSampleRefAllTemp )

                # add samples from the external dataset - incremental
                if( all([ (ll==0) ]) ):
                    Cal_dfUpInT     = Cal_dfUpInT + [ DataPOutSelT.loc[ selSampleRefAllT[ll], :] ]
                else:
                    Cal_dfUpInTemp  = DataPOutSelT.loc[ selSampleRefAllT[ll], :]
                    Cal_dfUpInT[kk] = Cal_dfUpInT[kk].append( Cal_dfUpInTemp )  # not ignoring indices >> keep original row numbers  -  used for reference

                # Bootstrap internal repetition
                InterBooTemp  = []
                PLSRFactTemp  = []
                R2EvalTemp    = []
                RMSEEvalTemp  = []
                R2EEvalTemp   = []
                RMSEEEvalTemp = []
                R2ValTemp     = []
                RMSEValTemp   = []
                R2OutTemp     = []
                RMSEOutTemp   = []
                R2IndTemp     = []
                RMSEIndTemp   = []
                R2IndVTemp    = []
                RMSEIndVTemp  = []
                sampleBooTemp = []
                for mm in range( len(seedRepTmF[jj]) ):  # repetition for bootstrap (according to trait)

                    # get model fitting and test sets >> in and out of bag samples - only source data
                    Cal_dfT   = Cal_df.copy(deep=True)
                    GroupFTrT = copy.deepcopy(GroupFTr)
                    sizeRef   =  int( (Cal_dfT.shape[0]*0.3333) / len(DataP.Date.unique().tolist()) )
                    # divide by nbr of acquistions since its grouped by this factor and the selected samples will be multiplied by the same factor
                    gsBoo = sklearn.model_selection.GroupShuffleSplit(n_splits=1, test_size=sizeRef, random_state=seedRepTmF[jj][mm])
                    for trainI, testI in gsBoo.split(Cal_dfT.loc[:,BRef], Cal_dfT.Trait, GroupFTrT):
                        testIF    = testI.tolist()
                        testIFidx = Cal_dfT.iloc[testIF,:].index.tolist()
                    # save reference to test set within each boot repetition - attention: based on row nbr not index
                    sampleBooTemp.append( testIF )
                    Cal_dfT      = Cal_df.drop( labels = testIFidx )
                    GroupFTrTidx = [ val for val in range(Cal_df.shape[0]) if Cal_df.index.tolist()[val] not in testIFidx ]
                    GroupFTrT    = [ GroupFTr[val] for val in GroupFTrTidx ]
                    Eval_dfT     = Cal_df.loc[ testIFidx, :]

                    # add samples to the training data
                    Cal_dfUpT      = Cal_dfT.copy(deep=True)  # updated dataset
                    DataPOut_dfUpT = []
                    for elmnt in range( len(Cal_dfUpInT) ):  # date in the outside dataset (target)
                        DataPOutSelT = DataPOut_df.loc[ DataPOut.Date.isin( [ np.array(DateOutRefT[elmnt]).astype('str').tolist() ] ).values ]  # selected data for a given target date - outside samples
                        if( elmnt != kk ):  # previous dates
                            selSampleRefAllTT_sel = itertools.chain(*selSampleRefAllTT[elmnt][:(bestSampleSelTT[elmnt]+1) ])
                            selSampleRefAllTT_sel = [ True if Cal_dfUpInT[elmnt].index.tolist()[val] in selSampleRefAllTT_sel else False for val in range( Cal_dfUpInT[elmnt].shape[0] ) ]
                            Cal_dfUpT = Cal_dfUpT.append( Cal_dfUpInT[elmnt].loc[ selSampleRefAllTT_sel,: ], ignore_index=True )  # only optimal nbr of samples (for previous dates) - added at the end of the DataFrame
                            OutSampleIdxRefT = Cal_dfUpInT[elmnt].loc[ selSampleRefAllTT_sel,: ].index.tolist()  # samples included in the calibration model
                        else:  # current date
                            Cal_dfUpT = Cal_dfUpT.append( Cal_dfUpInT[elmnt], ignore_index=True )   # all samples selected until now (for current date)
                            OutSampleIdxRefT = Cal_dfUpInT[elmnt].index.tolist()  # samples included in the calibration model
                        # remove selected samples from available pool - incremental (date by date)
                        if( len(DataPOut_dfUpT) == 0 ):
                            DataPOut_dfUpT = DataPOutSelT.loc[ [ True if DataPOutSelT.index.tolist()[val] not in OutSampleIdxRefT else False for val in range(DataPOutSelT.shape[0]) ],: ]
                        else:
                            DataPOut_dfUpT = DataPOut_dfUpT.append( DataPOutSelT.loc[ [ True if DataPOutSelT.index.tolist()[val] not in OutSampleIdxRefT else False for val in range(DataPOutSelT.shape[0]) ],: ], \
                                                                    ignore_index=False )

                    # outputs according to number of PLS factors - not saved for posterior use , only outputs for best model are retained
                    RMSETO     = []
                    R2TO       = []
                    RMSETO_Out = []
                    R2TO_Out   = []
                    nbrFactMax = np.min( [ min(Cal_dfUpT.shape), nbrFactMax_thrshld ] )  # maximun nbr of PLS factors should be <= nbr of samples
                    for elmnt in range( (nbrFactMax-nbrFactMin)+1 ):  # nbr of PLS factors to try
                        plsM = PLSRegression( n_components=range(nbrFactMin, nbrFactMax+1, 1)[elmnt], scale=ScaleOpt )
                        plsM.fit( Cal_dfUpT.loc[:,BRef].values,  Cal_dfUpT.loc[:,'Trait'].values.reshape( (Cal_dfUpT.loc[:,'Trait'].shape[0], 1) ) )
                        # evaluation over the source domain
                        PredEvalO = plsM.predict(Eval_dfT.loc[:,BRef].values).reshape(Eval_dfT.shape[0],).tolist()
                        ResT =  Eval_dfT.Trait - np.array(PredEvalO)
                        uVal = (ResT**2).sum()
                        vVal = ((Eval_dfT.Trait.values.reshape(-1, 1) - Eval_dfT.Trait.values.reshape(-1, 1).mean())**2).sum()
                        R2TO.append( (1-(uVal/vVal)) )
                        RMSETO.append( np.sqrt( uVal/Eval_dfT.shape[0] ) )
                        # evaluation over the outside data in the source domain (calibration set - source)
                        DataPOut_df_EvalT = Cal_dfUpT.iloc[ (Cal_dfT.shape[0]):(Cal_dfUpT.shape[0]), :]  # only samples outside dataset - incremental
                        PredEvalO = plsM.predict(DataPOut_df_EvalT.loc[:,BRef].values).reshape(DataPOut_df_EvalT.shape[0],).tolist()
                        ResT =  DataPOut_df_EvalT.Trait - np.array(PredEvalO)
                        uVal = (ResT**2).sum()
                        vVal = ((DataPOut_df_EvalT.Trait.values.reshape(-1, 1) - DataPOut_df_EvalT.Trait.values.reshape(-1, 1).mean())**2).sum()
                        R2TO_Out.append( (1-(uVal/vVal)) )
                        RMSETO_Out.append( np.sqrt( uVal/DataPOut_df_EvalT.shape[0] ) )

                    # select factors | compromise between source and target accuracy
                    RMSETO_Out_sortedT = list( enumerate( RMSETO_Out ) )
                    RMSETO_Out_sortedT = sorted( RMSETO_Out_sortedT, key=lambda x:x[1], reverse=True )
                    RMSETO_Out_sortedTRef = [ RMSETO_Out_sortedT[val][0] for val in range(len(RMSETO_Out_sortedT)) ]
                    RMSETO_Out_sortedT    = [ RMSETO_Out_sortedT[val][1] for val in range(len(RMSETO_Out_sortedT)) ]
                    RMSETO_sortedT = [ RMSETO[ RMSETO_Out_sortedTRef[val] ] for val in range(len(RMSETO_Out_sortedTRef)) ]
                    RMSETO_joinT   = [ RMSETO_Out_sortedT[val]+RMSETO_sortedT[val] for val in range(len(RMSETO_Out_sortedT)) ]      # sum of RMSE for source and target domains
                    nbrFactSel = range(nbrFactMin, nbrFactMax+1, 1)[ np.where( (RMSETO_joinT == np.nanmin(RMSETO_joinT)) )[0][0] ]
                    PLSRFactTemp.append( nbrFactSel )
                    plsM = PLSRegression( n_components=nbrFactSel, scale=ScaleOpt )
                    plsM.fit( Cal_dfUpT.loc[:,BRef].values,  Cal_dfUpT.loc[:,'Trait'].values.reshape( (Cal_dfUpT.loc[:,'Trait'].shape[0], 1) ) )

                    # get predictions + model parameters
                    TestUn[kk][ll][ Cal_dfT.index.tolist(), mm] = plsM.predict(Cal_dfT.loc[:,BRef].values).reshape(len(Cal_dfT.index.tolist()),)  # same as >> plsM.predict(Cal_dfT.loc[:,BRef].values)
                                                                                                                                                  # source data - complete calibration set
                    # outside samples added to the training set
                    for elmnt in range( len(Cal_dfUpInT) ):  # date in the outside dataset (target)
                        # results from previous dates and current date are stored together - best set of samples for previous dates (need to be selected) + incremental dataset for current date
                        if( elmnt != kk ):  # previous dates
                            selSampleRefAllTT_sel = itertools.chain(*selSampleRefAllTT[elmnt][:(bestSampleSelTT[elmnt]+1) ])
                            selSampleRefAllTT_sel = [ True if Cal_dfUpInT[elmnt].index.tolist()[val] in selSampleRefAllTT_sel else False for val in range( Cal_dfUpInT[elmnt].shape[0] ) ]  # reference to best set of samples for previous dates
                            ETestUn[kk][ll][  Cal_dfUpInT[elmnt].loc[ selSampleRefAllTT_sel, :].index.tolist(), mm] = \
                                plsM.predict( Cal_dfUpInT[elmnt].loc[ selSampleRefAllTT_sel, BRef ].values ).reshape( Cal_dfUpInT[elmnt].loc[ selSampleRefAllTT_sel, :].shape[0], )
                        else:   # current date
                            ETestUn[kk][ll][ Cal_dfUpInT[elmnt].index.tolist(), mm] = plsM.predict( Cal_dfUpInT[elmnt].loc[:,BRef].values ).reshape( Cal_dfUpInT[elmnt].shape[0], )

                    EvalUn[kk][ll][testIFidx, mm] = plsM.predict( Eval_dfT.loc[:,BRef].values).reshape(len(testIFidx),)   # test samples - source dataset - boot repetition
                    ValiUn[kk][ll][ Val_df.index.tolist(), mm] = plsM.predict(Val_df.loc[:,BRef].values).reshape(len(Val_df.index.tolist()),)
                    OutDUn[kk][ll][ DataPOut_dfUpT.index.tolist(), mm] = plsM.predict( DataPOut_dfUpT.loc[:,BRef].values ).reshape( DataPOut_dfUpT.shape[0], )
                    VIPOUn[kk][ll][:, mm] = vip(plsM)
                    CoefUn[kk][ll][:, mm] = [ plsM.coef_.tolist()[val][0] for val in range(len(plsM.coef_.tolist())) ]
                    InterBooTemp.append( (plsM._y_mean - np.dot(plsM._x_mean, plsM.coef_)).tolist()[0] )
                    # https://stackoverflow.com/questions/41188410/how-obtain-the-intercept-of-the-pls-regression-sklearn

                    # Predict pixel-wise values - mapping
                    if( SpecDataSelOpt == 'Full' ):
                        FullDataSel   = SpecAllFull_sel_0_1_res[ HarvRef.index(DateOutRefT[kk]) ]
                    else:
                        FullDataSel   = SpecAll_sel_0_1_res[ HarvRef.index(DateOutRefT[kk]) ]
                    PredMapT_OutT = plsM.predict( FullDataSel[0,:,:] ).tolist()

                    # save outputs
                    PredMapUn[kk][ll][:, mm] =  np.array(PredMapT_OutT).reshape( (len(PredMapT_OutT),) )

                    # R2 and RMSE for calib set - Eval (source) - out-of-bag samples
                    PredT = copy.deepcopy( EvalUn[kk][ll][testIFidx, mm] )
                    ResT = Eval_dfT.Trait.values - np.array( PredT )
                    uVal = (ResT**2).sum()
                    vVal = ((Eval_dfT.Trait.values.reshape(-1, 1) - Eval_dfT.Trait.values.reshape(-1, 1).mean())**2).sum()
                    R2EvalTemp.append( (1-(uVal/vVal)) )
                    RMSEEvalTemp.append( np.sqrt( uVal/Eval_dfT.shape[0] ) )

                    # R2 and RMSE for external dataset - added to the training data - EEval
                    PredT = copy.deepcopy( ETestUn[kk][ll][ :, mm] )
                    ECal_dfUpT    = np.empty( ETestUn[kk][ll].shape[0], )  # same as ETestUn[kk][ll][:,mm] but for traits not predictions
                    ECal_dfUpT[:] = np.nan
                    for elmnt in range( len(Cal_dfUpInT) ):  # date in the outside dataset (target)
                        # samples from previous dates and current date are stored together - best set of samples for previous dates (need to be selected) + incremental dataset (for current date)
                        if( elmnt != kk ):  # previous dates
                            selSampleRefAllTT_sel = itertools.chain(*selSampleRefAllTT[elmnt][:(bestSampleSelTT[elmnt]+1) ])
                            selSampleRefAllTT_sel = [ True if Cal_dfUpInT[elmnt].index.tolist()[val] in selSampleRefAllTT_sel else False for val in range( Cal_dfUpInT[elmnt].shape[0] ) ]  # reference to best set of samples for previous dates
                            ECal_dfUpT[ Cal_dfUpInT[elmnt].loc[ selSampleRefAllTT_sel, :].index.tolist(), ] = \
                                Cal_dfUpInT[elmnt].loc[ selSampleRefAllTT_sel,'Trait'].values.reshape( Cal_dfUpInT[elmnt].loc[ selSampleRefAllTT_sel, :].shape[0], )
                        else:   # current date
                            ECal_dfUpT[ Cal_dfUpInT[elmnt].index.tolist(), ] = Cal_dfUpInT[elmnt].loc[:,'Trait'].values.reshape( Cal_dfUpInT[elmnt].shape[0], )
                    ResT = ECal_dfUpT - np.array( PredT )
                    uVal = np.nansum( (ResT**2) )
                    vVal = np.nansum( ((ECal_dfUpT.reshape(-1, 1) - np.nanmean(ECal_dfUpT.reshape(-1, 1)))**2) )
                    R2EEvalTemp.append( (1-(uVal/vVal)) )
                    RMSEEEvalTemp.append( np.sqrt( uVal/(ECal_dfUpT.shape[0]-np.isnan(ECal_dfUpT).sum()) ) )

                    # R2 and RMSE - validation set - source
                    PredT = copy.deepcopy( ValiUn[kk][ll][ Val_df.index.tolist(), mm] )
                    ResT = Val_df.Trait.values - np.array( PredT )
                    uVal = (ResT**2).sum()
                    vVal = ((Val_df.Trait.values.reshape(-1, 1) - Val_df.Trait.values.reshape(-1, 1).mean())**2).sum()
                    R2ValTemp.append( (1-(uVal/vVal)) )
                    RMSEValTemp.append( np.sqrt( uVal/Val_df.shape[0] ) )

                    # R2 and RMSE - outside dataset - not included in the training data
                    PredT = copy.deepcopy( OutDUn[kk][ll][ DataPOut_dfUpT.index.tolist(), mm] )
                    ResT = DataPOut_dfUpT.Trait.values - np.array( PredT )
                    uVal = (ResT**2).sum()
                    vVal = ((DataPOut_dfUpT.Trait.values.reshape(-1, 1) - DataPOut_dfUpT.Trait.values.reshape(-1, 1).mean())**2).sum()
                    R2OutTemp.append( (1-(uVal/vVal)) )
                    RMSEOutTemp.append( np.sqrt( uVal/ DataPOut_dfUpT.shape[0] ) )

                    # R2 and RMSE - all independent data (source + target) - careful source samples are out-of-bag not validation
                    ResT = np.append( (DataPOut_dfUpT.Trait.values - OutDUn[kk][ll][ DataPOut_dfUpT.index.tolist(), mm]), \
                                      (Eval_dfT.Trait.values - EvalUn[kk][ll][ testIFidx, mm]) )
                    uVal = (ResT**2).sum()
                    vVal = (( np.append(DataPOut_dfUpT.Trait.values.reshape(-1, 1), Eval_dfT.Trait.values.reshape(-1, 1)) - \
                              np.append(DataPOut_dfUpT.Trait.values.reshape(-1, 1), Eval_dfT.Trait.values.reshape(-1, 1)).mean() )**2).sum()
                    R2IndTemp.append( (1-(uVal/vVal)) )
                    RMSEIndTemp.append( np.sqrt( uVal/ (DataPOut_dfUpT.shape[0]+Eval_dfT.shape[0]) ) )

                    # R2 and RMSE - all independent data (source + target) >> validation (source ; stable) + samples not included in the training (target ; dynamic)
                    ResT = np.append( (DataPOut_dfUpT.Trait.values - OutDUn[kk][ll][ DataPOut_dfUpT.index.tolist(), mm]), \
                                      (Val_df.Trait.values - ValiUn[kk][ll][ Val_df.index.tolist(), mm]) )
                    uVal = (ResT**2).sum()
                    vVal = (( np.append(DataPOut_dfUpT.Trait.values.reshape(-1, 1), Val_df.Trait.values.reshape(-1, 1)) - \
                              np.append(DataPOut_dfUpT.Trait.values.reshape(-1, 1), Val_df.Trait.values.reshape(-1, 1)).mean() )**2).sum()
                    R2IndVTemp.append( (1-(uVal/vVal)) )
                    RMSEIndVTemp.append( np.sqrt( uVal/ (DataPOut_dfUpT.shape[0]+Val_df.shape[0]) ) )

                InterBooT.append( InterBooTemp )
                PLSRFactT.append( PLSRFactTemp )
                R2EvalT.append( R2EvalTemp )
                RMSEEvalT.append( RMSEEvalTemp )
                R2EEvalT.append( R2EEvalTemp )
                RMSEEEvalT.append( RMSEEEvalTemp )
                R2ValT.append( R2ValTemp )
                RMSEValT.append( RMSEValTemp )
                R2OutT.append( R2OutTemp )
                RMSEOutT.append( RMSEOutTemp )
                R2IndT.append( R2IndTemp )
                RMSEIndT.append( RMSEIndTemp )
                R2IndVT.append( R2IndVTemp )
                RMSEIndVT.append( RMSEIndVTemp )
                sampleBooT.append( sampleBooTemp )

                # find best number of samples to retain for a given outside dataset (target) - only after final samples have been considered
                if( ll == int((MaxNbrSmplsOutRefTemp/clstCntrlRef)-1) ):

                    if( EmodeRef != 'centroid' ):
                        # estimate variance (not absolute error)
                        UncSummT       = []
                        UncSummCalT    = []
                        RMSEEvalT_Summ = []
                        RMSEIndVT_Summ = []  # all validation data
                        for elmnt in range( len(OutDUn[kk]) ):  # nbr of samples to add from the external dataset (target domain) >> progressive seletion
                            # samples from the target dateset
                            PredSamplMaskT = np.isnan( OutDUn[kk][elmnt][:,0] )
                            OutDUn_selT    = OutDUn[kk][elmnt][ ~PredSamplMaskT,:]
                            UncEstAllT     = scipy.stats.iqr(OutDUn_selT, axis=1, nan_policy='omit') if modeRef == 'med' else np.nanstd(OutDUn_selT, axis=1)
                            UncSummT.append( np.nanmedian(UncEstAllT) if modeRef == 'med' else np.nanmean(UncEstAllT) )
                            # RMSEIndVT >> RMSE for external data - cannot be known if samples havee not been analysed before hand

                           # samples from the source dateset - out-of-bag samples
                            PredSamplMaskT_calib = np.isnan( EvalUn[kk][elmnt][:,0] )
                            EvalUn_selT          = EvalUn[kk][elmnt][ ~PredSamplMaskT_calib,:]
                            UncEstAllT           = scipy.stats.iqr(EvalUn_selT, axis=1, nan_policy='omit') if modeRef == 'med' else np.nanstd(EvalUn_selT, axis=1)
                            UncSummCalT.append( np.nanmedian(UncEstAllT) if modeRef == 'med' else np.nanmean(UncEstAllT) )
                            RMSEEvalT_Summ.append( scipy.stats.iqr(RMSEEvalT[elmnt], axis=0, nan_policy='omit') if modeRef == 'med' else np.nanstd(RMSEEvalT[elmnt], axis=0) )  # RMSE for out of thebag samples
                            RMSEIndVT_Summ.append( scipy.stats.iqr(RMSEIndVT[elmnt], axis=0, nan_policy='omit') if modeRef == 'med' else np.nanstd(RMSEIndVT[elmnt], axis=0) )
                        UncSummT_all = [ UncSummT[val] + UncSummCalT[val] for val in range(len(UncSummT)) ]
                        if( criteriaOptT != 'UncAvgO' ):  # UncO, etc...
                            selThrshldCalcO = np.nanmin(UncSummT_all)+(selThrshld*(np.nanmax(UncSummT_all)-np.nanmin(UncSummT_all)))
                        else:
                            selThrshldCalcO = (selThrshld*(np.nanmax(UncSummT_all)-np.nanmin(UncSummT_all)))

                        if( criteriaOptT == 'UncO' ):  # UncO
                            bestSampleSelTT.append(  np.nanmin( np.where((np.array(UncSummT_all) <= selThrshldCalcO)) ) )
                        elif( criteriaOptT == 'UncRMSETO' ):  # UncRMSETO
                            # v2.1 - two criteria | uncertainty outside dataset + RMSE for out-of-bag samples - uncertainty considered twice
                            bestSampleSelToPT      = np.where((np.array(UncSummT_all) <= selThrshldCalcO))
                            if( bestSampleSelToPT[0].shape[0] > 1 ):  # select dataset with lowest RMSE for out-of-bag samples
                                RMSERMSEEvalT_SummSelT = [ RMSEEvalT_Summ[val] for val in range(len(RMSEEvalT_Summ)) if val in bestSampleSelToPT[0].tolist() ]
                                RMSEThrsldSelT = np.where( (RMSERMSEEvalT_SummSelT <=
                                                            (np.nanmin(RMSERMSEEvalT_SummSelT)+((np.nanmax(RMSEEvalT_Summ)-np.nanmin(RMSEEvalT_Summ))*selThrshldRMSE))) )[0].tolist()
                                RMSEThrsldSelT = [ RMSERMSEEvalT_SummSelT[val] for val in RMSEThrsldSelT ]
                                RMSEThrsldSelT = [ np.where( RMSEEvalT_Summ == RMSEThrsldSelT[val] )[0][0] for val in range(len(RMSEThrsldSelT)) ]
                                UncThrsldSelT  = [ UncSummT_all[val] for val in RMSEThrsldSelT ]
                                bestSampleSelTT.append( np.where( UncSummT_all == np.nanmin(UncThrsldSelT) )[0][0] )  # select nbr of samples resulting in the lowest uncertainty after RMSE ccriterion was considered
                            else:
                                bestSampleSelTT.append(  np.nanmin( np.where((np.array(UncSummT_all) <= selThrshldCalcO)) ) )  # same as v1 in this case - lowest uncertainty
                        elif( criteriaOptT == 'UncRMSEO' ):  # UncRMSEO
                            # v2.2 - two criteria | uncertainty outside dataset + RMSE for out-of-bag samples - uncertainty considered only once (min RMSE)
                            bestSampleSelToPT      = np.where((np.array(UncSummT_all) <= selThrshldCalcO))
                            if( bestSampleSelToPT[0].shape[0] > 1 ):  # select dataset with lowest RMSE for out-of-bag samples
                                RMSERMSEEvalT_SummSelT = [ RMSEEvalT_Summ[val] for val in range(len(RMSEEvalT_Summ)) if val in bestSampleSelToPT[0].tolist() ]
                                RMSERMSEEvalT_SummSelT = [ RMSERMSEEvalT_SummSelT[val] for val in np.where( (RMSERMSEEvalT_SummSelT <= (np.nanmin(RMSERMSEEvalT_SummSelT)+
                                                                                        ((np.nanmax(RMSERMSEEvalT_SummSelT)-np.nanmin(RMSERMSEEvalT_SummSelT))*selThrshldRMSE))) )[0].tolist() ]
                                # select nbr of samples considering minimum RMSE only
                                bestSampleSelTT.append( np.where( (RMSEEvalT_Summ == np.nanmin(RMSERMSEEvalT_SummSelT)) )[0][0] )
                            else:
                                bestSampleSelTT.append(  np.nanmin( np.where((np.array(UncSummT_all) <= selThrshldCalcO)) ) )  # same as v1 in this case - lowest uncertainty
                        elif( criteriaOptT == 'UncRMSEOO' ): # UncRMSEOO
                            # v2.3 - two criteria | uncertainty outside dataset + RMSE for out-of-bag samples - uncertainty considered only once (min nbr of samples)
                            bestSampleSelToPT      = np.where((np.array(UncSummT_all) <= selThrshldCalcO))
                            if( bestSampleSelToPT[0].shape[0] > 1 ):  # select dataset with lowest RMSE for out-of-bag samples
                                RMSERMSEEvalT_SummSelT = [ RMSEEvalT_Summ[val] for val in range(len(RMSEEvalT_Summ)) if val in bestSampleSelToPT[0].tolist() ]
                                # select minimum nbr of samples considering RMSE only
                                bestSampleSelTT.append( np.where(RMSEEvalT_Summ ==
                                                                 RMSERMSEEvalT_SummSelT[ np.nanmin( np.where( (RMSERMSEEvalT_SummSelT <=
                                                                                                              (np.nanmin(RMSERMSEEvalT_SummSelT)+
                                                                                                              ((np.nanmax(RMSERMSEEvalT_SummSelT)-np.nanmin(RMSERMSEEvalT_SummSelT))*selThrshldRMSE))) )[0] ) ])[0][0] )
                            else:
                                bestSampleSelTT.append(  np.nanmin( np.where((np.array(UncSummT_all) <= selThrshldCalcO)) ) )  # same as v1 in this case - lowest uncertainty
                        else: # UncAvgO
                            lowThrshldT, highThrshldT = (np.nanmean(UncSummT_all) - selThrshldCalcO), (np.nanmean(UncSummT_all) + selThrshldCalcO)
                            bestSampleSelToPT         = np.where( ((np.array(UncSummT_all) >= lowThrshldT) & (np.array(UncSummT_all) <= highThrshldT)) )[0].tolist()
                            if( (len(bestSampleSelToPT) == 0) ):
                                UncSummT_allUpD = copy.deepcopy(UncSummT_all)
                                while( (len(bestSampleSelToPT) == 0) ):
                                    UncSummT_allUpD   = [ UncSummT_allUpD[val] for val in range(len(UncSummT_allUpD)) if UncSummT_allUpD[val] != np.nanmax(UncSummT_allUpD) ]
                                    lowThrshldT, highThrshldT = (np.nanmean(UncSummT_allUpD) - selThrshldCalcO), (np.nanmean(UncSummT_allUpD) + selThrshldCalcO)
                                    bestSampleSelToPT = np.where( ((np.array(UncSummT_allUpD) >= lowThrshldT) & (np.array(UncSummT_allUpD) <= highThrshldT)) )[0].tolist()
                                UncSummT_allUpD = [ UncSummT_allUpD[val] for val in bestSampleSelToPT ]
                            else:
                                UncSummT_allUpD = [ UncSummT_all[val] for val in bestSampleSelToPT ]
                            if( minSelCritOpt == 'MinUnc' ):    # select nbr of samples given minimum uncertainty
                                bestSampleSelTT.append( np.nanmin( np.where( (np.array(UncSummT_all) == np.nanmin(UncSummT_allUpD)) )[0] ) )
                            elif( minSelCritOpt == 'MinSamp' ): # select nbr of samples resulting in the min number of observations  # 'MinSamp'
                                bestSampleSelTT.append( np.nanmin( np.where( (np.array(UncSummT_all) == UncSummT_allUpD[0]) )[0] ) )
                            else: # MidSamp
                                bestSampleSelTT.append( np.floor( np.nanmedian( np.where( [ x in UncSummT_allUpD for x in UncSummT_all ] )[0] ) ).astype('int') ) # cahneg ceil or floor to increase or decrease nbr of samples elected

                    # use the same number of samples indicated as 'optimal' based on uncertainty - used for date specific comparison
                    else:
                        bestSampleSelTT.append( bestSampleSelF_allO[ii][jj][kk] )

            InterBooTT.append( InterBooT )
            PLSRFactTT.append( PLSRFactT )
            R2EvalTT.append( R2EvalT )
            RMSEEvalTT.append( RMSEEvalT )
            R2EEvalTT.append( R2EEvalT )
            RMSEEEvalTT.append( RMSEEEvalT )
            R2ValTT.append( R2ValT )
            RMSEValTT.append( RMSEValT )
            R2OutTT.append( R2OutT )
            RMSEOutTT.append( RMSEOutT )
            R2IndTT.append( R2IndT )
            RMSEIndTT.append( RMSEIndT )
            R2IndVTT.append( R2IndVT )
            RMSEIndVTT.append( RMSEIndVT )

            sampleBooTT.append( sampleBooT )
            selSampleRefAllTT.append( selSampleRefAllT )
        InterBooF.append( InterBooTT )
        PLSRFactF.append( PLSRFactTT )
        R2EvalF.append( R2EvalTT )
        RMSEEvalF.append( RMSEEvalTT )
        R2EEvalF.append( R2EEvalTT )
        RMSEEEvalF.append( RMSEEEvalTT )
        R2ValF.append( R2ValTT )
        RMSEValF.append( RMSEValTT )
        R2OutF.append( R2OutTT )
        RMSEOutF.append( RMSEOutTT )
        R2IndF.append( R2IndTT )
        RMSEIndF.append( RMSEIndTT )
        R2IndVF.append( R2IndVTT )
        RMSEIndVF.append( RMSEIndVTT )
        sampleBooF.append( sampleBooTT )
        bestSampleSelF.append( bestSampleSelTT )
        TestUnF.append( TestUn )
        EvalUnF.append( EvalUn )
        ETestUnF.append( ETestUn )
        ValiUnF.append( ValiUn )
        OutDUnF.append( OutDUn )
        VIPOUnF.append( VIPOUn )
        CoefUnF.append( CoefUn )
        PredMapUnF.append( PredMapUn )
        selSampleRefAllF.append( selSampleRefAllTT )
    InterBooF_all.append(InterBooF)
    PLSRFactF_all.append(PLSRFactF)
    R2EvalF_all.append(R2EvalF)
    RMSEEvalF_all.append(RMSEEvalF)
    R2EEvalF_all.append(R2EEvalF)
    RMSEEEvalF_all.append(RMSEEEvalF)
    R2ValF_all.append(R2ValF)
    RMSEValF_all.append(RMSEValF)
    R2OutF_all.append(R2OutF)
    RMSEOutF_all.append(RMSEOutF)
    R2IndF_all.append(R2IndF)
    RMSEIndF_all.append(RMSEIndF)
    R2IndVF_all.append(R2IndVF)
    RMSEIndVF_all.append(RMSEIndVF)
    sampleBooF_all.append( sampleBooF )
    bestSampleSelF_all.append( bestSampleSelF )
    TestUnF_all.append(TestUnF)
    EvalUnF_all.append(EvalUnF)
    ETestUnF_all.append(ETestUnF)
    ValiUnF_all.append(ValiUnF)
    OutDUnF_all.append(OutDUnF)
    VIPOUnF_all.append(VIPOUnF)
    CoefUnF_all.append(CoefUnF)
    PredMapUnF_all.append( PredMapUnF )
    sampleValF_all.append( sampleValF )
    selSampleRefAllF_all.append( selSampleRefAllF )

### Save or load the predictions

Reads the saved predictions. Set `SAVE_OUTPUTS` to save a new run first. *(Original cell 41.)*

In [ ]:
os.chdir( r'<PROJECT_ROOT>\1_JypiterLabProcessing\5_Outputs' )

versionRef = '106'

# indicate which data to use for mapping
SpecDataSelOpt = 'Full'

if( SpecDataSelOpt == 'Full' ):
    if( SAVE_OUTPUTS ):
        with open('PredMapUnF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'wb') as f:
            pickle.dump(PredMapUnF_all, f)
    # load
    with open('PredMapUnF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '.pkl', 'rb') as f:
        PredMapUnF_allO = pickle.load(f)
else:
    if( SAVE_OUTPUTS ):
        with open('PredMapUnF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '_' + str(SpecDataSelOpt) + '.pkl', 'wb') as f:
            pickle.dump(PredMapUnF_all, f)
    # load
    with open('PredMapUnF_allO_BootPLS_spike_opt_v' + versionRef + '_' + str(EmodeRef) + '_' + str(SpecDataSelOpt) + '.pkl', 'rb') as f:
        PredMapUnF_allO = pickle.load(f)

### Files of the predicted maps

Saves the predictions of each plot as vector and raster files, used for Figures 9 and 10. *(Original cell 42.)*

In [ ]:
TraitRef    = [ 'DM_dt_ha', 'N_perc_DM', 'N_kg_ha' ]

HarvRef = [ ['20140515', '20141015'], ['20170510', '20170830', '20171026'] ]
acqRefAll = [ [1, 2], [1, 3, 4] ]

# buffer to be added to image patches before rotation - in pixels
buffRefT = 20

MetaFieldRefT = [ 'Id', 'Naam', 'Treat', 'Block', 'Replicate', 'TreatCode' ]

NoData_value = np.nan

for ii in range( len(PredMapUnF_allO) ):  # dataset source |  year
    if( ii == 0 ):
        DateOutRefT = HarvRef[1]
        acqRefSelT  = acqRefAll[1]
    else:
        DateOutRefT = HarvRef[0]
        acqRefSelT  = acqRefAll[0]

    for jj in pbar.progressbar( range( len(PredMapUnF_allO[ii]) ) ):  # trait
        for kk in range( len(PredMapUnF_allO[ii][jj]) ):  # date in the outside dataset - target
            # metadata
            if( SpecDataSelOpt == 'Full' ):
                MetaDataSelT = SpecAllFull[ np.concatenate(HarvRef).tolist().index(DateOutRefT[kk]) ].loc[:, MetaFieldRefT]
            else:
                MetaDataSelT = SpecAll[ np.concatenate(HarvRef).tolist().index(DateOutRefT[kk]) ].loc[:, MetaFieldRefT ]
            BlockRefT    = np.sort( MetaDataSelT.Block.unique() ).tolist()
            yearRef = int(DateOutRefT[kk][0:4])
            acqRef  = int(acqRefSelT[kk])
            if(yearRef == 2017):
                if(acqRef == 1):
                    HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20170509\HYMSY'  # Hyper - Acq 1
                else:
                    if(acqRef == 3):
                        HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20170829\HYMSY\End Products_Manual'  # Hyper - Acq 3
                    else:
                        HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20171026\HYMSY\F1' # F1
            else:
                if(acqRef == 1):
                    HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20140515\End Products manual_update'  # Hyper - Acq 1
                else:
                    HomeDir = r'<PROJECT_ROOT>\DataProcessing\2_Data\1_UAV\20141014\End Products\Update_manual_20190314'  # Hyper - Acq 3  (calib02)
            os.chdir( HomeDir )
            PatternUpload = '.\\' + '*Manual_A.bsq'
            HypFilesAll = glob.glob(PatternUpload)
            resRefImT = []
            for val in range( len(HypFilesAll) ):  # flight line
                ImgProcessGL = gdal.Open( HypFilesAll[val], gdal.GA_ReadOnly )  # open one band with gdal just to get metadata
                resRefImT.append( ImgProcessGL.GetGeoTransform()[1] )
            ProjectionD = ImgProcessGL.GetProjection()

            # nbr of samples to add from the external dataset (target domain)
            ll = bestSampleSelF_allO[ii][jj][kk]

            for elmnt1 in range( len(BlockRefT) ):  # block
                RepRefT = np.sort( MetaDataSelT.loc[ (MetaDataSelT.Block == BlockRefT[elmnt1]), 'Replicate' ].unique() ).tolist()
                for elmnt2 in range( len(RepRefT) ):  # replicate
                    TreatRefT = MetaDataSelT.loc[ (MetaDataSelT.Block     == BlockRefT[elmnt1]) & \
                                                  (MetaDataSelT.Replicate == RepRefT[elmnt2]), 'TreatCode' ].unique().tolist()
                    TreatRefT = np.sort( [ TreatRefT[val].split('.')[-1] for val in range( len(TreatRefT) ) ] ).tolist()
                    for elmnt3 in range( len(TreatRefT) ):  # treatment
                        DataToPT    = PredMapUnF_allO[ii][jj][kk][ll][ (MetaDataSelT.Block == BlockRefT[elmnt1]) & \
                                                                       (MetaDataSelT.Replicate == RepRefT[elmnt2]) & \
                                                                       (MetaDataSelT.TreatCode == str(RepRefT[elmnt2]) + '.' + str(TreatRefT[elmnt3])), : ]  # predictions
                        if( SpecDataSelOpt == 'Full' ):
                            DataRefGeoT = SpecAll_geoFull[ np.concatenate(HarvRef).tolist().index(DateOutRefT[kk]) ].loc[ (MetaDataSelT.Block == BlockRefT[elmnt1]) & \
                                                                                                                          (MetaDataSelT.Replicate == RepRefT[elmnt2]) & \
                                                                                                                          (MetaDataSelT.TreatCode == str(RepRefT[elmnt2]) + '.' + str(TreatRefT[elmnt3])), ]
                        else:
                            DataRefGeoT = SpecAll_geo[ np.concatenate(HarvRef).tolist().index(DateOutRefT[kk]) ].loc[ (MetaDataSelT.Block == BlockRefT[elmnt1]) & \
                                                                                                                      (MetaDataSelT.Replicate == RepRefT[elmnt2]) & \
                                                                                                                      (MetaDataSelT.TreatCode == str(RepRefT[elmnt2]) + '.' + str(TreatRefT[elmnt3])), ]

                        xCoordTemp  =  DataRefGeoT.geometry.tolist()
                        xCoordTemp  =  [ xCoordTemp[val].coords.xy[0][0] for val in range( len(xCoordTemp) ) ]
                        yCoordTemp  =  DataRefGeoT.geometry.tolist()
                        yCoordTemp  =  [ yCoordTemp[val].coords.xy[1][0] for val in range( len(yCoordTemp) ) ]
                        xCoordT  =  [ min(xCoordTemp), max(xCoordTemp) ]
                        yCoordT  =  [ min(yCoordTemp), max(yCoordTemp) ]

                        # without buffer of -1 pixel left and upper sides
                        geoTransform     = [ xCoordT[0]-(min(resRefImT)/2), min(resRefImT), 0, yCoordT[1]-(min(resRefImT)/2), 0, -1*min(resRefImT) ]
                        geoTransformRstr = affine.Affine( min(resRefImT), 0, xCoordT[0]-(min(resRefImT)/2), \
                                                          0, -1*min(resRefImT), yCoordT[1]-(min(resRefImT)/2) )

                        x_res = int( np.floor( ((xCoordT[1]+min(resRefImT)) - (xCoordT[0]-min(resRefImT))) / min(resRefImT) ) )
                        y_res = int( np.floor( ((yCoordT[1]+min(resRefImT)) - (yCoordT[0]-min(resRefImT))) / min(resRefImT) ) )

                        # add buffer around image to avoid loss of data
                        geoTransform[0], geoTransform[3]  = geoTransform[0]-(buffRefT*min(resRefImT)), geoTransform[3]+(buffRefT*min(resRefImT))
                        geoTransformRstr = affine.Affine( min(resRefImT), 0, geoTransformRstr[2]-(buffRefT*min(resRefImT)), \
                                                          0, -1*min(resRefImT), geoTransformRstr[5]+(buffRefT*min(resRefImT)) )
                        x_res, y_res = x_res+(buffRefT*2), y_res+(buffRefT*2)  # multiply by 2 to consider both extremities in each direction

                        # remove predicted values below zero and interpolate missing info

                        # get row col reference for all pxls
                        rowCoordT, colCoordT = rasterio.transform.rowcol( transform=geoTransformRstr, xs=xCoordTemp , ys=yCoordTemp )

                        # replace negative values
                        DataToPT_int    = np.empty( DataToPT.shape )
                        DataToPT_int[:] = np.nan
                        for val in range( DataToPT.shape[1] ):  # info layers - boot repetition
                            DataSelT     = DataToPT[:,val]
                            valKeepMaskT = (DataSelT >= 0)
                            DataSelTT = DataSelT[valKeepMaskT]
                            rowCoordTT, colCoordTT = np.array(rowCoordT)[valKeepMaskT], np.array(colCoordT)[valKeepMaskT]
                            PtsRefT  = np.concatenate( (np.expand_dims(rowCoordTT,1), np.expand_dims(colCoordTT,1)), axis=1 )
                            gridOutT = scipy.interpolate.griddata( points=PtsRefT, values=DataSelTT, \
                                                                   xi=(rowCoordT, colCoordT), method='linear' )
                            DataToPT_int[:,val] = gridOutT

                        # summary of predictions
                        DataToPT_avg = np.nanmean( DataToPT_int, axis=1 ).reshape( (DataToPT_int.shape[0],1) )
                        DataToPT_std =  np.nanstd( DataToPT_int, axis=1 ).reshape( (DataToPT_int.shape[0],1) )
                        DataToPT     = np.concatenate( (DataToPT_avg, DataToPT_std), axis=1 )
                        if( any(np.isnan(DataToPT[:,0])) ):
                            DataRefGeoT = DataRefGeoT[ ~np.isnan(DataToPT[:,0]) ].copy(deep=True)
                            DataToPT    = copy.deepcopy( DataToPT[ ~np.isnan(DataToPT[:,0]) ] )
                        ColRefTemp  = [ 'avg', 'std' ]
                        DataToPGeoT = gpd.GeoDataFrame( data=DataToPT, columns=ColRefTemp, geometry=DataRefGeoT.geometry.tolist(), crs=DataRefGeoT.crs )
                        # save predictions as .shp
                        destFolderT = r'<PROJECT_ROOT>\1_JypiterLabProcessing\5_Outputs\PLSR_pred_res\Plot-wise'
                        os.chdir( destFolderT )
                        DataToPGeoT.to_file( filename= DateOutRefT[kk] + '_' + TraitRef[jj] + '_' + str(SpecDataSelOpt) + '_Block0' + str(BlockRefT[elmnt1]) + '_Repl0' + str(RepRefT[elmnt2]) + '_Treat' + \
                                                       str(TreatRefT[elmnt3]) + '_' + 'PLSR_pred_res' + '.shp' )

                        # create output image - n spectral bands
                        destFolderT = r'<PROJECT_ROOT>\1_JypiterLabProcessing\5_Outputs\PLSR_pred_res\Plot-wise'
                        os.chdir( destFolderT )
                        output_raster = DateOutRefT[kk] + '_' + TraitRef[jj] + '_' + str(SpecDataSelOpt) + '_Block0' + str(BlockRefT[elmnt1]) + '_Repl0' + str(RepRefT[elmnt2]) + '_Treat' + \
                                        str(TreatRefT[elmnt3]) + '_' + 'PLSR_pred_res' + '.tif'
                        driver   = gdal.GetDriverByName('GTiff')
                        out_data = driver.Create( os.path.join(destFolderT, output_raster), int(x_res+1), int(y_res+1), DataToPT.shape[1], gdalconst.GDT_Float64)
                        out_data.SetProjection( ProjectionD )
                        out_data.FlushCache()
                        out_data.SetGeoTransform( geoTransform )
                        out_data  = None

                        # gdal version - save images for each info layer
                        for mm in range( DataToPT.shape[1] ):  # info layer
                            out_data     = gdal.Open(output_raster, gdal.GA_Update)  # GA_ReadOnly
                            out_data.GetRasterBand( int(mm+1) ).SetNoDataValue( NoData_value )
                            out_data.GetRasterBand( int(mm+1) ).Fill( NoData_value )
                            source_ds    = ogr.Open( DateOutRefT[kk] + '_' + TraitRef[jj] + '_' + str(SpecDataSelOpt) + '_Block0' + str(BlockRefT[elmnt1]) + '_Repl0' + str(RepRefT[elmnt2]) + '_Treat' + \
                                                     str(TreatRefT[elmnt3]) + '_' + 'PLSR_pred_res' + '.shp' )
                            dataInT_gdal = source_ds.GetLayer()
                            gdal.RasterizeLayer( out_data, [int(mm+1)], dataInT_gdal, options=['ALL_TOUCHED=True', 'ATTRIBUTE='+ ColRefTemp[mm]] )
                            out_data  = None
                            source_ds = None

### Value ranges

Finds the range of each trait, used for the legends of the maps. *(Original cell 46.)*

In [ ]:
HarvRef = [ ['20140515', '20141015'], ['20170510', '20170830', '20171026'] ]
acqRefAll = [ [1, 2], [1, 3, 4] ]

MetaFieldRefT = [ 'Id', 'Naam', 'Treat', 'Block', 'Replicate', 'TreatCode' ]

TraitRef    = [ 'DM_dt_ha', 'N_perc_DM', 'N_kg_ha' ]

ColRefTemp  = [ 'avg', 'std' ]

# indicate which data to use for mapping
SpecDataSelOpt = 'Full'  # Full or Buffer >>> only Full used for the moment

NoData_value = np.nan

minThrshld, maxThrshld = 2.5, 97.5

TransfRangeAll = []
OrigRangeAll  = []
ResidRangeAll = []
for ii in range( len(HarvRef) ):  # dataset source |  year
    if( ii == 0 ):
        DateOutRefT = HarvRef[1]
        acqRefSelT  = acqRefAll[1]
    else:
        DateOutRefT = HarvRef[0][:-1]  # focus on first date of 2014
        acqRefSelT  = acqRefAll[0][:-1]  # focus on first date of 2014
    TransfRangeAllT = []
    OrigRangeAllT  = []
    ResidRangeAllT = []
    for jj in pbar.progressbar( range( len(TraitRef) ) ):  # trait
        # metadata
        MetaDataSelT = SpecAllFull[ np.concatenate(HarvRef).tolist().index(DateOutRefT[-1]) ].loc[:, MetaFieldRefT]
        BlockRefT    = np.sort( MetaDataSelT.Block.unique() ).tolist()
        for elmnt1 in range( len(BlockRefT) ):  # block
            RepRefT = np.sort( MetaDataSelT.loc[ (MetaDataSelT.Block == BlockRefT[elmnt1]), 'Replicate' ].unique() ).tolist()
            for elmnt2 in range( len(RepRefT) ):  # replicate
                TreatRefT = MetaDataSelT.loc[ (MetaDataSelT.Block     == BlockRefT[elmnt1]) & \
                                              (MetaDataSelT.Replicate == RepRefT[elmnt2]), 'TreatCode' ].unique().tolist()
                TreatRefT = np.sort( [ TreatRefT[val].split('.')[-1] for val in range( len(TreatRefT) ) ] ).tolist()
                for elmnt3 in range( len(TreatRefT) ):  # treatment
                   # load images
                    destFolderT = r'<PROJECT_ROOT>\1_JypiterLabProcessing\5_Outputs\PLSR_pred_res\Plot-wise'
                    os.chdir( destFolderT )
                    transfer_raster = DateOutRefT[-1] + '_' + TraitRef[jj] + '_' + str(SpecDataSelOpt) + '_Block0' + str(BlockRefT[elmnt1]) + '_Repl0' + str(RepRefT[elmnt2]) + '_Treat' + \
                                      str(TreatRefT[elmnt3]) + '_' + 'PLSR_pred_res' + '.tif'
                    ImgProcess_transfer  = gdal.Open(transfer_raster, gdal.GA_ReadOnly)
                    ImgProcess_transferR = ImgProcess_transfer.GetRasterBand( int(ColRefTemp.index('avg')+1) )
                    ImgProcess_transferR = ImgProcess_transferR.ReadAsArray()

                    orig_raster = DateOutRefT[-1] + '_' + TraitRef[jj] + '_' + str(SpecDataSelOpt) + '_Block0' + str(BlockRefT[elmnt1]) + '_Repl0' + str(RepRefT[elmnt2]) + '_Treat' + \
                                  str(TreatRefT[elmnt3]) + '_' + 'PLSR_pred_orig_res' + '.tif'
                    ImgProcess_orig  = gdal.Open(orig_raster, gdal.GA_ReadOnly)
                    ImgProcess_origR = ImgProcess_orig.GetRasterBand( int(ColRefTemp.index('avg')+1) )
                    ImgProcess_origR = ImgProcess_origR.ReadAsArray()

                    resid_raster = DateOutRefT[-1] + '_' + TraitRef[jj] + '_' + str(SpecDataSelOpt) + '_Block0' + str(BlockRefT[elmnt1]) + '_Repl0' + str(RepRefT[elmnt2]) + '_Treat' + \
                                    str(TreatRefT[elmnt3]) + '_' + 'PLSR_residuals_res' + '.tif'
                    ImgProcess_resid  = gdal.Open(resid_raster, gdal.GA_ReadOnly)
                    ImgProcess_residR = ImgProcess_resid.GetRasterBand( int(ColRefTemp.index('avg')+1) )
                    ImgProcess_residR = ImgProcess_residR.ReadAsArray()

                    if( all([ (elmnt1==0), (elmnt2==0), (elmnt2==0) ]) ):
                        TransfRangeAllTT = ImgProcess_transferR.reshape( np.prod(ImgProcess_transferR.shape), 1 )
                        OrigRangeAllTT   = ImgProcess_origR.reshape( np.prod(ImgProcess_origR.shape), 1 )
                        ResidRangeAllTT  = ImgProcess_residR.reshape( np.prod(ImgProcess_residR.shape), 1 )
                    else:
                        TransfRangeAllTT = np.concatenate( [TransfRangeAllTT, \
                                                           ImgProcess_transferR.reshape( np.prod(ImgProcess_transferR.shape), 1 ) ], axis=0 )
                        OrigRangeAllTT   = np.concatenate( [OrigRangeAllTT, \
                                                            ImgProcess_origR.reshape( np.prod(ImgProcess_origR.shape), 1 ) ], axis=0 )
                        ResidRangeAllTT = np.concatenate( [ResidRangeAllTT, \
                                                           ImgProcess_residR.reshape( np.prod(ImgProcess_residR.shape), 1 ) ] )
        TransfRangeAllT.append( [ np.nanpercentile(TransfRangeAllTT, minThrshld), np.nanpercentile(TransfRangeAllTT, maxThrshld) ] )
        OrigRangeAllT.append(   [ np.nanpercentile(OrigRangeAllTT,   minThrshld), np.nanpercentile(OrigRangeAllTT,   maxThrshld) ] )
        ResidRangeAllT.append(  [ np.nanpercentile(ResidRangeAllTT,  minThrshld), np.nanpercentile(ResidRangeAllTT,  maxThrshld) ]  )
    TransfRangeAll.append( TransfRangeAllT )
    OrigRangeAll.append( OrigRangeAllT )
    ResidRangeAll.append( ResidRangeAllT )

### Data for Figure 9

Joins the predictions of the baseline and transferred models for each pixel. *(Original cell 48.)*

In [ ]:
HarvRef = [ ['20140515', '20141015'], ['20170510', '20170830', '20171026'] ]
acqRefAll = [ [1, 2], [1, 3, 4] ]

MetaFieldRefT = [ 'Id', 'Naam', 'Treat', 'Block', 'Replicate', 'TreatCode' ]

TraitRef    = [ 'DM_dt_ha', 'N_perc_DM', 'N_kg_ha' ]

ColRefTemp  = [ 'avg', 'std' ]

# indicate which data to use for mapping
SpecDataSelOpt = 'Full'  # Full or Buffer >>> only Full used for the moment

DataJoint_All = []
for ii in range( len(HarvRef) ):  # dataset source |  year
    if( ii == 0 ):
         continue  # only for 2014 for the moment
    else:
        DateOutRefT = HarvRef[0][:-1]    # focus on first date of 2014
        acqRefSelT  = acqRefAll[0][:-1]  # focus on first date of 2014
    DataJoint_AllTT = []
    for jj in range( len(TraitRef) ):  # trait

        # reference for files to upload
        destFolderT = r'<PROJECT_ROOT>\1_JypiterLabProcessing\5_Outputs\PLSR_pred_res\Plot-wise'
        os.chdir( destFolderT )
        PatternUpload  = DateOutRefT[-1] + '_' + TraitRef[jj] + '_' + str(SpecDataSelOpt) + '*PLSR_pred_res.tif'
        DataExt_geoAll = glob.glob(PatternUpload)
        PatternUpload  = DateOutRefT[-1] + '_' + TraitRef[jj] + '_' + str(SpecDataSelOpt) + '*PLSR_pred_orig_res.tif'
        DataExt_geoAll_orig = glob.glob(PatternUpload)

        MetaRef_AllT = [ DataExt_geoAll[val].split('Block')[1][0:2] + '_' + DataExt_geoAll[val].split('Repl')[1][0:2] + '_' + DataExt_geoAll[val].split('Treat')[1][0:2] for \
                         val in range(len(DataExt_geoAll)) ]
        MetaRef_All_origT = [ DataExt_geoAll_orig[val].split('Block')[1][0:2] + '_' + DataExt_geoAll_orig[val].split('Repl')[1][0:2] + '_' + DataExt_geoAll_orig[val].split('Treat')[1][0:2] for \
                              val in range(len(DataExt_geoAll_orig)) ]

        for kk in pbar.progressbar( range(len(MetaRef_AllT)) ):  # experimental plot
            ImgProcess_transfer  = gdal.Open(DataExt_geoAll[kk], gdal.GA_ReadOnly)
            ImgProcess_transferR = ImgProcess_transfer.GetRasterBand( int(ColRefTemp.index('avg')+1) )
            ImgProcess_transferR = ImgProcess_transferR.ReadAsArray()
            ImgProcess_orig  = gdal.Open(DataExt_geoAll_orig[ np.where( [MetaRef_All_origT[val] == MetaRef_AllT[kk] for val in range(len(MetaRef_All_origT))])[0][0] ], gdal.GA_ReadOnly)
            ImgProcess_origR = ImgProcess_orig.GetRasterBand( int(ColRefTemp.index('avg')+1) )
            ImgProcess_origR = ImgProcess_origR.ReadAsArray()

            # arrange for pixel-wise data for comparison
            ImgProcess_concatT = np.concatenate( (np.expand_dims(ImgProcess_origR.flatten(), axis=1), np.expand_dims(ImgProcess_transferR.flatten(), axis=1)), axis=1 )
            ImgProcess_concatT = ImgProcess_concatT[ ~np.isnan(ImgProcess_concatT[:,0]),: ]
            if( kk == 0 ):
                DataJoint_AllT = copy.deepcopy(ImgProcess_concatT)
            else:
                DataJoint_AllT = np.append( DataJoint_AllT, copy.deepcopy(ImgProcess_concatT), axis=0 )
        DataJoint_AllTT.append( DataJoint_AllT )
    DataJoint_All.append( DataJoint_AllTT )

### Figure 9

Pixel-wise comparison of the predictions, with their marginal distributions. *(Original cell 50.)*

In [ ]:
from scipy.stats import gaussian_kde

from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator, AutoLocator, MaxNLocator)
minorLocator = AutoMinorLocator()
majorLocator = MaxNLocator(2)

matplotlib.rcParams['axes.linewidth'] = 0.8
matplotlib.rcParams['font.family']    = 'Arial'

fontS0=11
fontS1=11
fontS2=12
fontS3=13
lwidth0=0.4
lwidth1=0.8
lwidth2=1.5
lwidth3=2.5

lenMajorT = 3.10
lenMinorT = 1.90

HarvRef = [ ['20140515', '20141015'], ['20170510', '20170830', '20171026'] ]
acqRefAll = [ [1, 2], [1, 3, 4] ]

MetaFieldRefT = [ 'Id', 'Naam', 'Treat', 'Block', 'Replicate', 'TreatCode' ]
TraitRef    = [ 'DM_dt_ha', 'N_perc_DM', 'N_kg_ha' ]
TraitRefL  = [ r'Dry matter $[\/dt\/\/ha^{-1}\/]$',
               r'N content $[\/\%\/in\/DM\/]$',
               r'N total $[\/kg\/\/ha^{-1}\/]$'  ]

# indicate which data to use for mapping
SpecDataSelOpt = 'Full'  # Full or Buffer >>> only Full used for the moment

cmapA = [ sns.color_palette('mako', as_cmap=True), plt.get_cmap('cividis'), sns.color_palette('inferno', as_cmap=True) ]

colorRefT    = []
colorDensRef = []
for val in range(len(cmapA)):
    colorRefT.append(    cmapA[val]( (0.550if val != 1 else 0.8) ) )
    colorDensRef.append( cmapA[val]( (0.55 if val != 1 else 0.8) ) )

axisLimRef   = [ [0, 80], [0.40, 4.10], [0, 160] ]
tickDistRefT = [ 20, 0.925, 40 ]
gridDimRef   = [ [25,16], [25,16], [25,16] ]

# dates - use for file name
DateOutRefT = HarvRef[0][:-1]    # focus on first date of 2014

saveOptT = True

addColorBarT = True

binSizeRef               = 100
ratioYAxisDensRef_top    = 0.1
ratioYAxisDensRef_bottom = 0.225

for ii in range( len(DataJoint_All) ):  # dataset source |  year
    for jj in range( len(DataJoint_All[ii]) ):  # trait

        fig = pylab.figure( figsize=(2.9, 2.9) ) # 1:1
        fig.subplots_adjust(left=0.1, right=0.75, bottom=0.1, top=0.95)

        gs  = matplotlib.gridspec.GridSpec(2, 1, height_ratios=[1.55,7.00] )
        gs.update( wspace=0.04, hspace=0.04 )
        ax0 = plt.subplot(gs[1,0])
        ax1 = plt.subplot(gs[0,0])

        hbPlotT = ax0.hexbin( x=DataJoint_All[ii][jj][:,1], y=DataJoint_All[ii][jj][:,0], gridsize=gridDimRef[jj],
                              extent=list(itertools.chain(*[axisLimRef[jj], axisLimRef[jj]])), linewidths=lwidth0, edgecolors='w',
                              cmap=cmapA[jj], mincnt=5)

        # set ticks
        ticksSeqRefT = np.arange( axisLimRef[jj][0],  axisLimRef[jj][1]+tickDistRefT[jj], tickDistRefT[jj] )
        ax0.set_xticks( ticksSeqRefT )
        fmt = ('.1f' if jj in [0,2] else '.2f')
        ax0.set_xticklabels( [ format(ticksSeqRefT[val], fmt) for val in range(len(ticksSeqRefT)) ] )
        ax0.set_yticks( ticksSeqRefT )
        ax0.set_yticklabels( [ format(ticksSeqRefT[val], fmt) for val in range(len(ticksSeqRefT)) ] )

        minorLocator1 = AutoMinorLocator()
        ax0.xaxis.set_minor_locator(minorLocator1)
        ax0.yaxis.set_minor_locator(minorLocator1)

        ax0.tick_params(axis='x', which='major', length=lenMajorT , width=lwidth1, direction="in", labelsize=fontS1, pad=5)
        ax0.tick_params(axis='y', which='major', length=lenMajorT , width=lwidth1, direction="in", labelsize=fontS1)
        ax0.tick_params(axis='x', which='minor', length=lenMinorT , width=lwidth1, direction="in", labelsize=fontS1, pad=5)
        ax0.tick_params(axis='y', which='minor', length=lenMinorT , width=lwidth1, direction="in", labelsize=fontS1)

        # set limits
        ax0.set_xlim( [ axisLimRef[jj][0]-(axisLimRef[jj][1]/gridDimRef[jj][0]), axisLimRef[jj][1]+(axisLimRef[jj][1]/gridDimRef[jj][0]) ] )
        ax0.set_ylim( [ axisLimRef[jj][0]-(axisLimRef[jj][1]/gridDimRef[jj][1]), axisLimRef[jj][1]+(axisLimRef[jj][1]/gridDimRef[jj][1]) ] )
        # 1:1 line
        ax0.plot( [ axisLimRef[jj][0]-(axisLimRef[jj][1]/gridDimRef[jj][0]), axisLimRef[jj][1]+(axisLimRef[jj][1]/gridDimRef[jj][0]) ], \
                  [ axisLimRef[jj][0]-(axisLimRef[jj][1]/gridDimRef[jj][1]), axisLimRef[jj][1]+(axisLimRef[jj][1]/gridDimRef[jj][1]) ], \
                    color='k', linewidth=lwidth1 )

        sns.kdeplot( x=DataJoint_All[ii][jj][:,1], shade=1, gridsize=100, cut=3, color=colorRefT[jj], ax=ax1,
                     linewidth=lwidth1, alpha=1, zorder=1 )
        sns.kdeplot( x=DataJoint_All[ii][jj][:,0], shade=0, gridsize=100, cut=3, color='xkcd:neon green', ax=ax1,
                     linewidth=lwidth2, linestyle='--', alpha=1, zorder=2 )
        ax1.set_ylabel('')
        ax1.set_xlim( [ axisLimRef[jj][0]-(axisLimRef[jj][1]/gridDimRef[jj][0]), axisLimRef[jj][1]+(axisLimRef[jj][1]/gridDimRef[jj][0]) ] )
        ax1.set_ylim( ax1.get_ylim()[0]-( (ax1.get_ylim()[1]-ax1.get_ylim()[0])*ratioYAxisDensRef_bottom ),  ax1.get_ylim()[1]+( (ax1.get_ylim()[1]-ax1.get_ylim()[0])*ratioYAxisDensRef_top ) )

        ax1.set_xticks( ticksSeqRefT )
        ax1.set_xticklabels( [] )
        ax1.xaxis.set_minor_locator(minorLocator1)
        ax1.tick_params(axis='x', which='major', length=lenMajorT, width=lwidth1, direction="in", labelsize=fontS1, pad=5)
        ax1.tick_params(axis='x', which='minor', length=lenMinorT, width=lwidth1, direction="in", labelsize=fontS1)
        ax1.set_yticks( [] )
        ax1.set_yticklabels( [] )

        # add legend - color bar
        if( addColorBarT == True ):
            cax  = ax0.figure.add_axes([0.925, 0.10, 0.030, 0.30 ])
            cbar = ax0.figure.colorbar(hbPlotT, cax=cax, orientation='vertical')
            cbar.ax.tick_params(direction='in', labelsize=fontS1, labelcolor='k', width=lwidth1, color='k')   # , length=lineW1
            cbar.outline.set_edgecolor('k')
            cbar.outline.set_linewidth(lwidth1)
            cbar.set_ticks( np.linspace(hbPlotT.get_array().min(), hbPlotT.get_array().max(), 3) )
            cbar.set_ticklabels( np.linspace(0, 1., 3) )
            fmt = '.1f'
            cbar.set_ticklabels( [ format(np.linspace(0, 1., 3).tolist()[val], fmt) for val in range(len(np.linspace(0, 1., 3).tolist())) ] )
            fig.text( 0.940, 0.60, 'Density [-]', rotation=90,
                      fontsize=fontS1, fontstyle='normal', va='center', ha='center', color='k' )

        # axis title
        fig.text( (-0.09 if jj in [0,1] else -0.11), 0.425, TraitRefL[jj] + '\n' + 'Source 2014', rotation=90,
                  fontsize=fontS1, fontstyle='normal', va='center', ha='center', color='k' )
        fig.text( 0.425, -0.025, TraitRefL[jj] + '\n' + 'Source 2017+', rotation=0,
                  fontsize=fontS1, fontstyle='normal', va='center', ha='center', color='k' )

        # save figure
        if( saveOptT == True ):
            rootDirRefT = r'<PROJECT_ROOT>\1_JypiterLabProcessing\4_Figures'
            os.chdir( rootDirRefT )
            plt.savefig( 'Hexbin_single_PLSRAll_spike_MapAll_' + str(TraitRef[jj]) + '_' + str(DateOutRefT[ii])  + '.png',
                        dpi=600, facecolor='w', edgecolor='w',
                        orientation='portrait', papertype=None, format=None,
                        transparent=False, bbox_inches='tight', pad_inches=0.1)